In [1]:
import requests
import pandas as pd
from datetime import datetime

# La SEC exige identificar la aplicación en el User-Agent
HEADERS = {
    "User-Agent": "TFM Fraude Contable contacto@ejemplo.com"
}

# Datos iniciales de Belden
empresa = "Belden Inc."
cik = "0000913142"
t0 = "2018-02-01"

print("Empresa:", empresa)
print("CIK:", cik)
print("Fecha t0:", t0)

Empresa: Belden Inc.
CIK: 0000913142
Fecha t0: 2018-02-01


In [2]:
# URL oficial de submissions de la SEC para Belden
url = f"https://data.sec.gov/submissions/CIK{cik}.json"

response = requests.get(url, headers=HEADERS)

print("Status code:", response.status_code)

data = response.json()

# Mostramos información básica
print("Nombre SEC:", data.get("name"))
print("Tickers:", data.get("tickers"))
print("SIC:", data.get("sic"))
print("SIC description:", data.get("sicDescription"))


Status code: 200
Nombre SEC: BELDEN INC.
Tickers: ['BDC']
SIC: 3357
SIC description: Drawing & Insulating of  Nonferrous Wire


In [3]:
# Convertimos los filings recientes en un DataFrame
filings = pd.DataFrame(data["filings"]["recent"])

# Convertimos la fecha de filing a formato fecha
filings["filingDate"] = pd.to_datetime(filings["filingDate"])

# Convertimos t0 a fecha
t0_date = pd.to_datetime(t0)

# Filtramos solo 10-K presentados antes de t0
filings_10k = filings[
    (filings["form"] == "10-K") &
    (filings["filingDate"] < t0_date)
].copy()

# Ordenamos del más reciente al más antiguo
filings_10k = filings_10k.sort_values(
    by="filingDate",
    ascending=False
)

# Mostramos las columnas más útiles
resultado_10k = filings_10k[
    ["filingDate", "reportDate", "accessionNumber", "primaryDocument"]
]

print(resultado_10k.head(10))

    filingDate  reportDate       accessionNumber       primaryDocument
767 2017-02-17  2016-12-31  0000913142-17-000005  bdc-20163112x10k.htm
846 2016-02-25  2015-12-31  0001193125-16-477756        d48667d10k.htm
929 2015-02-23  2014-12-31  0001193125-15-057344       d838386d10k.htm


In [4]:
# Seleccionamos los 3 10-K válidos más recientes antes de t0
belden_filings = resultado_10k.head(3).copy()

# CIK sin ceros iniciales para construir las URLs de EDGAR
cik_sin_ceros = str(int(cik))

def construir_url_filing(row):
    accession_sin_guiones = row["accessionNumber"].replace("-", "")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_sin_ceros}/{accession_sin_guiones}/"
        f"{row['primaryDocument']}"
    )

belden_filings["url_10k"] = belden_filings.apply(
    construir_url_filing,
    axis=1
)

# Añadimos etiqueta temporal
belden_filings["periodo"] = ["t-1", "t-2", "t-3"]

print(
    belden_filings[
        ["periodo", "filingDate", "reportDate", "url_10k"]
    ].to_string(index=False)
)

periodo filingDate reportDate                                                                                url_10k
    t-1 2017-02-17 2016-12-31 https://www.sec.gov/Archives/edgar/data/913142/000091314217000005/bdc-20163112x10k.htm
    t-2 2016-02-25 2015-12-31       https://www.sec.gov/Archives/edgar/data/913142/000119312516477756/d48667d10k.htm
    t-3 2015-02-23 2014-12-31      https://www.sec.gov/Archives/edgar/data/913142/000119312515057344/d838386d10k.htm


In [5]:
# Descargamos los Company Facts XBRL de Belden
url_facts = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

response_facts = requests.get(url_facts, headers=HEADERS)

print("Status code:", response_facts.status_code)

facts = response_facts.json()

print("Empresa:", facts["entityName"])

# Accedemos a los conceptos US-GAAP disponibles
us_gaap = facts["facts"]["us-gaap"]

print("Número de conceptos US-GAAP disponibles:", len(us_gaap))

Status code: 200
Empresa: BELDEN INC.
Número de conceptos US-GAAP disponibles: 556


In [6]:
# Variables financieras que queremos localizar
conceptos_objetivo = {
    "Revenue": [
        "Revenues",
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax"
    ],
    "Assets": [
        "Assets"
    ],
    "Liabilities": [
        "Liabilities"
    ],
    "NetIncome": [
        "NetIncomeLoss"
    ],
    "Receivables": [
        "AccountsReceivableNetCurrent",
        "AccountsReceivableNet"
    ],
    "Inventory": [
        "InventoryNet"
    ],
    "Cash": [
        "CashAndCashEquivalentsAtCarryingValue"
    ],
    "Equity": [
        "StockholdersEquity"
    ]
}

# Comprobamos qué concepto existe realmente en Belden
conceptos_encontrados = {}

for variable, alternativas in conceptos_objetivo.items():
    encontrado = None
    
    for concepto in alternativas:
        if concepto in us_gaap:
            encontrado = concepto
            break
    
    conceptos_encontrados[variable] = encontrado

print("Conceptos encontrados:")
for variable, concepto in conceptos_encontrados.items():
    print(f"{variable}: {concepto}")

Conceptos encontrados:
Revenue: Revenues
Assets: Assets
Liabilities: None
NetIncome: NetIncomeLoss
Receivables: None
Inventory: InventoryNet
Cash: CashAndCashEquivalentsAtCarryingValue
Equity: StockholdersEquity


In [7]:
# Buscar conceptos relacionados con pasivos y cuentas a cobrar

print("=== POSIBLES CONCEPTOS DE PASIVOS ===")
for concepto in us_gaap.keys():
    if "Liabilit" in concepto:
        print(concepto)

print("\n=== POSIBLES CONCEPTOS DE CUENTAS A COBRAR ===")
for concepto in us_gaap.keys():
    if (
        "Receiv" in concepto
        or "AccountsReceivable" in concepto
    ):
        print(concepto)

=== POSIBLES CONCEPTOS DE PASIVOS ===
AccountsPayableAndAccruedLiabilitiesCurrent
AccruedLiabilitiesCurrent
BusinessCombinationContingentConsiderationLiability
BusinessCombinationRecognizedIdentifiableAssetsAcquiredAndLiabilitiesAssumedIntangibles
BusinessCombinationRecognizedIdentifiableAssetsAcquiredAndLiabilitiesAssumedNoncurrentLiabilitiesLongTermDebt
ContractWithCustomerLiability
ContractWithCustomerLiabilityChangeInTimeframePerformanceObligationSatisfiedRevenueRecognized
ContractWithCustomerLiabilityCumulativeCatchUpAdjustmentToRevenueModificationOfContract
ContractWithCustomerLiabilityCurrent
ContractWithCustomerLiabilityIncreaseDecreaseForContractAcquiredInBusinessCombination
ContractWithCustomerLiabilityNoncurrent
ContractWithCustomerLiabilityRevenueRecognized
DeferredIncomeTaxesAndOtherTaxLiabilitiesNoncurrent
DeferredIncomeTaxLiabilities
DeferredIncomeTaxLiabilitiesNet
DeferredTaxAssetsLiabilitiesNet
DeferredTaxLiabilities
DeferredTaxLiabilitiesNoncurrent
DeferredTaxLiabilit

In [8]:
# Actualizamos los conceptos encontrados
conceptos_encontrados["LiabilitiesCurrent"] = "LiabilitiesCurrent"
conceptos_encontrados["Receivables"] = "ReceivablesNetCurrent"

# Años que vamos a usar para Belden
anios_objetivo = [2016, 2015, 2014]

def extraer_valor_anual(concepto, anio):
    """
    Extrae el valor anual de un concepto XBRL para un año concreto.
    Prioriza datos de formularios 10-K con periodo fiscal FY.
    """
    
    if concepto is None or concepto not in us_gaap:
        return None
    
    unidades = us_gaap[concepto]["units"]
    
    # Normalmente estas variables están expresadas en USD
    if "USD" not in unidades:
        return None
    
    registros = pd.DataFrame(unidades["USD"])
    
    if registros.empty:
        return None
    
    # Nos quedamos con datos 10-K
    registros = registros[registros["form"] == "10-K"].copy()
    
    # Filtramos por ejercicio fiscal
    if "fy" in registros.columns:
        registros = registros[registros["fy"] == anio]
    
    if registros.empty:
        return None
    
    # Ordenamos por fecha de presentación
    registros["filed"] = pd.to_datetime(registros["filed"])
    registros = registros.sort_values("filed")
    
    # Cogemos el registro más reciente correspondiente a ese FY
    return registros.iloc[-1]["val"]


filas = []

for anio in anios_objetivo:
    
    revenue = extraer_valor_anual(
        conceptos_encontrados["Revenue"], anio
    )
    
    assets = extraer_valor_anual(
        conceptos_encontrados["Assets"], anio
    )
    
    net_income = extraer_valor_anual(
        conceptos_encontrados["NetIncome"], anio
    )
    
    receivables = extraer_valor_anual(
        conceptos_encontrados["Receivables"], anio
    )
    
    inventory = extraer_valor_anual(
        conceptos_encontrados["Inventory"], anio
    )
    
    cash = extraer_valor_anual(
        conceptos_encontrados["Cash"], anio
    )
    
    equity = extraer_valor_anual(
        conceptos_encontrados["Equity"], anio
    )
    
    liabilities_current = extraer_valor_anual(
        conceptos_encontrados["LiabilitiesCurrent"], anio
    )
    
    # Calculamos pasivos totales
    liabilities_total = None
    
    if assets is not None and equity is not None:
        liabilities_total = assets - equity
    
    filas.append({
        "Empresa": empresa,
        "Año": anio,
        "Revenue": revenue,
        "Assets": assets,
        "Liabilities_Total": liabilities_total,
        "Liabilities_Current": liabilities_current,
        "NetIncome": net_income,
        "Receivables": receivables,
        "Inventory": inventory,
        "Cash": cash,
        "Equity": equity
    })


belden_financials = pd.DataFrame(filas)

print(belden_financials.to_string(index=False))

    Empresa  Año Revenue     Assets  Liabilities_Total  Liabilities_Current  NetIncome  Receivables  Inventory      Cash     Equity
Belden Inc. 2016    None 3806803000         2346490000            570279000  128003000    388059000  190408000 848116000 1460313000
Belden Inc. 2015    None 3315841000         2491742000            549263000   66204000    387386000  195942000 216751000  824099000
Belden Inc. 2014    None 3262827000         2455641000            525359000   15993000    379777000  228398000 741162000  807186000


In [9]:
# Revisamos los registros XBRL disponibles para Revenues

concepto_revenue = conceptos_encontrados["Revenue"]

revenue_units = us_gaap[concepto_revenue]["units"]

print("Concepto utilizado:", concepto_revenue)
print("Unidades disponibles:", revenue_units.keys())

revenue_df = pd.DataFrame(revenue_units["USD"])

# Nos quedamos con registros relacionados con nuestros años
columnas = [
    "start",
    "end",
    "val",
    "accn",
    "fy",
    "fp",
    "form",
    "filed"
]

columnas_disponibles = [
    c for c in columnas if c in revenue_df.columns
]

revenue_10k = revenue_df[
    revenue_df["form"] == "10-K"
][columnas_disponibles].copy()

revenue_10k["end"] = pd.to_datetime(revenue_10k["end"])

revenue_10k = revenue_10k[
    revenue_10k["end"].dt.year.isin([2014, 2015, 2016])
]

print(
    revenue_10k.sort_values(["end", "filed"]).to_string(index=False)
)

Concepto utilizado: Revenues
Unidades disponibles: dict_keys(['USD'])
Empty DataFrame
Columns: [start, end, val, accn, fy, fp, form, filed]
Index: []


In [10]:
# Buscamos todos los conceptos relacionados con ingresos/ventas
# que tengan datos 10-K para 2014, 2015 o 2016.

candidatos_revenue = []

for concepto, info in us_gaap.items():

    nombre = concepto.lower()

    if "revenue" in nombre or "sales" in nombre:

        unidades = info.get("units", {})

        if "USD" not in unidades:
            continue

        df = pd.DataFrame(unidades["USD"])

        if df.empty or "form" not in df.columns or "end" not in df.columns:
            continue

        df = df[df["form"] == "10-K"].copy()

        if df.empty:
            continue

        df["end"] = pd.to_datetime(df["end"], errors="coerce")

        df_objetivo = df[
            df["end"].dt.year.isin([2014, 2015, 2016])
        ]

        if not df_objetivo.empty:
            candidatos_revenue.append(concepto)


print("Conceptos candidatos encontrados:\n")

for concepto in candidatos_revenue:
    print(concepto)

Conceptos candidatos encontrados:

DeferredRevenueCurrent
RevenueFromContractWithCustomerExcludingAssessedTax
RevenueFromContractWithCustomerIncludingAssessedTax
RoyaltyRevenue
SalesRevenueGoodsNet


In [11]:
# Comparamos los conceptos candidatos de ingresos
# para comprobar cuál representa las ventas consolidadas de Belden.

for concepto in candidatos_revenue:

    print("\n" + "=" * 80)
    print("CONCEPTO:", concepto)
    print("=" * 80)

    unidades = us_gaap[concepto].get("units", {})

    if "USD" not in unidades:
        continue

    df = pd.DataFrame(unidades["USD"])

    if df.empty:
        continue

    # Solo filings 10-K
    df = df[df["form"] == "10-K"].copy()

    # Convertimos fechas
    df["end"] = pd.to_datetime(df["end"], errors="coerce")
    df["filed"] = pd.to_datetime(df["filed"], errors="coerce")

    # Solo ejercicios terminados en 2014, 2015 o 2016
    df = df[
        df["end"].dt.year.isin([2014, 2015, 2016])
    ]

    # IMPORTANTE:
    # Solo información presentada antes de nuestro t0
    df = df[
        df["filed"] < t0_date
    ]

    columnas = [
        "start",
        "end",
        "val",
        "fy",
        "fp",
        "form",
        "filed",
        "accn"
    ]

    columnas = [c for c in columnas if c in df.columns]

    if df.empty:
        print("Sin datos válidos antes de t0")
    else:
        print(
            df[columnas]
            .sort_values(["end", "filed"])
            .to_string(index=False)
        )
    


CONCEPTO: DeferredRevenueCurrent
       end       val     fy fp form      filed                 accn
2014-12-31  45139000 2014.0 FY 10-K 2015-02-23 0001193125-15-057344
2014-12-31  45139000 2015.0 FY 10-K 2016-02-25 0001193125-16-477756
2015-12-31 101460000 2015.0 FY 10-K 2016-02-25 0001193125-16-477756
2015-12-31 101460000 2016.0 FY 10-K 2017-02-17 0000913142-17-000005
2016-12-31  80503000 2016.0 FY 10-K 2017-02-17 0000913142-17-000005

CONCEPTO: RevenueFromContractWithCustomerExcludingAssessedTax
Sin datos válidos antes de t0

CONCEPTO: RevenueFromContractWithCustomerIncludingAssessedTax
Sin datos válidos antes de t0

CONCEPTO: RoyaltyRevenue
     start        end      val     fy fp form      filed                 accn
2016-10-03 2016-12-31 10300000 2016.0 FY 10-K 2017-02-17 0000913142-17-000005

CONCEPTO: SalesRevenueGoodsNet
     start        end        val     fy fp form      filed                 accn
2014-01-01 2014-03-30  487690000 2014.0 FY 10-K 2015-02-23 0001193125-15-05734

In [12]:
def extraer_valor_anual(concepto, anio, t0_date):
    
    if concepto is None or concepto not in us_gaap:
        return None
    
    unidades = us_gaap[concepto].get("units", {})
    
    if "USD" not in unidades:
        return None
    
    df = pd.DataFrame(unidades["USD"])
    
    if df.empty:
        return None
    
    # Solo 10-K
    df = df[df["form"] == "10-K"].copy()
    
    # Fechas
    df["filed"] = pd.to_datetime(df["filed"], errors="coerce")
    df["end"] = pd.to_datetime(df["end"], errors="coerce")
    
    # Nunca utilizar información presentada después de t0
    df = df[df["filed"] < t0_date]
    
    # El dato debe corresponder al cierre del año buscado
    df = df[df["end"].dt.year == anio]
    
    if df.empty:
        return None
    
    # Si el concepto tiene duración (Revenue, Net Income, etc.)
    if "start" in df.columns:
        
        df["start"] = pd.to_datetime(df["start"], errors="coerce")
        df["dias"] = (df["end"] - df["start"]).dt.days
        
        # Priorizamos registros aproximadamente anuales
        anuales = df[
            (df["dias"] >= 330) &
            (df["dias"] <= 380)
        ]
        
        if not anuales.empty:
            df = anuales
    
    # Preferimos el dato presentado más cerca del cierre,
    # evitando en lo posible comparativos repetidos en 10-K posteriores
    df = df.sort_values("filed")
    
    return df.iloc[0]["val"]

In [13]:
conceptos_encontrados["Revenue"] = "SalesRevenueGoodsNet"

In [14]:
filas = []

for anio in [2016, 2015, 2014]:

    revenue = extraer_valor_anual(
        conceptos_encontrados["Revenue"], anio, t0_date
    )

    assets = extraer_valor_anual(
        conceptos_encontrados["Assets"], anio, t0_date
    )

    net_income = extraer_valor_anual(
        conceptos_encontrados["NetIncome"], anio, t0_date
    )

    receivables = extraer_valor_anual(
        conceptos_encontrados["Receivables"], anio, t0_date
    )

    inventory = extraer_valor_anual(
        conceptos_encontrados["Inventory"], anio, t0_date
    )

    cash = extraer_valor_anual(
        conceptos_encontrados["Cash"], anio, t0_date
    )

    equity = extraer_valor_anual(
        conceptos_encontrados["Equity"], anio, t0_date
    )

    liabilities_current = extraer_valor_anual(
        conceptos_encontrados["LiabilitiesCurrent"], anio, t0_date
    )

    liabilities_total = (
        assets - equity
        if assets is not None and equity is not None
        else None
    )

    filas.append({
        "Empresa": empresa,
        "Año": anio,
        "Revenue": revenue,
        "Assets": assets,
        "Liabilities_Total": liabilities_total,
        "Liabilities_Current": liabilities_current,
        "NetIncome": net_income,
        "Receivables": receivables,
        "Inventory": inventory,
        "Cash": cash,
        "Equity": equity
    })

belden_financials = pd.DataFrame(filas)

print(belden_financials.to_string(index=False))

    Empresa  Año    Revenue     Assets  Liabilities_Total  Liabilities_Current  NetIncome  Receivables  Inventory      Cash     Equity
Belden Inc. 2016 2356672000 3806803000         2346490000            570279000  128003000    388059000  190408000 848116000 1460313000
Belden Inc. 2015 2309222000 3315841000         2491742000            549263000   66204000    387386000  195942000 216751000  824099000
Belden Inc. 2014 2308265000 3262827000         2455641000            525359000   74449000    379777000  228398000 741162000  807186000


In [15]:
empresas_fraude = [
    "Belden Inc.",
    "General Electric",
    "Manitex International",
    "Power Solutions International",
    "Revolution Lighting Technologies",
    "Super Micro Computer",
    "Bausch Health",
    "VEREIT",
    "Healthcare Services Group",
    "Kraft Heinz",
    "Pareteum",
    "Cronos Group",
    "Koppers Holdings",
    "Tupperware Brands",
    "Compass Minerals"
]

print("Número de empresas:", len(empresas_fraude))

for i, nombre in enumerate(empresas_fraude, start=1):
    print(i, nombre)

Número de empresas: 15
1 Belden Inc.
2 General Electric
3 Manitex International
4 Power Solutions International
5 Revolution Lighting Technologies
6 Super Micro Computer
7 Bausch Health
8 VEREIT
9 Healthcare Services Group
10 Kraft Heinz
11 Pareteum
12 Cronos Group
13 Koppers Holdings
14 Tupperware Brands
15 Compass Minerals


In [16]:
# Descargamos el listado oficial de tickers y CIK de la SEC
url_tickers = "https://www.sec.gov/files/company_tickers.json"

resp_tickers = requests.get(url_tickers, headers=HEADERS)
print("Status code:", resp_tickers.status_code)

tickers_json = resp_tickers.json()

# Lo convertimos en DataFrame
sec_companies = pd.DataFrame.from_dict(tickers_json, orient="index")

sec_companies["cik_str"] = (
    sec_companies["cik_str"]
    .astype(str)
    .str.zfill(10)
)

print(sec_companies.head())

Status code: 200
      cik_str ticker           title
0  0001045810   NVDA     NVIDIA CORP
1  0000320193   AAPL      Apple Inc.
2  0001652044  GOOGL   Alphabet Inc.
3  0000789019   MSFT  MICROSOFT CORP
4  0001018724   AMZN  AMAZON COM INC


In [17]:
from difflib import get_close_matches

resultados_match = []

for nombre in empresas_fraude:
    titulos = sec_companies["title"].tolist()

    coincidencias = get_close_matches(
        nombre.upper(),
        [t.upper() for t in titulos],
        n=3,
        cutoff=0.35
    )

    print("\n" + "=" * 70)
    print("BUSCANDO:", nombre)
    print("=" * 70)

    if coincidencias:
        for coincidencia in coincidencias:
            fila = sec_companies[
                sec_companies["title"].str.upper() == coincidencia
            ]

            for _, row in fila.iterrows():
                print(
                    f"{row['title']} | "
                    f"Ticker: {row['ticker']} | "
                    f"CIK: {row['cik_str']}"
                )
    else:
        print("Sin coincidencias")


BUSCANDO: Belden Inc.
BELDEN INC. | Ticker: BDC | CIK: 0000913142
BLADEX, INC. | Ticker: BLX | CIK: 0000890541
Clene Inc. | Ticker: CLNN | CIK: 0001822791

BUSCANDO: General Electric
GENERAL ELECTRIC CO | Ticker: GE | CIK: 0000040545
EMERSON ELECTRIC CO | Ticker: EMR | CIK: 0000032604
UNIVERSAL ELECTRONICS INC | Ticker: UEIC | CIK: 0000101984

BUSCANDO: Manitex International
Magnitude International Ltd | Ticker: MAGH | CIK: 0002046117
TRIO-TECH INTERNATIONAL | Ticker: TRT | CIK: 0000732026
MAGNA INTERNATIONAL INC | Ticker: MGA | CIK: 0000749098

BUSCANDO: Power Solutions International
POWER SOLUTIONS INTERNATIONAL, INC. | Ticker: PSIX | CIK: 0001137091
FLEXIBLE SOLUTIONS INTERNATIONAL INC | Ticker: FSI | CIK: 0001069394
EDISON INTERNATIONAL | Ticker: EIX | CIK: 0000827052

BUSCANDO: Revolution Lighting Technologies
Evolution Metals & Technologies Corp. | Ticker: EMAT | CIK: 0001866226
CORE MOLDING TECHNOLOGIES INC | Ticker: CMT | CIK: 0001026655
Freight Technologies, Inc. | Ticker: FR

In [18]:
datos_maestros = [
    ["Belden Inc.", "BDC", "0000913142"],
    ["General Electric", "GE", "0000040545"],
    ["Manitex International", None, None],
    ["Power Solutions International", "PSIX", "0001137091"],
    ["Revolution Lighting Technologies", None, None],
    ["Super Micro Computer", "SMCI", "0001375365"],
    ["Bausch Health / Valeant", "BHC", "0000885590"],
    ["VEREIT / ARCP", None, None],
    ["Healthcare Services Group", "HCSG", "0000731012"],
    ["Kraft Heinz", "KHC", "0001637459"],
    ["Pareteum", None, None],
    ["Cronos Group", "CRON", "0001656472"],
    ["Koppers Holdings", "KOP", "0001315257"],
    ["Tupperware Brands", None, None],
    ["Compass Minerals", None, None]
]

tabla_maestra = pd.DataFrame(
    datos_maestros,
    columns=["Empresa", "Ticker", "CIK"]
)

print(tabla_maestra.to_string(index=False))

                         Empresa Ticker        CIK
                     Belden Inc.    BDC 0000913142
                General Electric     GE 0000040545
           Manitex International   None       None
   Power Solutions International   PSIX 0001137091
Revolution Lighting Technologies   None       None
            Super Micro Computer   SMCI 0001375365
         Bausch Health / Valeant    BHC 0000885590
                   VEREIT / ARCP   None       None
       Healthcare Services Group   HCSG 0000731012
                     Kraft Heinz    KHC 0001637459
                        Pareteum   None       None
                    Cronos Group   CRON 0001656472
                Koppers Holdings    KOP 0001315257
               Tupperware Brands   None       None
                Compass Minerals   None       None


In [19]:
actualizaciones = {
    "Manitex International": ("MNTX", "0001302028"),
    "Revolution Lighting Technologies": ("RVLT", "0000917523"),
    "VEREIT / ARCP": ("VER", "0001507385"),
    "Pareteum": ("TEUM", "0001084384"),
    "Tupperware Brands": ("TUP", "0001008654"),
    "Compass Minerals": ("CMP", "0001227654")
}

for empresa_nombre, (ticker, cik_empresa) in actualizaciones.items():
    tabla_maestra.loc[
        tabla_maestra["Empresa"] == empresa_nombre,
        ["Ticker", "CIK"]
    ] = [ticker, cik_empresa]

print(tabla_maestra.to_string(index=False))

                         Empresa Ticker        CIK
                     Belden Inc.    BDC 0000913142
                General Electric     GE 0000040545
           Manitex International   MNTX 0001302028
   Power Solutions International   PSIX 0001137091
Revolution Lighting Technologies   RVLT 0000917523
            Super Micro Computer   SMCI 0001375365
         Bausch Health / Valeant    BHC 0000885590
                   VEREIT / ARCP    VER 0001507385
       Healthcare Services Group   HCSG 0000731012
                     Kraft Heinz    KHC 0001637459
                        Pareteum   TEUM 0001084384
                    Cronos Group   CRON 0001656472
                Koppers Holdings    KOP 0001315257
               Tupperware Brands    TUP 0001008654
                Compass Minerals    CMP 0001227654


In [20]:
import time

datos_sec = []

for _, row in tabla_maestra.iterrows():

    empresa_tfm = row["Empresa"]
    cik_empresa = row["CIK"]

    url = f"https://data.sec.gov/submissions/CIK{cik_empresa}.json"

    try:
        response = requests.get(url, headers=HEADERS)

        if response.status_code == 200:

            info = response.json()

            datos_sec.append({
                "Empresa": empresa_tfm,
                "Nombre_SEC": info.get("name"),
                "Ticker": row["Ticker"],
                "CIK": cik_empresa,
                "SIC": info.get("sic"),
                "Descripcion_SIC": info.get("sicDescription"),
                "Cierre_Fiscal": info.get("fiscalYearEnd")
            })

            print(f"OK - {empresa_tfm}")

        else:

            print(
                f"ERROR - {empresa_tfm}: "
                f"status {response.status_code}"
            )

        # Pequeña pausa para respetar los servidores de la SEC
        time.sleep(0.15)

    except Exception as e:
        print(f"ERROR - {empresa_tfm}: {e}")


tabla_sec = pd.DataFrame(datos_sec)

print("\nTABLA SEC:")
print(tabla_sec.to_string(index=False))

OK - Belden Inc.
OK - General Electric
OK - Manitex International
OK - Power Solutions International
OK - Revolution Lighting Technologies
OK - Super Micro Computer
OK - Bausch Health / Valeant
OK - VEREIT / ARCP
OK - Healthcare Services Group
OK - Kraft Heinz
OK - Pareteum
OK - Cronos Group
OK - Koppers Holdings
OK - Tupperware Brands
OK - Compass Minerals

TABLA SEC:
                         Empresa                             Nombre_SEC Ticker        CIK  SIC                                             Descripcion_SIC Cierre_Fiscal
                     Belden Inc.                            BELDEN INC.    BDC 0000913142 3357                    Drawing & Insulating of  Nonferrous Wire          1231
                General Electric                    GENERAL ELECTRIC CO     GE 0000040545 3600 Electronic & Other Electrical Equipment (No Computer Equip)          1231
           Manitex International            Manitex International, Inc.   MNTX 0001302028 3559                           

In [21]:
# Fechas t0 preliminares/confirmadas
t0_dict = {
    "Belden Inc.": "2018-02-01",
    "General Electric": None,
    "Manitex International": "2018-04-03",
    "Power Solutions International": "2016-08-01",
    "Revolution Lighting Technologies": None,
    "Super Micro Computer": None,
    "Bausch Health / Valeant": "2015-10-26",
    "VEREIT / ARCP": "2014-10-29",
    "Healthcare Services Group": None,
    "Kraft Heinz": None,
    "Pareteum": "2019-10-21",
    "Cronos Group": None,
    "Koppers Holdings": None,
    "Tupperware Brands": "2021-08-01",
    "Compass Minerals": "2018-10-23"
}

tabla_sec["t0"] = tabla_sec["Empresa"].map(t0_dict)

print(
    tabla_sec[
        ["Empresa", "Ticker", "CIK", "SIC", "Cierre_Fiscal", "t0"]
    ].to_string(index=False)
)

                         Empresa Ticker        CIK  SIC Cierre_Fiscal         t0
                     Belden Inc.    BDC 0000913142 3357          1231 2018-02-01
                General Electric     GE 0000040545 3600          1231       None
           Manitex International   MNTX 0001302028 3559          1231 2018-04-03
   Power Solutions International   PSIX 0001137091 3510          1231 2016-08-01
Revolution Lighting Technologies   RVLT 0000917523 3640          1231       None
            Super Micro Computer   SMCI 0001375365 3571          0630       None
         Bausch Health / Valeant    BHC 0000885590 2834          1231 2015-10-26
                   VEREIT / ARCP    VER 0001507385 6798          1231 2014-10-29
       Healthcare Services Group   HCSG 0000731012 8050          1231       None
                     Kraft Heinz    KHC 0001637459 2030          1226       None
                        Pareteum   TEUM 0001084384 7373          1231 2019-10-21
                    Cronos G

In [22]:
import time

def obtener_todos_filings(cik_empresa):
    """
    Descarga los filings de una empresa desde SEC submissions.
    Incluye filings recientes y archivos históricos si existen.
    """

    url = f"https://data.sec.gov/submissions/CIK{cik_empresa}.json"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return pd.DataFrame()

    data_empresa = response.json()

    # Filings recientes
    filings = pd.DataFrame(data_empresa["filings"]["recent"])

    # Algunos filings antiguos están en archivos JSON separados
    archivos_historicos = data_empresa["filings"].get("files", [])

    for archivo in archivos_historicos:
        nombre_archivo = archivo["name"]

        url_historico = (
            f"https://data.sec.gov/submissions/{nombre_archivo}"
        )

        r = requests.get(url_historico, headers=HEADERS)

        if r.status_code == 200:
            historicos = pd.DataFrame(r.json())
            filings = pd.concat(
                [filings, historicos],
                ignore_index=True
            )

        time.sleep(0.12)

    return filings


resultados_periodos = []

for _, row in tabla_sec.iterrows():

    empresa_actual = row["Empresa"]
    cik_actual = row["CIK"]
    t0_actual = row["t0"]

    # Solo procesamos empresas con t0 ya definido
    if pd.isna(t0_actual):
        print(f"PENDIENTE t0 - {empresa_actual}")
        continue

    print(f"\nProcesando: {empresa_actual}")

    filings_empresa = obtener_todos_filings(cik_actual)

    if filings_empresa.empty:
        print("  ERROR: no se encontraron filings")
        continue

    # Convertimos fechas
    filings_empresa["filingDate"] = pd.to_datetime(
        filings_empresa["filingDate"],
        errors="coerce"
    )

    filings_empresa["reportDate"] = pd.to_datetime(
        filings_empresa["reportDate"],
        errors="coerce"
    )

    t0_fecha = pd.to_datetime(t0_actual)

    # Solo 10-K presentados antes de t0
    diez_k = filings_empresa[
        (filings_empresa["form"] == "10-K") &
        (filings_empresa["filingDate"] < t0_fecha)
    ].copy()

    diez_k = diez_k.sort_values(
        "filingDate",
        ascending=False
    )

    # Evitamos duplicar el mismo ejercicio
    diez_k = diez_k.drop_duplicates(
        subset=["reportDate"],
        keep="first"
    )

    # Nos quedamos con los 3 más recientes
    diez_k = diez_k.head(3)

    if len(diez_k) < 3:
        print(
            f"  ATENCIÓN: solo encontramos "
            f"{len(diez_k)} 10-K válidos"
        )

    for posicion, (_, filing) in enumerate(
        diez_k.iterrows(),
        start=1
    ):

        resultados_periodos.append({
            "Empresa": empresa_actual,
            "Ticker": row["Ticker"],
            "CIK": cik_actual,
            "t0": t0_actual,
            "Periodo": f"t-{posicion}",
            "Ejercicio": (
                filing["reportDate"].year
                if pd.notna(filing["reportDate"])
                else None
            ),
            "Fecha_10K": filing["filingDate"],
            "ReportDate": filing["reportDate"],
            "Accession": filing["accessionNumber"],
            "Documento": filing["primaryDocument"]
        })

        print(
            f"  t-{posicion}: "
            f"{filing['reportDate'].date()} | "
            f"presentado {filing['filingDate'].date()}"
        )

    time.sleep(0.15)


tabla_periodos = pd.DataFrame(resultados_periodos)

print("\n\nTABLA DE PERIODOS:")
print(
    tabla_periodos[
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Fecha_10K",
            "t0"
        ]
    ].to_string(index=False)
)


Procesando: Belden Inc.


/tmp/ipykernel_173/4259077361.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  filings = pd.concat(


  t-1: 2016-12-31 | presentado 2017-02-17
  t-2: 2015-12-31 | presentado 2016-02-25
  t-3: 2014-12-31 | presentado 2015-02-23
PENDIENTE t0 - General Electric

Procesando: Manitex International
  t-1: 2016-12-31 | presentado 2017-03-10
  t-2: 2015-12-31 | presentado 2016-03-10
  t-3: 2014-12-31 | presentado 2015-03-16

Procesando: Power Solutions International
  t-1: 2015-12-31 | presentado 2016-02-26
  t-2: 2014-12-31 | presentado 2015-03-13
  t-3: 2013-12-31 | presentado 2014-02-28
PENDIENTE t0 - Revolution Lighting Technologies
PENDIENTE t0 - Super Micro Computer

Procesando: Bausch Health / Valeant


/tmp/ipykernel_173/4259077361.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  filings = pd.concat(


  t-1: 2014-12-31 | presentado 2015-02-25
  t-2: 2013-12-31 | presentado 2014-02-28
  t-3: 2012-12-31 | presentado 2013-02-28

Procesando: VEREIT / ARCP
  t-1: 2013-12-31 | presentado 2014-02-27
  t-2: 2012-12-31 | presentado 2013-02-28
  t-3: 2011-12-31 | presentado 2012-03-19
PENDIENTE t0 - Healthcare Services Group
PENDIENTE t0 - Kraft Heinz

Procesando: Pareteum
  t-1: 2018-12-31 | presentado 2019-03-18
  t-2: 2017-12-31 | presentado 2018-03-30
  t-3: 2016-12-31 | presentado 2017-03-29
PENDIENTE t0 - Cronos Group
PENDIENTE t0 - Koppers Holdings

Procesando: Tupperware Brands
  t-1: 2020-12-26 | presentado 2021-03-10
  t-2: 2019-12-28 | presentado 2020-03-12
  t-3: 2018-12-29 | presentado 2019-02-26

Procesando: Compass Minerals


/tmp/ipykernel_173/4259077361.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  filings = pd.concat(


  t-1: 2017-12-31 | presentado 2018-02-27
  t-2: 2016-12-31 | presentado 2017-03-01
  t-3: 2015-12-31 | presentado 2016-02-22


TABLA DE PERIODOS:
                      Empresa Periodo  Ejercicio  Fecha_10K         t0
                  Belden Inc.     t-1       2016 2017-02-17 2018-02-01
                  Belden Inc.     t-2       2015 2016-02-25 2018-02-01
                  Belden Inc.     t-3       2014 2015-02-23 2018-02-01
        Manitex International     t-1       2016 2017-03-10 2018-04-03
        Manitex International     t-2       2015 2016-03-10 2018-04-03
        Manitex International     t-3       2014 2015-03-16 2018-04-03
Power Solutions International     t-1       2015 2016-02-26 2016-08-01
Power Solutions International     t-2       2014 2015-03-13 2016-08-01
Power Solutions International     t-3       2013 2014-02-28 2016-08-01
      Bausch Health / Valeant     t-1       2014 2015-02-25 2015-10-26
      Bausch Health / Valeant     t-2       2013 2014-02-28 2015-10-26
 

In [23]:
t0_nuevos = {
    "Super Micro Computer": "2017-08-29",
    "Healthcare Services Group": "2021-08-24",
    "Kraft Heinz": "2019-02-21",
    "Cronos Group": "2020-03-17",
    "Koppers Holdings": "2022-11-01"
}

for empresa_nombre, fecha in t0_nuevos.items():
    tabla_sec.loc[
        tabla_sec["Empresa"] == empresa_nombre,
        "t0"
    ] = fecha

print(
    tabla_sec[
        ["Empresa", "Ticker", "CIK", "t0"]
    ].to_string(index=False)
)

                         Empresa Ticker        CIK         t0
                     Belden Inc.    BDC 0000913142 2018-02-01
                General Electric     GE 0000040545       None
           Manitex International   MNTX 0001302028 2018-04-03
   Power Solutions International   PSIX 0001137091 2016-08-01
Revolution Lighting Technologies   RVLT 0000917523       None
            Super Micro Computer   SMCI 0001375365 2017-08-29
         Bausch Health / Valeant    BHC 0000885590 2015-10-26
                   VEREIT / ARCP    VER 0001507385 2014-10-29
       Healthcare Services Group   HCSG 0000731012 2021-08-24
                     Kraft Heinz    KHC 0001637459 2019-02-21
                        Pareteum   TEUM 0001084384 2019-10-21
                    Cronos Group   CRON 0001656472 2020-03-17
                Koppers Holdings    KOP 0001315257 2022-11-01
               Tupperware Brands    TUP 0001008654 2021-08-01
                Compass Minerals    CMP 0001227654 2018-10-23


In [24]:
tabla_sec.loc[
    tabla_sec["Empresa"] == "General Electric",
    "t0"
] = "2017-07-01"

tabla_sec.loc[
    tabla_sec["Empresa"] == "Revolution Lighting Technologies",
    "t0"
] = "2018-10-19"

print(
    tabla_sec[
        ["Empresa", "Ticker", "CIK", "t0"]
    ].to_string(index=False)
)

                         Empresa Ticker        CIK         t0
                     Belden Inc.    BDC 0000913142 2018-02-01
                General Electric     GE 0000040545 2017-07-01
           Manitex International   MNTX 0001302028 2018-04-03
   Power Solutions International   PSIX 0001137091 2016-08-01
Revolution Lighting Technologies   RVLT 0000917523 2018-10-19
            Super Micro Computer   SMCI 0001375365 2017-08-29
         Bausch Health / Valeant    BHC 0000885590 2015-10-26
                   VEREIT / ARCP    VER 0001507385 2014-10-29
       Healthcare Services Group   HCSG 0000731012 2021-08-24
                     Kraft Heinz    KHC 0001637459 2019-02-21
                        Pareteum   TEUM 0001084384 2019-10-21
                    Cronos Group   CRON 0001656472 2020-03-17
                Koppers Holdings    KOP 0001315257 2022-11-01
               Tupperware Brands    TUP 0001008654 2021-08-01
                Compass Minerals    CMP 0001227654 2018-10-23


In [25]:
import time

def obtener_todos_filings(cik_empresa):
    url = f"https://data.sec.gov/submissions/CIK{cik_empresa}.json"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return pd.DataFrame()

    data_empresa = response.json()

    frames = []

    recent = pd.DataFrame(data_empresa["filings"]["recent"])
    if not recent.empty:
        frames.append(recent)

    for archivo in data_empresa["filings"].get("files", []):
        url_historico = (
            f"https://data.sec.gov/submissions/{archivo['name']}"
        )

        r = requests.get(url_historico, headers=HEADERS)

        if r.status_code == 200:
            historicos = pd.DataFrame(r.json())

            if not historicos.empty:
                frames.append(historicos)

        time.sleep(0.12)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


resultados_periodos = []

for _, row in tabla_sec.iterrows():

    empresa_actual = row["Empresa"]
    cik_actual = row["CIK"]
    t0_actual = row["t0"]

    print(f"\nProcesando: {empresa_actual}")

    filings_empresa = obtener_todos_filings(cik_actual)

    if filings_empresa.empty:
        print("  ERROR: no se encontraron filings")
        continue

    filings_empresa["filingDate"] = pd.to_datetime(
        filings_empresa["filingDate"],
        errors="coerce"
    )

    filings_empresa["reportDate"] = pd.to_datetime(
        filings_empresa["reportDate"],
        errors="coerce"
    )

    t0_fecha = pd.to_datetime(t0_actual)

    diez_k = filings_empresa[
        (filings_empresa["form"] == "10-K") &
        (filings_empresa["filingDate"] < t0_fecha)
    ].copy()

    diez_k = diez_k.sort_values(
        "filingDate",
        ascending=False
    )

    diez_k = diez_k.drop_duplicates(
        subset=["reportDate"],
        keep="first"
    )

    diez_k = diez_k.head(3)

    if len(diez_k) < 3:
        print(
            f"  ATENCIÓN: solo encontramos "
            f"{len(diez_k)} 10-K válidos"
        )

    for posicion, (_, filing) in enumerate(
        diez_k.iterrows(),
        start=1
    ):

        resultados_periodos.append({
            "Empresa": empresa_actual,
            "Ticker": row["Ticker"],
            "CIK": cik_actual,
            "t0": t0_actual,
            "Periodo": f"t-{posicion}",
            "Ejercicio": (
                filing["reportDate"].year
                if pd.notna(filing["reportDate"])
                else None
            ),
            "Fecha_10K": filing["filingDate"],
            "ReportDate": filing["reportDate"],
            "Accession": filing["accessionNumber"],
            "Documento": filing["primaryDocument"]
        })

        print(
            f"  t-{posicion}: "
            f"{filing['reportDate'].date()} | "
            f"presentado {filing['filingDate'].date()}"
        )

    time.sleep(0.15)


tabla_periodos = pd.DataFrame(resultados_periodos)

print("\nNúmero de observaciones:", len(tabla_periodos))

print(
    tabla_periodos[
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Fecha_10K",
            "t0"
        ]
    ].to_string(index=False)
)


Procesando: Belden Inc.


/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2016-12-31 | presentado 2017-02-17
  t-2: 2015-12-31 | presentado 2016-02-25
  t-3: 2014-12-31 | presentado 2015-02-23

Procesando: General Electric


/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2016-12-31 | presentado 2017-02-24
  t-2: 2015-12-31 | presentado 2016-02-26
  t-3: 2015-02-27 | presentado 2015-02-27

Procesando: Manitex International
  t-1: 2016-12-31 | presentado 2017-03-10
  t-2: 2015-12-31 | presentado 2016-03-10
  t-3: 2014-12-31 | presentado 2015-03-16

Procesando: Power Solutions International
  t-1: 2015-12-31 | presentado 2016-02-26
  t-2: 2014-12-31 | presentado 2015-03-13
  t-3: 2013-12-31 | presentado 2014-02-28

Procesando: Revolution Lighting Technologies
  t-1: 2017-12-31 | presentado 2018-03-08
  t-2: 2016-12-31 | presentado 2017-03-09
  t-3: 2015-12-31 | presentado 2016-03-10

Procesando: Super Micro Computer


...

/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2020-12-31 | presentado 2021-02-25
  t-2: 2019-12-31 | presentado 2020-02-21
  t-3: 2018-12-31 | presentado 2019-03-18

Procesando: Kraft Heinz
  t-1: 2017-12-30 | presentado 2018-02-16
  t-2: 2016-12-31 | presentado 2017-02-23
  t-3: 2016-01-03 | presentado 2016-03-03

Procesando: Pareteum
  t-1: 2018-12-31 | presentado 2019-03-18
  t-2: 2017-12-31 | presentado 2018-03-30
  t-3: 2016-12-31 | presentado 2017-03-29

Procesando: Cronos Group
  ATENCIÓN: solo encontramos 1 10-K válidos
  t-1: 2019-12-31 | presentado 2020-03-02

Procesando: Koppers Holdings


/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2021-12-31 | presentado 2022-02-23
  t-2: 2020-12-31 | presentado 2021-02-24
  t-3: 2019-12-31 | presentado 2020-02-27

Procesando: Tupperware Brands
  t-1: 2020-12-26 | presentado 2021-03-10
  t-2: 2019-12-28 | presentado 2020-03-12
  t-3: 2018-12-29 | presentado 2019-02-26

Procesando: Compass Minerals
  t-1: 2017-12-31 | presentado 2018-02-27
  t-2: 2016-12-31 | presentado 2017-03-01
  t-3: 2015-12-31 | presentado 2016-02-22

Número de observaciones: 43
                         Empresa Periodo  Ejercicio  Fecha_10K         t0
                     Belden Inc.     t-1       2016 2017-02-17 2018-02-01
                     Belden Inc.     t-2       2015 2016-02-25 2018-02-01
                     Belden Inc.     t-3       2014 2015-02-23 2018-02-01
                General Electric     t-1       2016 2017-02-24 2017-07-01
                General Electric     t-2       2015 2016-02-26 2017-07-01
                General Electric     t-3       2015 2015-02-27 2017-07-01
           Man

/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


In [26]:
# ============================================================
# DIAGNÓSTICO DE GENERAL ELECTRIC Y CRONOS
# ============================================================

def mostrar_filings_anuales(empresa_nombre, formularios):
    
    fila = tabla_sec[
        tabla_sec["Empresa"] == empresa_nombre
    ].iloc[0]

    cik_empresa = fila["CIK"]
    t0_empresa = pd.to_datetime(fila["t0"])

    filings = obtener_todos_filings(cik_empresa)

    filings["filingDate"] = pd.to_datetime(
        filings["filingDate"],
        errors="coerce"
    )

    filings["reportDate"] = pd.to_datetime(
        filings["reportDate"],
        errors="coerce"
    )

    resultado = filings[
        (filings["form"].isin(formularios)) &
        (filings["filingDate"] < t0_empresa)
    ].copy()

    resultado = resultado.sort_values(
        "filingDate",
        ascending=False
    )

    print("\n" + "=" * 80)
    print(empresa_nombre)
    print("=" * 80)

    print(
        resultado[
            [
                "form",
                "filingDate",
                "reportDate",
                "accessionNumber",
                "primaryDocument"
            ]
        ].head(10).to_string(index=False)
    )


mostrar_filings_anuales(
    "General Electric",
    ["10-K"]
)

mostrar_filings_anuales(
    "Cronos Group",
    ["10-K", "20-F", "40-F"]
)


General Electric
form filingDate reportDate      accessionNumber   primaryDocument
10-K 2017-02-24 2016-12-31 0000040545-17-000010     ge10k2016.htm
10-K 2016-02-26 2015-12-31 0000040545-16-000145     ge10k2015.htm
10-K 2015-02-27 2015-02-27 0000040545-15-000030     ge10k2014.htm
10-K 2014-02-27 2013-12-31 0000040554-14-000023 geform10k2013.htm
10-K 2013-02-26 2012-12-31 0000040545-13-000036 geform10k2012.htm
10-K 2012-02-24 2011-12-31 0000040545-12-000016         ge10k.htm
10-K 2011-02-25 2010-12-31 0001193125-11-047479          d10k.htm
10-K 2010-02-19 2009-12-31 0000040545-10-000010        frm10k.htm
10-K 2009-02-18 2008-12-31 0000040545-09-000012        frm10k.htm
10-K 2008-02-20 2007-12-31 0000040545-08-000011        frm10k.htm

Cronos Group
form filingDate reportDate      accessionNumber                primaryDocument
10-K 2020-03-02 2019-12-31 0001656472-20-000011 cronosdraftfy20193220filin.htm
40-F 2019-03-26 2018-12-31 0001193125-19-085847                d711365d40f.htm
40-F 

/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


In [27]:
def corregir_report_date(report_date, filing_date, cierre_fiscal):
    """
    Corrige reportDate anómalos utilizando el cierre fiscal de la empresa.
    Ejemplo: GE 2014 aparece incorrectamente como 2015-02-27.
    """

    report_date = pd.to_datetime(report_date, errors="coerce")
    filing_date = pd.to_datetime(filing_date, errors="coerce")

    if pd.isna(report_date) or pd.isna(filing_date):
        return report_date

    cierre = str(cierre_fiscal).zfill(4)

    mes_cierre = int(cierre[:2])
    dia_cierre = int(cierre[2:])

    # Comprobamos si reportDate coincide aproximadamente
    # con el cierre fiscal declarado
    if (
        report_date.month == mes_cierre
        and abs(report_date.day - dia_cierre) <= 7
    ):
        return report_date

    # Si no coincide con el cierre fiscal y está muy cerca
    # de la fecha de filing, probablemente reportDate es erróneo.
    diferencia = abs((filing_date - report_date).days)

    if diferencia < 90:

        anio = filing_date.year - 1

        try:
            return pd.Timestamp(
                year=anio,
                month=mes_cierre,
                day=dia_cierre
            )

        except ValueError:
            return report_date

    return report_date

In [28]:
resultados_periodos = []

FORMULARIOS_ANUALES = ["10-K", "20-F", "40-F"]

for _, row in tabla_sec.iterrows():

    empresa_actual = row["Empresa"]
    cik_actual = row["CIK"]
    t0_actual = row["t0"]
    cierre_actual = row["Cierre_Fiscal"]

    print(f"\nProcesando: {empresa_actual}")

    filings_empresa = obtener_todos_filings(cik_actual)

    if filings_empresa.empty:
        print("  ERROR: no se encontraron filings")
        continue

    filings_empresa["filingDate"] = pd.to_datetime(
        filings_empresa["filingDate"],
        errors="coerce"
    )

    filings_empresa["reportDate"] = pd.to_datetime(
        filings_empresa["reportDate"],
        errors="coerce"
    )

    t0_fecha = pd.to_datetime(t0_actual)

    # 10-K, 20-F o 40-F anteriores al descubrimiento
    anuales = filings_empresa[
        (filings_empresa["form"].isin(FORMULARIOS_ANUALES)) &
        (filings_empresa["filingDate"] < t0_fecha)
    ].copy()

    # Corregimos posibles reportDate anómalos
    anuales["reportDate_corregido"] = anuales.apply(
        lambda x: corregir_report_date(
            x["reportDate"],
            x["filingDate"],
            cierre_actual
        ),
        axis=1
    )

    anuales = anuales.sort_values(
        "filingDate",
        ascending=False
    )

    # Evitamos duplicar ejercicio
    anuales["Ejercicio"] = (
        anuales["reportDate_corregido"].dt.year
    )

    anuales = anuales.drop_duplicates(
        subset=["Ejercicio"],
        keep="first"
    )

    anuales = anuales.head(3)

    if len(anuales) < 3:
        print(
            f"  ATENCIÓN: solo encontramos "
            f"{len(anuales)} informes anuales válidos"
        )

    for posicion, (_, filing) in enumerate(
        anuales.iterrows(),
        start=1
    ):

        resultados_periodos.append({
            "Empresa": empresa_actual,
            "Ticker": row["Ticker"],
            "CIK": cik_actual,
            "SIC": row["SIC"],
            "t0": t0_actual,
            "Periodo": f"t-{posicion}",
            "Ejercicio": filing["Ejercicio"],
            "Formulario": filing["form"],
            "Fecha_10K": filing["filingDate"],
            "ReportDate": filing["reportDate_corregido"],
            "Accession": filing["accessionNumber"],
            "Documento": filing["primaryDocument"]
        })

        print(
            f"  t-{posicion}: "
            f"{filing['Ejercicio']} | "
            f"{filing['form']} | "
            f"presentado {filing['filingDate'].date()}"
        )

    time.sleep(0.15)


tabla_periodos = pd.DataFrame(resultados_periodos)

print("\nNúmero total de observaciones:", len(tabla_periodos))

print(
    tabla_periodos[
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Formulario",
            "Fecha_10K",
            "t0"
        ]
    ].to_string(index=False)
)


Procesando: Belden Inc.


/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2016 | 10-K | presentado 2017-02-17
  t-2: 2015 | 10-K | presentado 2016-02-25
  t-3: 2014 | 10-K | presentado 2015-02-23

Procesando: General Electric


/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2016 | 10-K | presentado 2017-02-24
  t-2: 2015 | 10-K | presentado 2016-02-26
  t-3: 2014 | 10-K | presentado 2015-02-27

Procesando: Manitex International
  t-1: 2016 | 10-K | presentado 2017-03-10
  t-2: 2015 | 10-K | presentado 2016-03-10
  t-3: 2014 | 10-K | presentado 2015-03-16

Procesando: Power Solutions International
  t-1: 2015 | 10-K | presentado 2016-02-26
  t-2: 2014 | 10-K | presentado 2015-03-13
  t-3: 2013 | 10-K | presentado 2014-02-28

Procesando: Revolution Lighting Technologies
  t-1: 2017 | 10-K | presentado 2018-03-08
  t-2: 2016 | 10-K | presentado 2017-03-09
  t-3: 2015 | 10-K | presentado 2016-03-10

Procesando: Super Micro Computer


...

/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2020 | 10-K | presentado 2021-02-25
  t-2: 2019 | 10-K | presentado 2020-02-21
  t-3: 2018 | 10-K | presentado 2019-03-18

Procesando: Kraft Heinz
  t-1: 2017 | 10-K | presentado 2018-02-16
  t-2: 2016 | 10-K | presentado 2017-02-23
  t-3: 2015 | 10-K | presentado 2016-03-03

Procesando: Pareteum
  t-1: 2018 | 10-K | presentado 2019-03-18
  t-2: 2017 | 10-K | presentado 2018-03-30
  t-3: 2016 | 10-K | presentado 2017-03-29

Procesando: Cronos Group
  t-1: 2019 | 10-K | presentado 2020-03-02
  t-2: 2018 | 40-F | presentado 2019-03-26
  t-3: 2017 | 40-F | presentado 2018-04-30

Procesando: Koppers Holdings


/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


  t-1: 2021 | 10-K | presentado 2022-02-23
  t-2: 2020 | 10-K | presentado 2021-02-24
  t-3: 2019 | 10-K | presentado 2020-02-27

Procesando: Tupperware Brands
  t-1: 2020 | 10-K | presentado 2021-03-10
  t-2: 2019 | 10-K | presentado 2020-03-12
  t-3: 2018 | 10-K | presentado 2019-02-26

Procesando: Compass Minerals
  t-1: 2017 | 10-K | presentado 2018-02-27
  t-2: 2016 | 10-K | presentado 2017-03-01
  t-3: 2015 | 10-K | presentado 2016-02-22

Número total de observaciones: 45
                         Empresa Periodo  Ejercicio Formulario  Fecha_10K         t0
                     Belden Inc.     t-1       2016       10-K 2017-02-17 2018-02-01
                     Belden Inc.     t-2       2015       10-K 2016-02-25 2018-02-01
                     Belden Inc.     t-3       2014       10-K 2015-02-23 2018-02-01
                General Electric     t-1       2016       10-K 2017-02-24 2017-07-01
                General Electric     t-2       2015       10-K 2016-02-26 2017-07-01
       

/tmp/ipykernel_173/360792070.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


In [29]:
# ============================================================
# EXTRACCIÓN XBRL GENERAL PARA LA MUESTRA POSITIVA
# ============================================================

CONCEPTOS_CANDIDATOS = {
    "Revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueNet",
        "SalesRevenueGoodsNet",
        "Revenues"
    ],
    "Assets": [
        "Assets"
    ],
    "Equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"
    ],
    "NetIncome": [
        "NetIncomeLoss",
        "ProfitLoss"
    ],
    "Receivables": [
        "AccountsReceivableNetCurrent",
        "ReceivablesNetCurrent",
        "AccountsReceivableNet"
    ],
    "Inventory": [
        "InventoryNet"
    ],
    "Cash": [
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"
    ],
    "LiabilitiesCurrent": [
        "LiabilitiesCurrent"
    ]
}


def descargar_companyfacts(cik_empresa):
    url = (
        f"https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik_empresa}.json"
    )

    r = requests.get(url, headers=HEADERS)

    if r.status_code != 200:
        return None

    return r.json()


def buscar_concepto(namespace, alternativas):
    for concepto in alternativas:
        if concepto in namespace:
            return concepto
    return None


def extraer_valor_xbrl(
    namespace,
    concepto,
    ejercicio,
    t0_fecha,
    formularios_validos
):
    if concepto is None or concepto not in namespace:
        return None

    unidades = namespace[concepto].get("units", {})

    # Intentamos primero USD
    if "USD" not in unidades:
        return None

    df = pd.DataFrame(unidades["USD"])

    if df.empty:
        return None

    df = df[
        df["form"].isin(formularios_validos)
    ].copy()

    df["filed"] = pd.to_datetime(
        df["filed"],
        errors="coerce"
    )

    df["end"] = pd.to_datetime(
        df["end"],
        errors="coerce"
    )

    # Evitar data leakage
    df = df[df["filed"] < t0_fecha]

    # Ejercicio buscado
    df = df[
        df["end"].dt.year == ejercicio
    ]

    if df.empty:
        return None

    # Para conceptos de duración, priorizamos valores anuales
    if "start" in df.columns:
        df["start"] = pd.to_datetime(
            df["start"],
            errors="coerce"
        )

        df["dias"] = (
            df["end"] - df["start"]
        ).dt.days

        anuales = df[
            (df["dias"] >= 330) &
            (df["dias"] <= 380)
        ]

        if not anuales.empty:
            df = anuales

    # Preferimos el primer dato presentado
    # correspondiente al ejercicio
    df = df.sort_values("filed")

    return df.iloc[0]["val"]


print("Funciones XBRL preparadas correctamente.")

Funciones XBRL preparadas correctamente.


In [30]:
# ============================================================
# CONSTRUCCIÓN DEL DATASET FINANCIERO POSITIVO
# ============================================================

filas_financieras = []

for empresa_actual, grupo in tabla_periodos.groupby("Empresa"):

    print(f"\nProcesando XBRL: {empresa_actual}")

    fila_empresa = tabla_sec[
        tabla_sec["Empresa"] == empresa_actual
    ].iloc[0]

    cik_actual = fila_empresa["CIK"]
    t0_actual = pd.to_datetime(fila_empresa["t0"])

    companyfacts = descargar_companyfacts(cik_actual)

    if companyfacts is None:
        print("  ERROR: no se pudieron descargar Company Facts")
        continue

    # Puede haber us-gaap e ifrs-full
    facts_all = companyfacts.get("facts", {})

    namespace_usgaap = facts_all.get("us-gaap", {})
    namespace_ifrs = facts_all.get("ifrs-full", {})

    for _, obs in grupo.iterrows():

        ejercicio = int(obs["Ejercicio"])
        formulario = obs["Formulario"]

        # Elegimos namespace
        if formulario in ["20-F", "40-F"] and namespace_ifrs:
            namespace = namespace_ifrs
            namespace_usado = "ifrs-full"
        else:
            namespace = namespace_usgaap
            namespace_usado = "us-gaap"

        # Buscamos conceptos disponibles
        conceptos_usados = {}

        for variable, alternativas in CONCEPTOS_CANDIDATOS.items():
            conceptos_usados[variable] = buscar_concepto(
                namespace,
                alternativas
            )

        # Formularios que aceptamos para esa observación
        formularios_validos = [formulario]

        valores = {}

        for variable, concepto in conceptos_usados.items():
            valores[variable] = extraer_valor_xbrl(
                namespace=namespace,
                concepto=concepto,
                ejercicio=ejercicio,
                t0_fecha=t0_actual,
                formularios_validos=formularios_validos
            )

        # Pasivos totales = Assets - Equity
        liabilities_total = None

        if (
            valores["Assets"] is not None
            and valores["Equity"] is not None
        ):
            liabilities_total = (
                valores["Assets"] - valores["Equity"]
            )

        filas_financieras.append({
            "Empresa": empresa_actual,
            "Ticker": obs["Ticker"],
            "CIK": obs["CIK"],
            "SIC": obs["SIC"],
            "Periodo": obs["Periodo"],
            "Ejercicio": ejercicio,
            "Formulario": formulario,
            "Fecha_10K": obs["Fecha_10K"],
            "t0": obs["t0"],
            "Namespace": namespace_usado,
            "Revenue": valores["Revenue"],
            "Assets": valores["Assets"],
            "Liabilities_Total": liabilities_total,
            "Liabilities_Current": valores["LiabilitiesCurrent"],
            "NetIncome": valores["NetIncome"],
            "Receivables": valores["Receivables"],
            "Inventory": valores["Inventory"],
            "Cash": valores["Cash"],
            "Equity": valores["Equity"]
        })

        print(
            f"  {obs['Periodo']} - {ejercicio} - "
            f"{namespace_usado}"
        )

    time.sleep(0.15)


dataset_positivo = pd.DataFrame(filas_financieras)

print("\nNúmero de filas:", len(dataset_positivo))

print(
    dataset_positivo[
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Assets",
            "NetIncome",
            "Receivables",
            "Inventory"
        ]
    ].to_string(index=False)
)


Procesando XBRL: Bausch Health / Valeant
  t-1 - 2014 - us-gaap
  t-2 - 2013 - us-gaap
  t-3 - 2012 - us-gaap

Procesando XBRL: Belden Inc.
  t-1 - 2016 - us-gaap
  t-2 - 2015 - us-gaap
  t-3 - 2014 - us-gaap

Procesando XBRL: Compass Minerals
  t-1 - 2017 - us-gaap
  t-2 - 2016 - us-gaap
  t-3 - 2015 - us-gaap

Procesando XBRL: Cronos Group
  t-1 - 2019 - us-gaap
  t-2 - 2018 - ifrs-full
  t-3 - 2017 - ifrs-full

Procesando XBRL: General Electric
  t-1 - 2016 - us-gaap
  t-2 - 2015 - us-gaap
  t-3 - 2014 - us-gaap

Procesando XBRL: Healthcare Services Group
  t-1 - 2020 - us-gaap
  t-2 - 2019 - us-gaap
  t-3 - 2018 - us-gaap

Procesando XBRL: Koppers Holdings
  t-1 - 2021 - us-gaap
  t-2 - 2020 - us-gaap
  t-3 - 2019 - us-gaap

Procesando XBRL: Kraft Heinz
  t-1 - 2017 - us-gaap
  t-2 - 2016 - us-gaap
  t-3 - 2015 - us-gaap

Procesando XBRL: Manitex International
  t-1 - 2016 - us-gaap
  t-2 - 2015 - us-gaap
  t-3 - 2014 - us-gaap

Procesando XBRL: Pareteum
  t-1 - 2018 - us-gaap
  t

In [31]:
# ============================================================
# DIAGNÓSTICO DE COMPLETITUD DEL DATASET
# ============================================================

variables_financieras = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

print("COMPLETITUD POR VARIABLE")
print("=" * 60)

resumen_variables = pd.DataFrame({
    "Variable": variables_financieras,
    "Disponibles": [
        dataset_positivo[v].notna().sum()
        for v in variables_financieras
    ],
    "Faltantes": [
        dataset_positivo[v].isna().sum()
        for v in variables_financieras
    ]
})

resumen_variables["Completitud_%"] = (
    resumen_variables["Disponibles"]
    / len(dataset_positivo)
    * 100
).round(1)

print(resumen_variables.to_string(index=False))


print("\n\nDATOS FALTANTES POR EMPRESA")
print("=" * 60)

resumen_empresas = (
    dataset_positivo
    .groupby("Empresa")[variables_financieras]
    .apply(lambda x: x.isna().sum())
)

resumen_empresas["Total_Faltantes"] = (
    resumen_empresas.sum(axis=1)
)

resumen_empresas = resumen_empresas.sort_values(
    "Total_Faltantes",
    ascending=False
)

print(resumen_empresas.to_string())

COMPLETITUD POR VARIABLE
           Variable  Disponibles  Faltantes  Completitud_%
            Revenue           15         30           33.3
             Assets           41          4           91.1
  Liabilities_Total           41          4           91.1
Liabilities_Current           35         10           77.8
          NetIncome           39          6           86.7
        Receivables           37          8           82.2
          Inventory           32         13           71.1
               Cash           34         11           75.6
             Equity           41          4           91.1


DATOS FALTANTES POR EMPRESA
                                  Revenue  Assets  Liabilities_Total  Liabilities_Current  NetIncome  Receivables  Inventory  Cash  Equity  Total_Faltantes
Empresa                                                                                                                                                    
Cronos Group                            3  

In [32]:
# ============================================================
# DIAGNÓSTICO AUTOMÁTICO DE CONCEPTOS DE REVENUE
# ============================================================

empresas_sin_revenue = (
    dataset_positivo.loc[
        dataset_positivo["Revenue"].isna(),
        "Empresa"
    ]
    .drop_duplicates()
    .tolist()
)

print("Empresas a revisar:", len(empresas_sin_revenue))
print(empresas_sin_revenue)

for empresa_actual in empresas_sin_revenue:

    print("\n" + "=" * 90)
    print("EMPRESA:", empresa_actual)
    print("=" * 90)

    fila_empresa = tabla_sec[
        tabla_sec["Empresa"] == empresa_actual
    ].iloc[0]

    cik_actual = fila_empresa["CIK"]
    t0_actual = pd.to_datetime(fila_empresa["t0"])

    companyfacts = descargar_companyfacts(cik_actual)

    if companyfacts is None:
        print("ERROR descargando Company Facts")
        continue

    facts = companyfacts.get("facts", {})
    usgaap = facts.get("us-gaap", {})

    candidatos = [
        concepto
        for concepto in usgaap.keys()
        if (
            "revenue" in concepto.lower()
            or "sales" in concepto.lower()
        )
    ]

    ejercicios_empresa = (
        dataset_positivo.loc[
            dataset_positivo["Empresa"] == empresa_actual,
            "Ejercicio"
        ]
        .astype(int)
        .tolist()
    )

    encontrados = []

    for concepto in candidatos:

        unidades = usgaap[concepto].get("units", {})

        if "USD" not in unidades:
            continue

        df = pd.DataFrame(unidades["USD"])

        if df.empty:
            continue

        df["filed"] = pd.to_datetime(
            df["filed"],
            errors="coerce"
        )

        df["end"] = pd.to_datetime(
            df["end"],
            errors="coerce"
        )

        df = df[
            (df["form"] == "10-K") &
            (df["filed"] < t0_actual)
        ].copy()

        if df.empty:
            continue

        # Nos interesan solo nuestros ejercicios
        df = df[
            df["end"].dt.year.isin(ejercicios_empresa)
        ].copy()

        if df.empty:
            continue

        # Si existe start, calculamos duración
        if "start" in df.columns:

            df["start"] = pd.to_datetime(
                df["start"],
                errors="coerce"
            )

            df["dias"] = (
                df["end"] - df["start"]
            ).dt.days

            # Preferimos periodos aproximadamente anuales
            df_anual = df[
                (df["dias"] >= 330) &
                (df["dias"] <= 380)
            ].copy()

            if not df_anual.empty:
                df = df_anual

        anos_disponibles = sorted(
            df["end"].dt.year
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        encontrados.append(
            (
                concepto,
                anos_disponibles,
                len(df)
            )
        )

    if encontrados:

        encontrados = sorted(
            encontrados,
            key=lambda x: (
                -len(set(x[1]) & set(ejercicios_empresa)),
                x[0]
            )
        )

        for concepto, anos, n in encontrados[:15]:
            print(
                f"{concepto:65} "
                f"| años: {anos} | registros: {n}"
            )

    else:
        print("No se encontraron candidatos US-GAAP útiles.")

Empresas a revisar: 10
['Bausch Health / Valeant', 'Belden Inc.', 'Compass Minerals', 'Cronos Group', 'General Electric', 'Manitex International', 'Pareteum', 'Power Solutions International', 'Revolution Lighting Technologies', 'Super Micro Computer']

EMPRESA: Bausch Health / Valeant
BusinessAcquisitionsProFormaRevenue                               | años: [2012, 2013, 2014] | registros: 6
ProceedsFromSaleAndMaturityOfAvailableForSaleSecurities           | años: [2012, 2013, 2014] | registros: 6
Revenues                                                          | años: [2012, 2013, 2014] | registros: 6
RoyaltyRevenue                                                    | años: [2012, 2013, 2014] | registros: 6
SalesAndExciseTaxPayableCurrent                                   | años: [2012, 2013, 2014] | registros: 5
SalesRevenueGoodsNet                                              | años: [2012, 2013, 2014] | registros: 6
AvailableForSaleSecurities                                        

In [33]:
# ============================================================
# CORRECCIÓN DE REVENUE - CONCEPTOS VALIDADOS
# ============================================================

MAPEO_REVENUE = {
    "Bausch Health / Valeant": "SalesRevenueGoodsNet",
    "Belden Inc.": "SalesRevenueGoodsNet",
    "Compass Minerals": "SalesRevenueGoodsNet",
    "Manitex International": "SalesRevenueNet",
    "Pareteum": "Revenues",
    "Power Solutions International": "SalesRevenueGoodsNet",
    "Revolution Lighting Technologies": "SalesRevenueNet",
    "Super Micro Computer": "SalesRevenueNet"
}

for empresa_actual, concepto in MAPEO_REVENUE.items():

    print(f"\n{empresa_actual} -> {concepto}")

    fila_empresa = tabla_sec[
        tabla_sec["Empresa"] == empresa_actual
    ].iloc[0]

    cik_actual = fila_empresa["CIK"]
    t0_actual = pd.to_datetime(fila_empresa["t0"])

    companyfacts = descargar_companyfacts(cik_actual)

    usgaap = (
        companyfacts
        .get("facts", {})
        .get("us-gaap", {})
    )

    for idx in dataset_positivo[
        dataset_positivo["Empresa"] == empresa_actual
    ].index:

        ejercicio = int(
            dataset_positivo.loc[idx, "Ejercicio"]
        )

        formulario = dataset_positivo.loc[
            idx, "Formulario"
        ]

        valor = extraer_valor_xbrl(
            namespace=usgaap,
            concepto=concepto,
            ejercicio=ejercicio,
            t0_fecha=t0_actual,
            formularios_validos=[formulario]
        )

        dataset_positivo.loc[
            idx, "Revenue"
        ] = valor

        print(
            f"  {ejercicio}: "
            f"{valor:,.0f}"
            if valor is not None
            else f"  {ejercicio}: None"
        )


Bausch Health / Valeant -> SalesRevenueGoodsNet
  2014: 8,103,600,000
  2013: 5,640,333,000
  2012: 3,309,895,000

Belden Inc. -> SalesRevenueGoodsNet
  2016: 2,356,672,000
  2015: 2,309,222,000
  2014: 2,308,265,000

Compass Minerals -> SalesRevenueGoodsNet
  2017: 1,364,400,000
  2016: 1,138,000,000
  2015: 1,098,700,000

Manitex International -> SalesRevenueNet
  2016: 288,959,000
  2015: 386,737,000
  2014: 264,081,000

Pareteum -> Revenues
  2018: 32,435,736
  2017: 13,547,507
  2016: 12,855,811

Power Solutions International -> SalesRevenueGoodsNet
  2015: 389,446,000
  2014: 347,995,000
  2013: 237,842,000

Revolution Lighting Technologies -> SalesRevenueNet
  2017: 152,312,000
  2016: 172,121,000
  2015: 129,656,000

Super Micro Computer -> SalesRevenueNet
  2016: 2,215,573,000
  2015: 1,991,155,000
  2014: 1,467,202,000


In [34]:
# ============================================================
# DIAGNÓSTICO ESPECÍFICO DE REVENUE - GENERAL ELECTRIC
# ============================================================

empresa_actual = "General Electric"

fila_empresa = tabla_sec[
    tabla_sec["Empresa"] == empresa_actual
].iloc[0]

cik_actual = fila_empresa["CIK"]
t0_actual = pd.to_datetime(fila_empresa["t0"])

companyfacts = descargar_companyfacts(cik_actual)

usgaap = (
    companyfacts
    .get("facts", {})
    .get("us-gaap", {})
)

# Conceptos potencialmente relacionados con ingresos
palabras_clave = [
    "Revenue",
    "Revenues",
    "SalesRevenue",
    "SalesOf",
    "OperatingRevenue"
]

candidatos = [
    concepto
    for concepto in usgaap.keys()
    if any(
        palabra.lower() in concepto.lower()
        for palabra in palabras_clave
    )
]

resultados = []

for concepto in candidatos:

    unidades = usgaap[concepto].get("units", {})

    if "USD" not in unidades:
        continue

    df = pd.DataFrame(unidades["USD"])

    if df.empty:
        continue

    df["filed"] = pd.to_datetime(
        df["filed"],
        errors="coerce"
    )

    df["end"] = pd.to_datetime(
        df["end"],
        errors="coerce"
    )

    df = df[
        (df["form"] == "10-K") &
        (df["filed"] < t0_actual) &
        (df["end"].dt.year.isin([2014, 2015, 2016]))
    ].copy()

    if df.empty:
        continue

    if "start" in df.columns:

        df["start"] = pd.to_datetime(
            df["start"],
            errors="coerce"
        )

        df["dias"] = (
            df["end"] - df["start"]
        ).dt.days

        anual = df[
            (df["dias"] >= 330) &
            (df["dias"] <= 380)
        ]

        if not anual.empty:
            df = anual

    resultados.append({
        "Concepto": concepto,
        "Años": sorted(
            df["end"]
            .dt.year
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        ),
        "Registros": len(df),
        "Valor_max": df["val"].max()
    })


resultado_ge = pd.DataFrame(resultados)

if not resultado_ge.empty:

    resultado_ge = resultado_ge.sort_values(
        "Valor_max",
        ascending=False
    )

    print(
        resultado_ge.head(30).to_string(index=False)
    )

else:
    print("No se encontraron candidatos.")

                                          Concepto               Años  Registros    Valor_max
                                          Revenues [2014, 2015, 2016]          6 148589000000
                                   SalesRevenueNet [2014, 2015, 2016]          3 110391000000
                              SalesRevenueGoodsNet [2014, 2015, 2016]          6  76568000000
                          FinancialServicesRevenue [2014, 2015, 2016]          6  41053000000
                           SalesRevenueServicesNet [2014, 2015, 2016]          6  34976000000
DisposalGroupIncludingDiscontinuedOperationRevenue [2014, 2015, 2016]          6  31136000000
  GainLossOnSalesOfAssetsAndAssetImpairmentCharges       [2014, 2015]          2    127000000
      ProceedsFromSalesOfAssetsInvestingActivities       [2014, 2015]          3            0


In [35]:
# ============================================================
# CORRECCIÓN REVENUE - GENERAL ELECTRIC
# ============================================================

empresa_actual = "General Electric"
concepto_ge = "Revenues"

fila_empresa = tabla_sec[
    tabla_sec["Empresa"] == empresa_actual
].iloc[0]

companyfacts = descargar_companyfacts(
    fila_empresa["CIK"]
)

usgaap = (
    companyfacts
    .get("facts", {})
    .get("us-gaap", {})
)

t0_actual = pd.to_datetime(
    fila_empresa["t0"]
)

for idx in dataset_positivo[
    dataset_positivo["Empresa"] == empresa_actual
].index:

    ejercicio = int(
        dataset_positivo.loc[idx, "Ejercicio"]
    )

    valor = extraer_valor_xbrl(
        namespace=usgaap,
        concepto=concepto_ge,
        ejercicio=ejercicio,
        t0_fecha=t0_actual,
        formularios_validos=["10-K"]
    )

    dataset_positivo.loc[
        idx, "Revenue"
    ] = valor

    print(
        f"{ejercicio}: "
        f"{valor:,.0f}"
        if valor is not None
        else f"{ejercicio}: None"
    )

print(
    "\nRevenue disponibles:",
    dataset_positivo["Revenue"].notna().sum(),
    "/ 45"
)


2016: 123,693,000,000
2015: 117,386,000,000
2014: 148,589,000,000

Revenue disponibles: 42 / 45


In [36]:
# ============================================================
# DIAGNÓSTICO REVENUE - CRONOS GROUP
# ============================================================

empresa_actual = "Cronos Group"

fila_empresa = tabla_sec[
    tabla_sec["Empresa"] == empresa_actual
].iloc[0]

companyfacts = descargar_companyfacts(
    fila_empresa["CIK"]
)

facts = companyfacts.get("facts", {})

print("Namespaces disponibles:")
print(list(facts.keys()))

# Revisamos todos los namespaces disponibles
for nombre_namespace, namespace in facts.items():

    print("\n" + "=" * 90)
    print("NAMESPACE:", nombre_namespace)
    print("=" * 90)

    candidatos = [
        concepto
        for concepto in namespace.keys()
        if (
            "revenue" in concepto.lower()
            or "sales" in concepto.lower()
            or "income" in concepto.lower()
        )
    ]

    if not candidatos:
        print("No hay candidatos.")
        continue

    encontrados = []

    for concepto in candidatos:

        unidades = namespace[concepto].get("units", {})

        for unidad, registros in unidades.items():

            df = pd.DataFrame(registros)

            if df.empty or "end" not in df.columns:
                continue

            df["end"] = pd.to_datetime(
                df["end"],
                errors="coerce"
            )

            if "filed" in df.columns:
                df["filed"] = pd.to_datetime(
                    df["filed"],
                    errors="coerce"
                )

            df = df[
                df["end"].dt.year.isin(
                    [2017, 2018, 2019]
                )
            ].copy()

            if df.empty:
                continue

            encontrados.append({
                "Concepto": concepto,
                "Unidad": unidad,
                "Años": sorted(
                    df["end"]
                    .dt.year
                    .dropna()
                    .astype(int)
                    .unique()
                    .tolist()
                ),
                "Registros": len(df)
            })

    if encontrados:

        resultado = pd.DataFrame(encontrados)

        resultado = resultado.sort_values(
            ["Concepto", "Unidad"]
        )

        print(
            resultado.to_string(index=False)
        )

    else:
        print(
            "No hay conceptos con datos "
            "para 2017-2019."
        )

Namespaces disponibles:
['dei', 'ifrs-full', 'us-gaap', 'srt', 'ecd']

NAMESPACE: dei
No hay candidatos.

NAMESPACE: ifrs-full
                                                                               Concepto Unidad         Años  Registros
                                                    AccumulatedOtherComprehensiveIncome    CAD [2017, 2018]          3
                                                                    ComprehensiveIncome    CAD [2017, 2018]          3
                               ComprehensiveIncomeAttributableToNoncontrollingInterests    CAD       [2018]          1
                                        ComprehensiveIncomeAttributableToOwnersOfParent    CAD [2017, 2018]          2
                              CurrentPayablesOnSocialSecurityAndTaxesOtherThanIncomeTax    CAD [2017, 2018]          2
                                          CurrentReceivablesFromTaxesOtherThanIncomeTax    CAD [2017, 2018]          3
                                        

In [37]:
# ============================================================
# EXTRACCIÓN REVENUE - CRONOS GROUP
# ============================================================

empresa_actual = "Cronos Group"

fila_empresa = tabla_sec[
    tabla_sec["Empresa"] == empresa_actual
].iloc[0]

companyfacts = descargar_companyfacts(
    fila_empresa["CIK"]
)

facts = companyfacts["facts"]

usgaap = facts.get("us-gaap", {})
ifrs = facts.get("ifrs-full", {})

t0_actual = pd.to_datetime(
    fila_empresa["t0"]
)


def mostrar_valor_cronos(namespace, concepto, unidad, ejercicio, formulario):

    registros = namespace[
        concepto
    ]["units"][unidad]

    df = pd.DataFrame(registros)

    df["end"] = pd.to_datetime(
        df["end"],
        errors="coerce"
    )

    df["filed"] = pd.to_datetime(
        df["filed"],
        errors="coerce"
    )

    df = df[
        (df["form"] == formulario) &
        (df["filed"] < t0_actual) &
        (df["end"].dt.year == ejercicio)
    ].copy()

    if "start" in df.columns:

        df["start"] = pd.to_datetime(
            df["start"],
            errors="coerce"
        )

        df["dias"] = (
            df["end"] - df["start"]
        ).dt.days

        anual = df[
            (df["dias"] >= 330) &
            (df["dias"] <= 380)
        ]

        if not anual.empty:
            df = anual

    df = df.sort_values("filed")

    print("\n" + "=" * 70)
    print(
        f"{ejercicio} | {formulario} | "
        f"{concepto} | {unidad}"
    )
    print("=" * 70)

    columnas = [
        c for c in
        ["start", "end", "val", "form", "filed", "accn"]
        if c in df.columns
    ]

    print(
        df[columnas].to_string(index=False)
    )


# 2019 - US GAAP / USD
mostrar_valor_cronos(
    usgaap,
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "USD",
    2019,
    "10-K"
)

# 2018 - IFRS / CAD
mostrar_valor_cronos(
    ifrs,
    "Revenue",
    "CAD",
    2018,
    "40-F"
)

# 2017 - IFRS / CAD
mostrar_valor_cronos(
    ifrs,
    "Revenue",
    "CAD",
    2017,
    "40-F"
)


2019 | 10-K | RevenueFromContractWithCustomerExcludingAssessedTax | USD
Empty DataFrame
Columns: [start, end, val, form, filed, accn]
Index: []

2018 | 40-F | Revenue | CAD
     start        end      val form      filed                 accn
2018-01-01 2018-12-31 15703000 40-F 2019-03-26 0001193125-19-085847

2017 | 40-F | Revenue | CAD
     start        end     val form      filed                 accn
2017-01-01 2017-12-31 4082000 40-F 2018-04-30 0001193125-18-140678
2017-01-01 2017-12-31 4082000 40-F 2019-03-26 0001193125-19-085847


In [38]:
# ============================================================
# CRONOS 2019 - INSPECCIÓN COMPLETA DEL CONCEPTO REVENUE
# ============================================================

concepto = "RevenueFromContractWithCustomerExcludingAssessedTax"

registros = usgaap[
    concepto
]["units"]["USD"]

df_cronos_2019 = pd.DataFrame(registros)

columnas = [
    c for c in [
        "start",
        "end",
        "val",
        "accn",
        "fy",
        "fp",
        "form",
        "filed",
        "frame"
    ]
    if c in df_cronos_2019.columns
]

print(
    df_cronos_2019[columnas]
    .to_string(index=False)
)

     start        end       val                 accn   fy fp   form      filed    frame
2017-01-01 2017-12-31   3147000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30   CY2017
2018-01-01 2018-03-31   2329000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30 CY2018Q1
2018-04-01 2018-06-30   2630000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30 CY2018Q2
2018-07-01 2018-09-30   2877000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30 CY2018Q3
2018-01-01 2018-12-31  12121000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30      NaN
2018-01-01 2018-12-31  12121000 0001656472-21-000009 2020 FY   10-K 2021-02-26   CY2018
2018-10-01 2018-12-31   4285000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30 CY2018Q4
2019-01-01 2019-03-31   3004000 0001656472-20-000033 2019 FY 10-K/A 2020-03-30      NaN
2019-01-01 2019-03-31   3004000 0001656472-20-000052 2020 Q1   10-Q 2020-05-08 CY2019Q1
2019-01-01 2019-06-30  10657000 0001656472-20-000073 2020 Q2   10-Q 2020-08-06      NaN
2019-04-01 2019-06-30   7653000 

In [40]:
# ============================================================
# CRONOS 2019 - COMPROBAR SI EL 10-K ORIGINAL
# TIENE DATOS EN COMPANYFACTS
# ============================================================

accession_original = "0001656472-20-000011"

registros_originales = []

for nombre_namespace, namespace in facts.items():

    for concepto, info in namespace.items():

        unidades = info.get("units", {})

        for unidad, registros in unidades.items():

            df = pd.DataFrame(registros)

            if df.empty or "accn" not in df.columns:
                continue

            df_filtrado = df[
                df["accn"] == accession_original
            ].copy()

            if df_filtrado.empty:
                continue

            for _, reg in df_filtrado.iterrows():

                registros_originales.append({
                    "Namespace": nombre_namespace,
                    "Concepto": concepto,
                    "Etiqueta": info.get("label"),
                    "Unidad": unidad,
                    "Start": reg.get("start"),
                    "End": reg.get("end"),
                    "Valor": reg.get("val"),
                    "Form": reg.get("form"),
                    "Filed": reg.get("filed")
                })


cron_original = pd.DataFrame(registros_originales)

print("Número de registros encontrados:", len(cron_original))

if cron_original.empty:
    print(
        "El 10-K original de Cronos no aparece desglosado "
        "en Company Facts."
    )
else:
    print(
        cron_original[
            [
                "Namespace",
                "Concepto",
                "Etiqueta",
                "Unidad",
                "Start",
                "End",
                "Valor"
            ]
        ].head(100).to_string(index=False)
    )

Número de registros encontrados: 0
El 10-K original de Cronos no aparece desglosado en Company Facts.


In [41]:
# ============================================================
# CRONOS 2019 - DESCARGAR 10-K ORIGINAL
# ============================================================

cik_cronos = "0001656472"
accession_original = "0001656472-20-000011"
documento_original = "cronosdraftfy20193220filin.htm"

cik_sin_ceros = str(int(cik_cronos))
accession_sin_guiones = accession_original.replace("-", "")

url_cronos_2019 = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{cik_sin_ceros}/"
    f"{accession_sin_guiones}/"
    f"{documento_original}"
)

print("URL:")
print(url_cronos_2019)

r = requests.get(url_cronos_2019, headers=HEADERS)

print("\nStatus code:", r.status_code)
print("Tamaño del documento:", len(r.text))

URL:
https://www.sec.gov/Archives/edgar/data/1656472/000165647220000011/cronosdraftfy20193220filin.htm

Status code: 200
Tamaño del documento: 618441


In [42]:
# ============================================================
# CRONOS 2019 - BUSCAR REVENUE EN EL 10-K ORIGINAL
# ============================================================

from bs4 import BeautifulSoup
import re

soup = BeautifulSoup(r.text, "html.parser")

# Extraemos todo el texto del filing
texto_cronos = soup.get_text(" ", strip=True)

# Buscamos fragmentos alrededor de términos relacionados con ingresos
patrones = [
    "net revenue",
    "total revenue",
    "revenue",
    "net revenues",
    "sales"
]

for patron in patrones:
    print("\n" + "=" * 90)
    print("PATRÓN:", patron)
    print("=" * 90)

    coincidencias = list(
        re.finditer(
            patron,
            texto_cronos,
            flags=re.IGNORECASE
        )
    )

    print("Coincidencias encontradas:", len(coincidencias))

    # Mostramos solo las primeras 10
    for match in coincidencias[:10]:

        inicio = max(
            0,
            match.start() - 250
        )

        fin = min(
            len(texto_cronos),
            match.end() + 350
        )

        fragmento = texto_cronos[
            inicio:fin
        ]

        print("\n---")
        print(fragmento)


PATRÓN: net revenue
Coincidencias encontradas: 1

---
ntario Cannabis Store (the cannabis control authority and sole wholesaler and distributor of cannabis in Ontario), Radient Technologies Inc. and MediPharm, sales to each of which are expected to equal or exceed 10% of the Company’s consolidated 2019 net revenues. The Company’s arrangement with MediPharm is described above. We mitigate credit risk through verification of the customers’ liquidity prior to the authorization of material transactions. Government Contracts In Canada, we sell cannabis and cannabis products to cannabis control authorities in various provinces, including, Ontario, Québec, British

PATRÓN: total revenue
Coincidencias encontradas: 0

PATRÓN: revenue
Coincidencias encontradas: 20

---
s in cannabinoids, or the success thereof; • expectations regarding acquisitions and the anticipated benefits therefrom, including the Redwood Acquisition and the acquisition of certain assets from AFI (as defined herein); 1 • ex

In [43]:
# ============================================================
# CRONOS 2019 - BUSCAR CONCEPTOS INLINE XBRL DE REVENUE
# EN EL 10-K ORIGINAL DEL 02/03/2020
# ============================================================

from bs4 import BeautifulSoup
import re

soup = BeautifulSoup(r.text, "html.parser")

hechos_revenue = []

for tag in soup.find_all(True):

    nombre_concepto = tag.get("name")

    if not nombre_concepto:
        continue

    if not re.search(
        r"revenue|sales",
        nombre_concepto,
        flags=re.IGNORECASE
    ):
        continue

    texto_valor = tag.get_text(
        " ",
        strip=True
    )

    hechos_revenue.append({
        "Concepto": nombre_concepto,
        "ContextRef": tag.get("contextref"),
        "UnitRef": tag.get("unitref"),
        "Scale": tag.get("scale"),
        "Sign": tag.get("sign"),
        "Valor_texto": texto_valor
    })


hechos_revenue_df = pd.DataFrame(hechos_revenue)

print(
    "Número de hechos relacionados con Revenue/Sales:",
    len(hechos_revenue_df)
)

if not hechos_revenue_df.empty:

    print(
        hechos_revenue_df
        .drop_duplicates()
        .to_string(index=False)
    )
else:
    print(
        "No se localizaron conceptos Revenue/Sales "
        "mediante etiquetas Inline XBRL."
    )

Número de hechos relacionados con Revenue/Sales: 0
No se localizaron conceptos Revenue/Sales mediante etiquetas Inline XBRL.


In [44]:
# ============================================================
# CRONOS 2019 - LISTAR ARCHIVOS DEL FILING ORIGINAL
# ============================================================

cik_sin_ceros = str(int(cik_cronos))
accession_sin_guiones = accession_original.replace("-", "")

url_index_json = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{cik_sin_ceros}/"
    f"{accession_sin_guiones}/index.json"
)

r_index = requests.get(
    url_index_json,
    headers=HEADERS
)

print("Status code:", r_index.status_code)

index_data = r_index.json()

archivos = index_data["directory"]["item"]

archivos_df = pd.DataFrame(archivos)

print(
    archivos_df[
        ["name", "size", "type"]
    ].to_string(index=False)
)


Status code: 200
                                   name   size       type
0001656472-20-000011-index-headers.html          text.gif
        0001656472-20-000011-index.html          text.gif
               0001656472-20-000011.txt          text.gif
                              a1010.htm 139047   text.gif
                              a1011.htm   2819   text.gif
                              a1012.htm 120478   text.gif
                              a1013.htm  75789   text.gif
                              a1014.htm 121128   text.gif
                              a1015.htm 102421   text.gif
                              a1016.htm  16075   text.gif
                              a1017.htm  99406   text.gif
                              a1018.htm 121589   text.gif
                              a1019.htm 121855   text.gif
                              a1020.htm  51721   text.gif
                              a1021.htm  76904   text.gif
                              a1022.htm  48028   text.g

In [45]:
# ============================================================
# CRONOS 2019 - BUSCAR TABLAS FINANCIERAS EN EL 10-K ORIGINAL
# ============================================================

import pandas as pd

# Extraer todas las tablas HTML
tablas_cronos = pd.read_html(r.text)

print("Número total de tablas:", len(tablas_cronos))

candidatas = []

for i, tabla in enumerate(tablas_cronos):

    # Convertimos toda la tabla a texto para buscar términos
    texto_tabla = " ".join(
        tabla.astype(str)
        .fillna("")
        .values
        .flatten()
    )

    palabras = [
        "revenue",
        "net revenue",
        "net revenues",
        "statement of operations",
        "statements of operations",
        "statement of income",
        "statements of income"
    ]

    if any(
        palabra.lower() in texto_tabla.lower()
        for palabra in palabras
    ):
        candidatas.append(i)


print("\nTablas candidatas:", candidatas)

for i in candidatas:

    print("\n" + "=" * 100)
    print("TABLA", i)
    print("=" * 100)

    print(
        tablas_cronos[i]
        .to_string(index=False)
    )

/tmp/ipykernel_173/3576003546.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas_cronos = pd.read_html(r.text)


Número total de tablas: 102

Tablas candidatas: [25, 70]

TABLA 25
  0                                                                     1
NaN                                                                   NaN
  • expectations regarding revenues, expenses and anticipated cash needs;

TABLA 70
  0                                                                                                                                                                                                                                                                                      1
NaN                                                                                                                                                                                                                                                                                    NaN
  • the profits or revenues derived therefrom could be subject to money laundering statutes, including the Money Laundering Control Act

In [48]:
# ============================================================
# CRONOS 2019 - BUSCAR ESTADOS FINANCIEROS EN LOS ANEXOS HTML
# ============================================================

from bs4 import BeautifulSoup
import requests
import re
import time

base_url = (
    "https://www.sec.gov/Archives/edgar/data/"
    f"{cik_sin_ceros}/{accession_sin_guiones}/"
)

archivos_htm = [
    nombre
    for nombre in archivos_df["name"]
    if str(nombre).lower().endswith((".htm", ".html"))
]

resultados_busqueda = []

patrones = [
    "consolidated statements of operations",
    "consolidated statements of income",
    "consolidated statements of comprehensive",
    "net revenue",
    "net revenues",
    "revenue",
    "net income",
    "net loss"
]

for nombre_archivo in archivos_htm:

    url_archivo = base_url + nombre_archivo

    resp = requests.get(
        url_archivo,
        headers=HEADERS
    )

    if resp.status_code != 200:
        continue

    soup_archivo = BeautifulSoup(
        resp.text,
        "html.parser"
    )

    texto = soup_archivo.get_text(
        " ",
        strip=True
    )

    encontrados = [
        patron
        for patron in patrones
        if re.search(
            re.escape(patron),
            texto,
            flags=re.IGNORECASE
        )
    ]

    if encontrados:

        resultados_busqueda.append({
            "Archivo": nombre_archivo,
            "Tamaño": len(resp.text),
            "Patrones": ", ".join(encontrados)
        })

    time.sleep(0.1)


resultado_anexos = pd.DataFrame(
    resultados_busqueda
)

print("ARCHIVOS CANDIDATOS")
print("=" * 100)

if resultado_anexos.empty:

    print("No se encontraron archivos candidatos.")

else:

    print(
        resultado_anexos
        .sort_values("Tamaño", ascending=False)
        .to_string(index=False)
    )
    

ARCHIVOS CANDIDATOS
                       Archivo  Tamaño                                       Patrones
cronosdraftfy20193220filin.htm  618441 net revenue, net revenues, revenue, net income
                      a108.htm  232146                                        revenue
                     a1019.htm  121855                                        revenue
                     a1018.htm  121589                                        revenue
                     a1014.htm  121128                                        revenue
                     a1012.htm  120478                                        revenue
                      a109.htm  112401                                        revenue
                     a1015.htm  102421                                        revenue
                     a1021.htm   76904                                        revenue
                     a1020.htm   51721                                        revenue
                       a42.htm   3

In [49]:
from io import StringIO

candidatos_archivos = [
    "a108.htm",
    "a1019.htm",
    "a1018.htm",
    "a1014.htm",
    "a1012.htm",
    "a109.htm",
    "a1015.htm",
    "a1021.htm",
    "a1020.htm",
    "a42.htm"
]

for archivo in candidatos_archivos:

    print("\n" + "=" * 100)
    print("ARCHIVO:", archivo)
    print("=" * 100)

    url = base_url + archivo

    try:
        resp = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        )

        if resp.status_code != 200:
            print("Error HTTP:", resp.status_code)
            continue

        # Texto plano
        soup_archivo = BeautifulSoup(
            resp.text,
            "html.parser"
        )

        texto = soup_archivo.get_text(
            " ",
            strip=True
        )

        # Fragmento alrededor de "revenue"
        match = re.search(
            r"revenue",
            texto,
            flags=re.IGNORECASE
        )

        if match:
            inicio = max(0, match.start() - 250)
            fin = min(len(texto), match.end() + 500)

            print("\nFRAGMENTO:")
            print(texto[inicio:fin])

        # Intentamos detectar tablas
        try:
            tablas = pd.read_html(
                StringIO(resp.text)
            )

            print("\nNúmero de tablas:", len(tablas))

            for i, tabla in enumerate(tablas):

                texto_tabla = " ".join(
                    tabla.astype(str)
                    .fillna("")
                    .values
                    .flatten()
                )

                if (
                    "2019" in texto_tabla
                    and "2018" in texto_tabla
                    and (
                        "revenue" in texto_tabla.lower()
                        or "net income" in texto_tabla.lower()
                        or "net loss" in texto_tabla.lower()
                    )
                ):

                    print(
                        f"\n>>> TABLA FINANCIERA CANDIDATA "
                        f"{i}"
                    )

                    print(
                        tabla.to_string(index=False)
                    )

        except Exception as e:
            print("Sin tablas útiles:", e)

    except Exception as e:
        print("ERROR:", e)
        


ARCHIVO: a108.htm

FRAGMENTO:
purposes of this paragraph, an “ Incumbent Director ” shall mean any member of the Board who is a member of the Board immediately prior to the occurrence of a contested election of directors of the Company). “ Code ” means the United States Internal Revenue Code of 1986, as amended, and any applicable United States Treasury Regulations and other binding regulatory guidance thereunder. “ Committee ” means the Compensation Committee of the Board, or such other committee of the Board as is designated by the Board, by way of resolution, adoption of a policy or committee mandate, or otherwise, to administer the Plan from time to time. “ Company ” means Cronos Group Inc. and includes any successor corporation thereto. “ Exercise Notice ” means a notice 

Número de tablas: 105

ARCHIVO: a1019.htm

FRAGMENTO:
ll taxes that are required to be withheld pursuant to any applicable law or regulation. 8.5 Section 409A Compliance .  To the extent applicable, this Agreem

In [52]:
# Ver los DataFrames que tenemos actualmente en memoria
# usando una copia de globals() para evitar el RuntimeError

for nombre, objeto in list(globals().items()):
    if isinstance(objeto, pd.DataFrame):
        print(
            nombre,
            "| filas:", len(objeto),
            "| columnas:", list(objeto.columns)
        )

filings | filas: 1000 | columnas: ['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber', 'items', 'core_type', 'size', 'isXBRL', 'isInlineXBRL', 'isXBRLNumeric', 'primaryDocument', 'primaryDocDescription']
filings_10k | filas: 3 | columnas: ['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber', 'items', 'core_type', 'size', 'isXBRL', 'isInlineXBRL', 'isXBRLNumeric', 'primaryDocument', 'primaryDocDescription']
resultado_10k | filas: 3 | columnas: ['filingDate', 'reportDate', 'accessionNumber', 'primaryDocument']
belden_filings | filas: 3 | columnas: ['filingDate', 'reportDate', 'accessionNumber', 'primaryDocument', 'url_10k', 'periodo']
belden_financials | filas: 3 | columnas: ['Empresa', 'Año', 'Revenue', 'Assets', 'Liabilities_Total', 'Liabilities_Current', 'NetIncome', 'Receivables', 'Inventory', 'Cash', 'Equity']
revenue_df | filas: 105 | columnas: ['start', 'end', 'va

In [53]:
# ============================================================
# MARCAR CRONOS COMO CASO ESPECIAL PARA EL MODELO FINANCIERO
# ============================================================

dataset_positivo["Usar_Modelo_Financiero"] = True
dataset_positivo["Motivo_Exclusion_Financiera"] = None

mascara_cronos = (
    dataset_positivo["Empresa"] == "Cronos Group"
)

dataset_positivo.loc[
    mascara_cronos,
    "Usar_Modelo_Financiero"
] = False

dataset_positivo.loc[
    mascara_cronos,
    "Motivo_Exclusion_Financiera"
] = (
    "Cambio IFRS/US-GAAP y CAD/USD; "
    "datos pre-t0 no comparables de forma homogénea"
)

print(
    dataset_positivo.loc[
        mascara_cronos,
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Namespace",
            "Usar_Modelo_Financiero",
            "Motivo_Exclusion_Financiera"
        ]
    ].to_string(index=False)
)

     Empresa Periodo  Ejercicio  Revenue Namespace  Usar_Modelo_Financiero                                                   Motivo_Exclusion_Financiera
Cronos Group     t-1       2019      NaN   us-gaap                   False Cambio IFRS/US-GAAP y CAD/USD; datos pre-t0 no comparables de forma homogénea
Cronos Group     t-2       2018      NaN ifrs-full                   False Cambio IFRS/US-GAAP y CAD/USD; datos pre-t0 no comparables de forma homogénea
Cronos Group     t-3       2017      NaN ifrs-full                   False Cambio IFRS/US-GAAP y CAD/USD; datos pre-t0 no comparables de forma homogénea


In [54]:
# ============================================================
# COMPLETITUD DEL DATASET FINANCIERO SIN CRONOS
# ============================================================

dataset_financiero = dataset_positivo[
    dataset_positivo["Usar_Modelo_Financiero"] == True
].copy()

variables_financieras = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

resumen_financiero = pd.DataFrame({
    "Variable": variables_financieras,
    "Disponibles": [
        dataset_financiero[v].notna().sum()
        for v in variables_financieras
    ],
    "Faltantes": [
        dataset_financiero[v].isna().sum()
        for v in variables_financieras
    ]
})

resumen_financiero["Completitud_%"] = (
    resumen_financiero["Disponibles"]
    / len(dataset_financiero)
    * 100
).round(1)

print("Número de observaciones financieras:", len(dataset_financiero))
print()
print(resumen_financiero.to_string(index=False))

Número de observaciones financieras: 42

           Variable  Disponibles  Faltantes  Completitud_%
            Revenue           42          0          100.0
             Assets           41          1           97.6
  Liabilities_Total           41          1           97.6
Liabilities_Current           35          7           83.3
          NetIncome           39          3           92.9
        Receivables           37          5           88.1
          Inventory           32         10           76.2
               Cash           34          8           81.0
             Equity           41          1           97.6


In [55]:
# ============================================================
# LOCALIZAR TODOS LOS DATOS FINANCIEROS FALTANTES
# ============================================================

variables_financieras = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

for variable in variables_financieras:

    faltantes = dataset_financiero[
        dataset_financiero[variable].isna()
    ][
        ["Empresa", "Periodo", "Ejercicio", "Formulario", variable]
    ]

    if len(faltantes) > 0:

        print("\n" + "=" * 85)
        print(
            f"{variable.upper()} "
            f"- {len(faltantes)} FALTANTES"
        )
        print("=" * 85)

        print(
            faltantes.to_string(index=False)
        )


ASSETS - 1 FALTANTES
    Empresa Periodo  Ejercicio Formulario  Assets
Kraft Heinz     t-3       2015       10-K     NaN

LIABILITIES_TOTAL - 1 FALTANTES
    Empresa Periodo  Ejercicio Formulario  Liabilities_Total
Kraft Heinz     t-3       2015       10-K                NaN

LIABILITIES_CURRENT - 7 FALTANTES
         Empresa Periodo  Ejercicio Formulario  Liabilities_Current
General Electric     t-1       2016       10-K                  NaN
General Electric     t-2       2015       10-K                  NaN
General Electric     t-3       2014       10-K                  NaN
     Kraft Heinz     t-3       2015       10-K                  NaN
   VEREIT / ARCP     t-1       2013       10-K                  NaN
   VEREIT / ARCP     t-2       2012       10-K                  NaN
   VEREIT / ARCP     t-3       2011       10-K                  NaN

NETINCOME - 3 FALTANTES
                Empresa Periodo  Ejercicio Formulario  NetIncome
Bausch Health / Valeant     t-1       2014       10-K 

In [57]:
resultados_kraft = []

for concepto, info in usgaap_kraft.items():

    etiqueta = info.get("label") or ""

    if any(
        palabra.lower() in concepto.lower()
        or palabra.lower() in etiqueta.lower()
        for palabra in palabras
    ):

        unidades = info.get("units", {})

        for unidad, registros in unidades.items():

            df_temp = pd.DataFrame(registros)

            if df_temp.empty or "end" not in df_temp.columns:
                continue

            df_temp["end"] = pd.to_datetime(
                df_temp["end"],
                errors="coerce"
            )

            df_2015 = df_temp[
                df_temp["end"].dt.year == 2015
            ].copy()

            if "form" in df_2015.columns:
                df_2015 = df_2015[
                    df_2015["form"]
                    .astype(str)
                    .str.contains("10-K", na=False)
                ]

            if not df_2015.empty:

                resultados_kraft.append({
                    "Concepto": concepto,
                    "Etiqueta": etiqueta,
                    "Unidad": unidad,
                    "Registros": len(df_2015),
                    "Valor_max": df_2015["val"].max()
                })

resultado_kraft = pd.DataFrame(resultados_kraft)

print("\nCONCEPTOS CANDIDATOS KRAFT HEINZ 2015")
print("=" * 100)

if resultado_kraft.empty:
    print("No se encontraron conceptos.")
else:
    print(
        resultado_kraft
        .sort_values(
            ["Concepto", "Valor_max"],
            ascending=[True, False]
        )
        .to_string(index=False)
    )


CONCEPTOS CANDIDATOS KRAFT HEINZ 2015
                                                                                                     Concepto                                                                                                                      Etiqueta Unidad  Registros   Valor_max
BusinessCombinationRecognizedIdentifiableAssetsAcquiredAndLiabilitiesAssumedIntangibleAssetsOtherThanGoodwill Business Combination, Recognized Identifiable Assets Acquired and Liabilities Assumed, Intangible Assets, Other than Goodwill    USD          2 45100000000
        BusinessCombinationRecognizedIdentifiableAssetsAcquiredAndLiabilitiesAssumedPropertyPlantAndEquipment         Business Combination, Recognized Identifiable Assets Acquired and Liabilities Assumed, Property, Plant, and Equipment    USD          1  4200000000
                                                 ImpairmentOfIntangibleAssetsIndefinitelivedExcludingGoodwill                                                      

In [58]:
# ============================================================
# KRAFT HEINZ - INSPECCIONAR CONCEPTOS CONTABLES PRINCIPALES
# ============================================================

conceptos_buscar = [
    "Assets",
    "Liabilities",
    "LiabilitiesCurrent",
    "StockholdersEquity",
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "CashAndCashEquivalentsAtCarryingValue",
    "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
    "InventoryNet",
    "AccountsReceivableNetCurrent",
    "ReceivablesNetCurrent"
]

for concepto in conceptos_buscar:

    print("\n" + "=" * 100)
    print("CONCEPTO:", concepto)
    print("=" * 100)

    if concepto not in usgaap_kraft:
        print("NO EXISTE")
        continue

    unidades = usgaap_kraft[concepto].get("units", {})

    if "USD" not in unidades:
        print("No hay unidad USD.")
        print("Unidades disponibles:", list(unidades.keys()))
        continue

    temp = pd.DataFrame(unidades["USD"])

    if temp.empty:
        print("Sin registros.")
        continue

    columnas = [
        c for c in
        ["end", "val", "fy", "fp", "form", "filed", "accn"]
        if c in temp.columns
    ]

    # Nos quedamos con filings cercanos al periodo de interés
    if "filed" in temp.columns:
        temp["filed_dt"] = pd.to_datetime(
            temp["filed"],
            errors="coerce"
        )

        temp = temp[
            (temp["filed_dt"] >= "2015-01-01")
            & (temp["filed_dt"] <= "2017-12-31")
        ]

    print(
        temp[columnas]
        .drop_duplicates()
        .sort_values(
            ["filed", "end"]
            if "filed" in columnas
            else ["end"]
        )
        .to_string(index=False)
    )


CONCEPTO: Assets
       end          val     fy   fp   form      filed                 accn
2013-12-28  23148000000    NaN None    8-K 2015-08-10 0001637459-15-000025
2014-12-27  22947000000    NaN None    8-K 2015-08-10 0001637459-15-000025
2014-12-28  36763000000 2015.0   Q2   10-Q 2015-08-10 0001637459-15-000021
2015-06-28  36062000000 2015.0   Q2   10-Q 2015-08-10 0001637459-15-000021
2014-12-28  36535000000 2015.0   Q3   10-Q 2015-11-06 0001637459-15-000049
2015-09-27 121792000000 2015.0   Q3   10-Q 2015-11-06 0001637459-15-000049
2014-12-28  36571000000 2015.0   FY   10-K 2016-03-03 0001637459-16-000100
2016-01-03 122973000000 2015.0   FY   10-K 2016-03-03 0001637459-16-000100
2016-01-03 122973000000 2016.0   Q1   10-Q 2016-05-05 0001637459-16-000147
2016-04-03 123273000000 2016.0   Q1   10-Q 2016-05-05 0001637459-16-000147
2016-01-03 122973000000 2016.0   Q2   10-Q 2016-08-05 0001637459-16-000179
2016-07-03 121684000000 2016.0   Q2   10-Q 2016-08-05 0001637459-16-000179
2016-01

In [59]:
# ============================================================
# COMPLETAR KRAFT HEINZ t-3 (FY2015)
# ============================================================

valores_kraft_2015 = {
    "Assets": 122_973_000_000,
    "Liabilities_Total": 56_737_000_000,
    "Liabilities_Current": 6_932_000_000,
    "Receivables": 871_000_000,
    "Inventory": 2_618_000_000,
    "Cash": 4_837_000_000,
    "Equity": 57_685_000_000
}

mascara = (
    (dataset_positivo["Empresa"] == "Kraft Heinz")
    & (dataset_positivo["Periodo"] == "t-3")
    & (dataset_positivo["Ejercicio"] == 2015)
)

for variable, valor in valores_kraft_2015.items():
    dataset_positivo.loc[mascara, variable] = valor


# Comprobación
columnas_comprobar = [
    "Empresa",
    "Periodo",
    "Ejercicio",
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "Kraft Heinz",
        columnas_comprobar
    ].to_string(index=False)
)

    Empresa Periodo  Ejercicio      Revenue       Assets  Liabilities_Total  Liabilities_Current    NetIncome  Receivables    Inventory         Cash       Equity
Kraft Heinz     t-1       2017 2.623200e+10 1.202320e+11       5.419800e+10         1.013200e+10 1.099900e+10  921000000.0 2815000000.0 1629000000.0 6.603400e+10
Kraft Heinz     t-2       2016 1.833800e+10 1.229730e+11       6.528800e+10         6.932000e+09 6.340000e+08  871000000.0 2618000000.0 4837000000.0 5.768500e+10
Kraft Heinz     t-3       2015 2.478000e+09 1.229730e+11       5.673700e+10         6.932000e+09 2.760000e+08  871000000.0 2618000000.0 4837000000.0 5.768500e+10


In [60]:
# ============================================================
# COMPROBACIÓN GLOBAL DE COMPLETITUD
# ============================================================

variables_financieras = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

# Solo empresas válidas para el modelo financiero
dataset_financiero = dataset_positivo[
    dataset_positivo["Usar_Modelo_Financiero"] == True
].copy()

resumen = []

for variable in variables_financieras:

    disponibles = dataset_financiero[variable].notna().sum()
    faltantes = dataset_financiero[variable].isna().sum()

    resumen.append({
        "Variable": variable,
        "Disponibles": disponibles,
        "Faltantes": faltantes,
        "Completitud_%": round(
            disponibles / len(dataset_financiero) * 100,
            1
        )
    })

resumen_actualizado = pd.DataFrame(resumen)

print(
    "Número de observaciones financieras:",
    len(dataset_financiero)
)

print()

print(
    resumen_actualizado.to_string(index=False)
)

Número de observaciones financieras: 42

           Variable  Disponibles  Faltantes  Completitud_%
            Revenue           42          0          100.0
             Assets           42          0          100.0
  Liabilities_Total           42          0          100.0
Liabilities_Current           36          6           85.7
          NetIncome           39          3           92.9
        Receivables           38          4           90.5
          Inventory           33          9           78.6
               Cash           35          7           83.3
             Equity           42          0          100.0


In [62]:
import requests
import pandas as pd

# ============================================================
# BUSCAR CONCEPTOS DE NET INCOME - BAUSCH / VALEANT
# ============================================================

# SEC solicita identificar las peticiones
headers = {
    "User-Agent": "Trabajo académico contacto@example.com",
    "Accept-Encoding": "gzip, deflate"
}

cik_bausch = "0000885590"

url = (
    f"https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_bausch}.json"
)

r = requests.get(url, headers=headers)
r.raise_for_status()

facts_bausch = r.json()

usgaap_bausch = facts_bausch["facts"].get("us-gaap", {})

palabras = [
    "NetIncome",
    "ProfitLoss",
    "IncomeLoss",
    "NetLoss",
    "NetEarnings"
]

resultados_bausch = []

for concepto, info in usgaap_bausch.items():

    etiqueta = info.get("label") or ""

    if any(
        palabra.lower() in concepto.lower()
        or palabra.lower() in etiqueta.lower()
        for palabra in palabras
    ):

        for unidad, registros in info.get("units", {}).items():

            df_temp = pd.DataFrame(registros)

            if df_temp.empty or "fy" not in df_temp.columns:
                continue

            df_temp = df_temp[
                df_temp["fy"].isin([2012, 2013, 2014])
            ]

            if df_temp.empty:
                continue

            resultados_bausch.append({
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Unidad": unidad,
                "Años": sorted(
                    df_temp["fy"].dropna().unique().tolist()
                ),
                "Registros": len(df_temp),
                "Valor_max": (
                    df_temp["val"].abs().max()
                    if "val" in df_temp.columns
                    else None
                )
            })

resultado_bausch = pd.DataFrame(resultados_bausch)

if resultado_bausch.empty:
    print("No se encontraron conceptos candidatos.")
else:
    resultado_bausch = resultado_bausch.sort_values(
        "Valor_max",
        ascending=False
    )

    print("CONCEPTOS CANDIDATOS - BAUSCH HEALTH / VALEANT")
    print("=" * 100)
    print(resultado_bausch.to_string(index=False))

CONCEPTOS CANDIDATOS - BAUSCH HEALTH / VALEANT
                                                                                                                               Concepto                                                                                                                                         Etiqueta Unidad                     Años  Registros  Valor_max
                                                                                                                    OperatingIncomeLoss                                                                                                                          Operating Income (Loss)    USD [2012.0, 2013.0, 2014.0]         39 2039700000
                                                                             IncomeLossFromContinuingOperationsBeforeIncomeTaxesForeign                                                                            Income (Loss) from Continuing Operations before Income Taxes, Foreign   

In [63]:
# ============================================================
# EXTRAER NET INCOME ANUAL - BAUSCH HEALTH / VALEANT
# ============================================================

concepto_bausch = "NetIncomeLossAvailableToCommonStockholdersBasic"

registros = (
    facts_bausch["facts"]["us-gaap"][concepto_bausch]
    ["units"]["USD"]
)

df_bausch_ni = pd.DataFrame(registros)

# Convertimos fechas
df_bausch_ni["start"] = pd.to_datetime(
    df_bausch_ni["start"],
    errors="coerce"
)

df_bausch_ni["end"] = pd.to_datetime(
    df_bausch_ni["end"],
    errors="coerce"
)

# Duración del periodo
df_bausch_ni["dias"] = (
    df_bausch_ni["end"] - df_bausch_ni["start"]
).dt.days

# Nos quedamos con los ejercicios que necesitamos
resultado_ni_bausch = df_bausch_ni[
    df_bausch_ni["fy"].isin([2012, 2013, 2014])
].copy()

# Solo periodos aproximadamente anuales
resultado_ni_bausch = resultado_ni_bausch[
    resultado_ni_bausch["dias"].between(330, 380)
]

columnas = [
    "start",
    "end",
    "val",
    "fy",
    "fp",
    "form",
    "filed",
    "accn",
    "dias"
]

columnas = [
    c for c in columnas
    if c in resultado_ni_bausch.columns
]

print("NET INCOME CANDIDATOS - BAUSCH / VALEANT")
print("=" * 100)

print(
    resultado_ni_bausch[columnas]
    .sort_values(["fy", "filed"])
    .to_string(index=False)
)

NET INCOME CANDIDATOS - BAUSCH / VALEANT
     start        end        val     fy fp form      filed                 accn  dias
2010-01-01 2010-12-31 -208193000 2012.0 FY 10-K 2013-02-28 0000885590-13-000014   364
2011-01-01 2011-12-31  159559000 2012.0 FY 10-K 2013-02-28 0000885590-13-000014   364
2012-01-01 2012-12-31 -116025000 2012.0 FY 10-K 2013-02-28 0000885590-13-000014   365
2011-01-01 2011-12-31  159559000 2013.0 FY 10-K 2014-02-28 0000885590-14-000025   364
2012-01-01 2012-12-31 -116025000 2013.0 FY 10-K 2014-02-28 0000885590-14-000025   365
2013-01-01 2013-12-31 -866142000 2013.0 FY 10-K 2014-02-28 0000885590-14-000025   364
2012-01-01 2012-12-31 -116000000 2014.0 FY 10-K 2015-02-25 0000885590-15-000015   365
2013-01-01 2013-12-31 -866100000 2014.0 FY 10-K 2015-02-25 0000885590-15-000015   364
2014-01-01 2014-12-31  913500000 2014.0 FY 10-K 2015-02-25 0000885590-15-000015   364


In [64]:
# ============================================================
# COMPLETAR NET INCOME - BAUSCH HEALTH / VALEANT
# ============================================================

netincome_bausch = {
    2012: -116_025_000,
    2013: -866_142_000,
    2014: 913_500_000
}

for anio, valor in netincome_bausch.items():

    mascara = (
        (dataset_positivo["Empresa"] == "Bausch Health / Valeant")
        & (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mascara,
        "NetIncome"
    ] = valor


# Comprobación
print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "Bausch Health / Valeant",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Assets",
            "NetIncome"
        ]
    ].to_string(index=False)
)

                Empresa Periodo  Ejercicio      Revenue       Assets    NetIncome
Bausch Health / Valeant     t-1       2014 8103600000.0 2.635300e+10  913500000.0
Bausch Health / Valeant     t-2       2013 5640333000.0 2.797080e+10 -866142000.0
Bausch Health / Valeant     t-3       2012 3309895000.0 1.795038e+10 -116025000.0


In [65]:
# ============================================================
# BUSCAR RECEIVABLES - HEALTHCARE SERVICES GROUP
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "Trabajo universitario contacto@example.com"
}

cik_hcsg = "0000731012"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_hcsg}.json"
)

r = requests.get(url, headers=headers)
print("Status code:", r.status_code)

facts_hcsg = r.json()

usgaap_hcsg = facts_hcsg["facts"].get("us-gaap", {})

palabras = [
    "receivable",
    "accountsreceivable",
    "account receivable",
    "trade receivable"
]

resultados = []

for concepto, info in usgaap_hcsg.items():

    etiqueta = info.get("label") or ""

    texto_busqueda = (
        str(concepto).lower()
        + " "
        + str(etiqueta).lower()
    )

    if any(
        palabra.lower() in texto_busqueda
        for palabra in palabras
    ):

        unidades = info.get("units", {})

        for unidad, registros in unidades.items():

            df_temp = pd.DataFrame(registros)

            if df_temp.empty:
                continue

            # Filtrar años que necesitamos
            if "fy" in df_temp.columns:
                df_temp = df_temp[
                    df_temp["fy"].isin([2018, 2019, 2020])
                ]

            if df_temp.empty:
                continue

            resultados.append({
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Unidad": unidad,
                "Años": sorted(
                    df_temp["fy"]
                    .dropna()
                    .unique()
                    .tolist()
                ),
                "Registros": len(df_temp),
                "Valor_max": (
                    df_temp["val"].max()
                    if "val" in df_temp.columns
                    else None
                )
            })


resultado_receivables_hcsg = pd.DataFrame(resultados)

print("\nCONCEPTOS CANDIDATOS - RECEIVABLES - HEALTHCARE SERVICES GROUP")
print("=" * 110)

if resultado_receivables_hcsg.empty:

    print("No se encontraron conceptos candidatos.")

else:

    resultado_receivables_hcsg = (
        resultado_receivables_hcsg
        .sort_values(
            "Valor_max",
            ascending=False
        )
    )

    print(
        resultado_receivables_hcsg
        .to_string(index=False)
    )

Status code: 200

CONCEPTOS CANDIDATOS - RECEIVABLES - HEALTHCARE SERVICES GROUP
                                                      Concepto                                                                    Etiqueta Unidad               Años  Registros  Valor_max
                                 AccountsAndNotesReceivableNet          Accounts and Financing Receivable, after Allowance for Credit Loss    USD [2018, 2019, 2020]         24  399390000
                     AccountsNotesAndLoansReceivableNetCurrent Accounts and Financing Receivable, after Allowance for Credit Loss, Current    USD [2018, 2019, 2020]         24  378720000
                  IncreaseDecreaseInAccountsAndNotesReceivable                        Increase (Decrease) in Accounts and Notes Receivable    USD [2018, 2019, 2020]         27  121639000
                 AllowanceForDoubtfulAccountsReceivableCurrent                     Accounts Receivable, Allowance for Credit Loss, Current    USD [2018, 2019, 2020]       

In [66]:
# ============================================================
# EXTRAER RECEIVABLES - HEALTHCARE SERVICES GROUP
# ============================================================

concepto_hcsg = "AccountsNotesAndLoansReceivableNetCurrent"

registros = (
    facts_hcsg["facts"]["us-gaap"][concepto_hcsg]
    ["units"]["USD"]
)

df_hcsg_rec = pd.DataFrame(registros)

# Convertir fecha de cierre
df_hcsg_rec["end"] = pd.to_datetime(
    df_hcsg_rec["end"],
    errors="coerce"
)

# Nos interesan los cierres de 2018, 2019 y 2020
df_hcsg_rec = df_hcsg_rec[
    df_hcsg_rec["end"].dt.year.isin([2018, 2019, 2020])
].copy()

columnas = [
    "end",
    "val",
    "fy",
    "fp",
    "form",
    "filed",
    "accn",
    "frame"
]

columnas = [
    c for c in columnas
    if c in df_hcsg_rec.columns
]

print("RECEIVABLES CANDIDATOS - HEALTHCARE SERVICES GROUP")
print("=" * 100)

print(
    df_hcsg_rec[columnas]
    .sort_values(["end", "filed"])
    .to_string(index=False)
)

RECEIVABLES CANDIDATOS - HEALTHCARE SERVICES GROUP
       end       val   fy fp form      filed                 accn     frame
2018-03-31 335014000 2018 Q1 10-Q 2018-04-27 0000731012-18-000051 CY2018Q1I
2018-06-30 343665000 2018 Q2 10-Q 2018-07-27 0000731012-18-000061 CY2018Q2I
2018-09-30 353484000 2018 Q3 10-Q 2018-10-19 0000731012-18-000073 CY2018Q3I
2018-12-31 341838000 2018 FY 10-K 2019-03-18 0000731012-19-000040       NaN
2018-12-31 341838000 2019 Q1 10-Q 2019-05-03 0000731012-19-000053       NaN
2018-12-31 341838000 2019 Q2 10-Q 2019-07-26 0000731012-19-000063       NaN
2018-12-31 341838000 2019 Q3 10-Q 2019-10-25 0000731012-19-000071       NaN
2018-12-31 341838000 2018 FY 10-K 2020-02-21 0000731012-20-000033 CY2018Q4I
2019-03-31 353106000 2019 Q1 10-Q 2019-05-03 0000731012-19-000053 CY2019Q1I
2019-06-30 346765000 2019 Q2 10-Q 2019-07-26 0000731012-19-000063 CY2019Q2I
2019-09-30 349654000 2019 Q3 10-Q 2019-10-25 0000731012-19-000071 CY2019Q3I
2019-12-31 340930000 2018 FY 10-K 202

In [67]:
# ============================================================
# COMPLETAR RECEIVABLES - HEALTHCARE SERVICES GROUP
# ============================================================

receivables_hcsg = {
    2018: 341_838_000,
    2019: 340_930_000,
    2020: 255_474_000
}

for anio, valor in receivables_hcsg.items():

    mascara = (
        (dataset_positivo["Empresa"] == "Healthcare Services Group")
        & (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mascara,
        "Receivables"
    ] = valor


# Comprobación
print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "Healthcare Services Group",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Assets",
            "Receivables"
        ]
    ].to_string(index=False)
)

                  Empresa Periodo  Ejercicio      Revenue      Assets  Receivables
Healthcare Services Group     t-1       2020 1760303000.0 785031000.0  255474000.0
Healthcare Services Group     t-2       2019 1840778000.0 722592000.0  340930000.0
Healthcare Services Group     t-3       2018 2002601000.0 692603000.0  341838000.0


In [73]:
# ============================================================
# VEREIT / ARCP 2011 - BÚSQUEDA AMPLIA DE RECEIVABLES
# ============================================================

cik_vereit = "0001507385"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_vereit}.json"
)

r = requests.get(url, headers=headers)
r.raise_for_status()

facts_vereit = r.json()
usgaap_vereit = facts_vereit["facts"].get("us-gaap", {})

candidatos = []

for concepto, info in usgaap_vereit.items():

    etiqueta = info.get("label") or ""

    texto = f"{concepto} {etiqueta}".lower()

    if any(
        palabra in texto
        for palabra in [
            "receiv",
            "tenant",
            "rent",
            "lease",
            "note",
            "loan"
        ]
    ):

        for unidad, registros in info.get("units", {}).items():

            df_temp = pd.DataFrame(registros)

            if df_temp.empty:
                continue

            if "end" in df_temp.columns:
                df_temp["end_dt"] = pd.to_datetime(
                    df_temp["end"],
                    errors="coerce"
                )

                df_2011 = df_temp[
                    df_temp["end_dt"].dt.year == 2011
                ].copy()
            else:
                continue

            if df_2011.empty:
                continue

            candidatos.append({
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Unidad": unidad,
                "Registros": len(df_2011),
                "Valor_max": (
                    df_2011["val"].max()
                    if "val" in df_2011.columns
                    else None
                )
            })

resultado_vereit_2011 = pd.DataFrame(candidatos)

print("CANDIDATOS AMPLIOS - VEREIT / ARCP 2011")
print("=" * 100)

if resultado_vereit_2011.empty:
    print("No se encontraron conceptos candidatos para 2011.")
else:
    print(
        resultado_vereit_2011
        .sort_values("Valor_max", ascending=False)
        .to_string(index=False)
    )

CANDIDATOS AMPLIOS - VEREIT / ARCP 2011
                                                                 Concepto                                                                                                         Etiqueta Unidad  Registros  Valor_max
                                                       StockholdersEquity                                                                      Stockholders' Equity Attributable to Parent    USD         10  137086000
                          FiniteLivedIntangibleAssetAcquiredInPlaceLeases                                                          Finite-Lived Intangible Asset, Acquired-in-Place Leases    USD          8   21777000
                                                 ProceedsFromNotesPayable                                                                                      Proceeds from Notes Payable    USD          3   21470000
                               OperatingLeasesIncomeStatementLeaseRevenue                       

In [74]:
# ============================================================
# BUSCAR LIABILITIES CURRENT - GENERAL ELECTRIC
# ============================================================

cik_ge = "0000040545"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_ge}.json"
)

r = requests.get(url, headers=headers)
r.raise_for_status()

facts_ge = r.json()

usgaap_ge = facts_ge["facts"].get("us-gaap", {})

palabras = [
    "liabilitiescurrent",
    "currentliabilities",
    "liabilities current",
    "current liabilities",
    "payable",
    "shorttermdebt"
]

resultados = []

for concepto, info in usgaap_ge.items():

    etiqueta = info.get("label") or ""

    texto = (
        str(concepto).lower()
        + " "
        + str(etiqueta).lower()
    )

    if any(
        palabra.lower() in texto
        for palabra in palabras
    ):

        for unidad, registros in info.get("units", {}).items():

            df_temp = pd.DataFrame(registros)

            if df_temp.empty or "end" not in df_temp.columns:
                continue

            df_temp["end_dt"] = pd.to_datetime(
                df_temp["end"],
                errors="coerce"
            )

            # Cierres correspondientes a 2014, 2015 y 2016
            df_obj = df_temp[
                df_temp["end_dt"].dt.year.isin(
                    [2014, 2015, 2016]
                )
            ].copy()

            if df_obj.empty:
                continue

            resultados.append({
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Unidad": unidad,
                "Años": sorted(
                    df_obj["end_dt"]
                    .dt.year
                    .dropna()
                    .unique()
                    .tolist()
                ),
                "Registros": len(df_obj),
                "Valor_max": (
                    df_obj["val"].max()
                    if "val" in df_obj.columns
                    else None
                )
            })


resultado_liab_current_ge = pd.DataFrame(resultados)

print(
    "CONCEPTOS CANDIDATOS - "
    "LIABILITIES CURRENT - GENERAL ELECTRIC"
)

print("=" * 115)

if resultado_liab_current_ge.empty:

    print("No se encontraron candidatos.")

else:

    print(
        resultado_liab_current_ge
        .sort_values(
            "Valor_max",
            ascending=False
        )
        .to_string(index=False)
    )

CONCEPTOS CANDIDATOS - LIABILITIES CURRENT - GENERAL ELECTRIC
                                                        Concepto                                                                        Etiqueta Unidad               Años  Registros   Valor_max
                            NotesPayableRelatedPartiesNoncurrent                                      Notes Payable, Related Parties, Noncurrent    USD             [2016]          3 47173000000
                                       AccruedLiabilitiesCurrent                                                    Accrued Liabilities, Current    USD [2014, 2015, 2016]         32 23597000000
                                          AccountsPayableCurrent                                                       Accounts Payable, Current    USD [2014, 2015, 2016]         32 17561000000
                     NotesPayableRelatedPartiesClassifiedCurrent                                         Notes Payable, Related Parties, Current    USD           

In [75]:
# ============================================================
# BÚSQUEDA AMPLIA DE PASIVOS - GENERAL ELECTRIC
# ============================================================

resultados_ge_liab = []

for concepto, info in usgaap_ge.items():

    etiqueta = info.get("label") or ""

    texto = f"{concepto} {etiqueta}".lower()

    if "liabilit" not in texto:
        continue

    for unidad, registros in info.get("units", {}).items():

        if unidad != "USD":
            continue

        df_temp = pd.DataFrame(registros)

        if df_temp.empty or "end" not in df_temp.columns:
            continue

        df_temp["end_dt"] = pd.to_datetime(
            df_temp["end"],
            errors="coerce"
        )

        df_obj = df_temp[
            df_temp["end_dt"].dt.year.isin([2014, 2015, 2016])
        ].copy()

        if df_obj.empty:
            continue

        resultados_ge_liab.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Años": sorted(
                df_obj["end_dt"]
                .dt.year
                .dropna()
                .unique()
                .tolist()
            ),
            "Registros": len(df_obj),
            "Valor_max": df_obj["val"].max()
        })


resultado_ge_liab_amplio = pd.DataFrame(resultados_ge_liab)

print("TODOS LOS CONCEPTOS DE PASIVOS - GENERAL ELECTRIC")
print("=" * 120)

if resultado_ge_liab_amplio.empty:

    print("No se encontraron conceptos.")

else:

    print(
        resultado_ge_liab_amplio
        .sort_values("Valor_max", ascending=False)
        .to_string(index=False)
    )

TODOS LOS CONCEPTOS DE PASIVOS - GENERAL ELECTRIC
                                                                                                                     Concepto                                                                                                                                        Etiqueta               Años  Registros    Valor_max
                                                                                             LiabilitiesAndStockholdersEquity                                                                                                                          Liabilities and Equity [2014, 2015, 2016]         32 654954000000
                                                                                                                  Liabilities                                                                                                                                     Liabilities [2014, 2015, 2016]         32 518023000000
           

In [76]:
# ============================================================
# BUSCAR CASH - GENERAL ELECTRIC
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM academic research contacto@example.com"
}

cik_ge = "0000040545"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_ge}.json"
)

r = requests.get(url, headers=headers)
print("Status code:", r.status_code)

facts_ge = r.json()

usgaap_ge = facts_ge["facts"].get("us-gaap", {})

palabras = [
    "cash",
    "cashequivalent",
    "cashandcash",
    "cashcash"
]

resultados_cash_ge = []

for concepto, info in usgaap_ge.items():

    etiqueta = info.get("label") or ""
    texto = f"{concepto} {etiqueta}".lower()

    if not any(p.lower() in texto for p in palabras):
        continue

    for unidad, registros in info.get("units", {}).items():

        if unidad != "USD":
            continue

        df_temp = pd.DataFrame(registros)

        if df_temp.empty or "end" not in df_temp.columns:
            continue

        df_temp["end_dt"] = pd.to_datetime(
            df_temp["end"],
            errors="coerce"
        )

        df_obj = df_temp[
            df_temp["end_dt"].dt.year.isin([2014, 2015, 2016])
        ].copy()

        if df_obj.empty:
            continue

        resultados_cash_ge.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Unidad": unidad,
            "Años": sorted(
                df_obj["end_dt"]
                .dt.year
                .dropna()
                .unique()
                .tolist()
            ),
            "Registros": len(df_obj),
            "Valor_max": df_obj["val"].max()
        })


resultado_cash_ge = pd.DataFrame(resultados_cash_ge)

print("\nCANDIDATOS CASH - GENERAL ELECTRIC")
print("=" * 120)

if resultado_cash_ge.empty:

    print("No se encontraron candidatos.")

else:

    print(
        resultado_cash_ge
        .sort_values("Valor_max", ascending=False)
        .to_string(index=False)
    )

Status code: 200

CANDIDATOS CASH - GENERAL ELECTRIC
                                                                                                                          Concepto                                                                                                                                                 Etiqueta Unidad               Años  Registros    Valor_max
                                                              CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations                                                                          Cash and Cash Equivalents, at Carrying Value, Including Discontinued Operations    USD [2014, 2015, 2016]         54 110611000000
                      CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsIncludingDisposalGroupAndDiscontinuedOperations                            Cash, Cash Equivalents, Restricted Cash and Restricted Cash Equivalents, Including Disposal Group and Discontinued Operat

In [77]:
# ============================================================
# EXTRAER CASH EXACTO - GENERAL ELECTRIC
# ============================================================

concepto_cash_ge = (
    "CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations"
)

registros = (
    facts_ge["facts"]["us-gaap"][concepto_cash_ge]
    ["units"]["USD"]
)

cash_ge = pd.DataFrame(registros)

# Convertimos fechas
cash_ge["end_dt"] = pd.to_datetime(
    cash_ge["end"],
    errors="coerce"
)

cash_ge["filed_dt"] = pd.to_datetime(
    cash_ge["filed"],
    errors="coerce"
)

# Nos quedamos con cierres de 2014, 2015 y 2016
cash_ge_obj = cash_ge[
    cash_ge["end_dt"].dt.year.isin([2014, 2015, 2016])
].copy()

# Priorizamos registros incluidos en 10-K
cash_ge_obj = cash_ge_obj[
    cash_ge_obj["form"].isin(["10-K", "10-K/A"])
]

print("CASH - GENERAL ELECTRIC")
print("=" * 110)

columnas = [
    "end",
    "val",
    "fy",
    "fp",
    "form",
    "filed",
    "accn",
    "frame"
]

columnas = [
    c for c in columnas
    if c in cash_ge_obj.columns
]

print(
    cash_ge_obj[columnas]
    .sort_values(["end", "filed"])
    .to_string(index=False)
)

CASH - GENERAL ELECTRIC
       end         val   fy fp form      filed                 accn     frame
2014-12-31 91017000000 2014 Q4 10-K 2015-02-27 0000040545-15-000030       NaN
2014-12-31 91017000000 2015 FY 10-K 2016-02-26 0000040545-16-000145       NaN
2014-12-31 91017000000 2016 FY 10-K 2017-02-24 0000040545-17-000010       NaN
2014-12-31 91017000000 2017 FY 10-K 2018-02-23 0000040545-18-000014 CY2014Q4I
2015-12-31 90879000000 2015 FY 10-K 2016-02-26 0000040545-16-000145       NaN
2015-12-31 90879000000 2016 FY 10-K 2017-02-24 0000040545-17-000010       NaN
2015-12-31 90879000000 2017 FY 10-K 2018-02-23 0000040545-18-000014 CY2015Q4I
2016-12-31 49558000000 2016 FY 10-K 2017-02-24 0000040545-17-000010       NaN
2016-12-31 49558000000 2017 FY 10-K 2018-02-23 0000040545-18-000014 CY2016Q4I


In [78]:
# ============================================================
# RELLENAR CASH - GENERAL ELECTRIC
# ============================================================

cash_ge_valores = {
    2014: 91017000000,
    2015: 90879000000,
    2016: 49558000000
}

for anio, valor in cash_ge_valores.items():

    mascara = (
        (dataset_positivo["Empresa"] == "General Electric")
        & (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[mascara, "Cash"] = valor


# Comprobación
print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "General Electric",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Assets",
            "Liabilities_Total",
            "Liabilities_Current",
            "NetIncome",
            "Receivables",
            "Inventory",
            "Cash",
            "Equity"
        ]
    ].to_string(index=False)
)

         Empresa Periodo  Ejercicio      Revenue       Assets  Liabilities_Total  Liabilities_Current     NetIncome  Receivables    Inventory         Cash       Equity
General Electric     t-1       2016 1.236930e+11 3.651830e+11       2.893550e+11                  NaN  8.831000e+09 2.407600e+10 2.235400e+10 4.955800e+10 7.582800e+10
General Electric     t-2       2015 1.173860e+11 4.926920e+11       3.944180e+11                  NaN -6.126000e+09 2.702200e+10 2.251500e+10 9.087900e+10 9.827400e+10
General Electric     t-3       2014 1.485890e+11 6.483490e+11       5.201900e+11                  NaN  1.523300e+10 2.323700e+10 1.768900e+10 9.101700e+10 1.281590e+11


In [79]:
# ============================================================
# BUSCAR CASH - HEALTHCARE SERVICES GROUP
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM academic research contacto@example.com"
}

cik_hcsg = "0000731012"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_hcsg}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_hcsg = r.json()
usgaap_hcsg = facts_hcsg["facts"].get("us-gaap", {})

palabras = [
    "cash",
    "cashequivalent",
    "cashandcash"
]

resultados_cash_hcsg = []

for concepto, info in usgaap_hcsg.items():

    etiqueta = info.get("label") or ""
    texto = f"{concepto} {etiqueta}".lower()

    if not any(p.lower() in texto for p in palabras):
        continue

    for unidad, registros in info.get("units", {}).items():

        if unidad != "USD":
            continue

        df_temp = pd.DataFrame(registros)

        if df_temp.empty or "end" not in df_temp.columns:
            continue

        df_temp["end_dt"] = pd.to_datetime(
            df_temp["end"],
            errors="coerce"
        )

        # Solo nos interesa el cierre de 2020
        df_obj = df_temp[
            df_temp["end_dt"].dt.year == 2020
        ].copy()

        if df_obj.empty:
            continue

        resultados_cash_hcsg.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Unidad": unidad,
            "Registros": len(df_obj),
            "Valor_max": df_obj["val"].max()
        })


resultado_cash_hcsg = pd.DataFrame(resultados_cash_hcsg)

print("\nCANDIDATOS CASH - HEALTHCARE SERVICES GROUP 2020")
print("=" * 120)

if resultado_cash_hcsg.empty:
    print("No se encontraron candidatos.")
else:
    print(
        resultado_cash_hcsg
        .sort_values("Valor_max", ascending=False)
        .to_string(index=False)
    )

Status code: 200

CANDIDATOS CASH - HEALTHCARE SERVICES GROUP 2020
                                                                                                      Concepto                                                                                                                            Etiqueta Unidad  Registros  Valor_max
                                                                    NetCashProvidedByUsedInOperatingActivities                                                                                 Net Cash Provided by (Used in) Operating Activities    USD          9  217213000
                                                 CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents                                                             Cash, Cash Equivalents, Restricted Cash and Restricted Cash Equivalents    USD         16  139330000
CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect Cash, 

In [80]:
# ============================================================
# EXTRAER CASH EXACTO - HEALTHCARE SERVICES GROUP 2020
# ============================================================

concepto_cash_hcsg = (
    "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"
)

registros = (
    facts_hcsg["facts"]["us-gaap"][concepto_cash_hcsg]
    ["units"]["USD"]
)

cash_hcsg = pd.DataFrame(registros)

cash_hcsg["end_dt"] = pd.to_datetime(
    cash_hcsg["end"],
    errors="coerce"
)

cash_hcsg["filed_dt"] = pd.to_datetime(
    cash_hcsg["filed"],
    errors="coerce"
)

# Nos quedamos con el cierre de 2020
cash_hcsg_obj = cash_hcsg[
    cash_hcsg["end_dt"] == pd.Timestamp("2020-12-31")
].copy()

# Preferimos información presentada en 10-K
cash_hcsg_obj = cash_hcsg_obj[
    cash_hcsg_obj["form"].isin(["10-K", "10-K/A"])
]

print("CASH - HEALTHCARE SERVICES GROUP - 31/12/2020")
print("=" * 110)

columnas = [
    "end",
    "val",
    "fy",
    "fp",
    "form",
    "filed",
    "accn",
    "frame"
]

columnas = [
    c for c in columnas
    if c in cash_hcsg_obj.columns
]

print(
    cash_hcsg_obj[columnas]
    .sort_values("filed")
    .to_string(index=False)
)

CASH - HEALTHCARE SERVICES GROUP - 31/12/2020
       end       val   fy fp form      filed                 accn     frame
2020-12-31 139330000 2020 FY 10-K 2021-02-25 0000731012-21-000026       NaN
2020-12-31 139330000 2021 FY 10-K 2022-02-18 0000731012-22-000026       NaN
2020-12-31 139330000 2022 FY 10-K 2023-02-17 0000731012-23-000024       NaN
2020-12-31 139330000 2023 FY 10-K 2024-02-16 0000731012-24-000025 CY2020Q4I


In [81]:
# ============================================================
# RELLENAR CASH - HEALTHCARE SERVICES GROUP 2020
# ============================================================

mascara = (
    (dataset_positivo["Empresa"] == "Healthcare Services Group")
    & (dataset_positivo["Ejercicio"] == 2020)
)

dataset_positivo.loc[mascara, "Cash"] = 139330000


# Comprobación
print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "Healthcare Services Group",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Assets",
            "Receivables",
            "Inventory",
            "Cash"
        ]
    ].to_string(index=False)
)

                  Empresa Periodo  Ejercicio      Revenue      Assets  Receivables  Inventory        Cash
Healthcare Services Group     t-1       2020 1760303000.0 785031000.0  255474000.0        NaN 139330000.0
Healthcare Services Group     t-2       2019 1840778000.0 722592000.0  340930000.0        NaN  27329000.0
Healthcare Services Group     t-3       2018 2002601000.0 692603000.0  341838000.0        NaN  26025000.0


In [82]:
# ============================================================
# BUSCAR CASH - POWER SOLUTIONS INTERNATIONAL
# ============================================================

cik_psi = "0001137091"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_psi}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_psi = r.json()
usgaap_psi = facts_psi["facts"].get("us-gaap", {})

resultados_cash_psi = []

for concepto, info in usgaap_psi.items():

    etiqueta = info.get("label") or ""
    texto = f"{concepto} {etiqueta}".lower()

    if "cash" not in texto:
        continue

    for unidad, registros in info.get("units", {}).items():

        if unidad != "USD":
            continue

        df_temp = pd.DataFrame(registros)

        if df_temp.empty or "end" not in df_temp.columns:
            continue

        df_temp["end_dt"] = pd.to_datetime(
            df_temp["end"],
            errors="coerce"
        )

        df_obj = df_temp[
            df_temp["end_dt"].dt.year.isin([2013, 2014, 2015])
        ].copy()

        if df_obj.empty:
            continue

        resultados_cash_psi.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Unidad": unidad,
            "Años": sorted(
                df_obj["end_dt"]
                .dt.year
                .dropna()
                .unique()
                .tolist()
            ),
            "Registros": len(df_obj),
            "Valor_max": df_obj["val"].max()
        })


resultado_cash_psi = pd.DataFrame(resultados_cash_psi)

print("\nCANDIDATOS CASH - POWER SOLUTIONS INTERNATIONAL")
print("=" * 120)

if resultado_cash_psi.empty:
    print("No se encontraron candidatos.")
else:
    print(
        resultado_cash_psi
        .sort_values("Valor_max", ascending=False)
        .to_string(index=False)
    )

Status code: 200

CANDIDATOS CASH - POWER SOLUTIONS INTERNATIONAL
                                                                                      Concepto                                                                                                    Etiqueta Unidad               Años  Registros  Valor_max
                                NetCashProvidedByUsedInFinancingActivitiesContinuingOperations                                  Net Cash Provided by (Used in) Financing Activities, Continuing Operations    USD [2013, 2014, 2015]         12   76481000
                                                    NetCashProvidedByUsedInFinancingActivities                                                         Net Cash Provided by (Used in) Financing Activities    USD [2013, 2014, 2015]         12   68504000
                                                  PaymentsToAcquireBusinessesNetOfCashAcquired                                                        Payments to Acquire Businesses,

In [83]:
# ============================================================
# EXTRAER CASH EXACTO - POWER SOLUTIONS INTERNATIONAL
# ============================================================

concepto_cash_psi = "CashAndCashEquivalentsAtCarryingValue"

registros = (
    facts_psi["facts"]["us-gaap"][concepto_cash_psi]
    ["units"]["USD"]
)

cash_psi = pd.DataFrame(registros)

cash_psi["end_dt"] = pd.to_datetime(
    cash_psi["end"],
    errors="coerce"
)

cash_psi["filed_dt"] = pd.to_datetime(
    cash_psi["filed"],
    errors="coerce"
)

# Solo cierres de los ejercicios que necesitamos
cash_psi_obj = cash_psi[
    cash_psi["end_dt"].dt.year.isin([2013, 2014, 2015])
].copy()

print("CASH - POWER SOLUTIONS INTERNATIONAL")
print("=" * 110)

columnas = [
    "end",
    "val",
    "fy",
    "fp",
    "form",
    "filed",
    "accn",
    "frame"
]

columnas = [
    c for c in columnas
    if c in cash_psi_obj.columns
]

print(
    cash_psi_obj[columnas]
    .sort_values(["end", "filed"])
    .to_string(index=False)
)

CASH - POWER SOLUTIONS INTERNATIONAL
       end     val   fy fp form      filed                 accn     frame
2013-12-31 6306000 2017 FY 10-K 2019-05-16 0001137091-19-000016 CY2013Q4I
2014-12-31 6561000 2017 FY 10-K 2019-05-16 0001137091-19-000016 CY2014Q4I
2015-12-31 8445000 2017 FY 10-K 2019-05-16 0001137091-19-000016 CY2015Q4I


In [84]:
# ============================================================
# RELLENAR CASH - POWER SOLUTIONS INTERNATIONAL
# ============================================================

cash_psi_valores = {
    2013: 6306000,
    2014: 6561000,
    2015: 8445000
}

for anio, valor in cash_psi_valores.items():

    mascara = (
        (dataset_positivo["Empresa"] == "Power Solutions International")
        & (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[mascara, "Cash"] = valor


# Comprobación
print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "Power Solutions International",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue",
            "Assets",
            "Liabilities_Total",
            "Liabilities_Current",
            "NetIncome",
            "Receivables",
            "Inventory",
            "Cash",
            "Equity"
        ]
    ].to_string(index=False)
)


# Comprobar completitud global de Cash
print("\nCOMPLETITUD DE CASH")
print("=" * 50)

disponibles = dataset_positivo["Cash"].notna().sum()
faltantes = dataset_positivo["Cash"].isna().sum()
porcentaje = disponibles / len(dataset_positivo) * 100

print("Disponibles:", disponibles)
print("Faltantes:", faltantes)
print(f"Completitud: {porcentaje:.1f}%")

                      Empresa Periodo  Ejercicio     Revenue      Assets  Liabilities_Total  Liabilities_Current   NetIncome  Receivables   Inventory      Cash      Equity
Power Solutions International     t-1       2015 389446000.0 359936000.0        253639000.0           99262000.0  14278000.0  104365000.0 130347000.0 8445000.0 106297000.0
Power Solutions International     t-2       2014 347995000.0 262637000.0        171838000.0           76048000.0  23726000.0   81740000.0  93903000.0 6561000.0  90799000.0
Power Solutions International     t-3       2013 237842000.0 126619000.0         76198000.0           32385000.0 -18760000.0   42730000.0  55986000.0 6306000.0  50421000.0

COMPLETITUD DE CASH
Disponibles: 42
Faltantes: 3
Completitud: 93.3%


In [85]:
# ============================================================
# COMPROBAR QUÉ FILAS SIGUEN SIN CASH
# ============================================================

print(
    dataset_positivo.loc[
        dataset_positivo["Cash"].isna(),
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Cash"
        ]
    ].to_string(index=False)
)

print("\nNúmero total de filas:", len(dataset_positivo))

     Empresa Periodo  Ejercicio  Cash
Cronos Group     t-1       2019   NaN
Cronos Group     t-2       2018   NaN
Cronos Group     t-3       2017   NaN

Número total de filas: 45


In [86]:
# ============================================================
# DATASET FINANCIERO DEFINITIVO Y COMPLETITUD ACTUALIZADA
# ============================================================

# Excluimos Cronos Group del modelo financiero
# por problemas de comparabilidad IFRS/US-GAAP y CAD/USD

dataset_financiero = dataset_positivo[
    dataset_positivo["Empresa"] != "Cronos Group"
].copy()

print("Observaciones financieras:", len(dataset_financiero))

variables = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

resumen_final = []

for variable in variables:

    disponibles = dataset_financiero[variable].notna().sum()
    faltantes = dataset_financiero[variable].isna().sum()

    resumen_final.append({
        "Variable": variable,
        "Disponibles": disponibles,
        "Faltantes": faltantes,
        "Completitud_%": round(
            disponibles / len(dataset_financiero) * 100,
            1
        )
    })

resumen_final = pd.DataFrame(resumen_final)

print("\nCOMPLETITUD - MUESTRA FINANCIERA")
print("=" * 60)

print(
    resumen_final.to_string(index=False)
)

Observaciones financieras: 42

COMPLETITUD - MUESTRA FINANCIERA
           Variable  Disponibles  Faltantes  Completitud_%
            Revenue           42          0          100.0
             Assets           42          0          100.0
  Liabilities_Total           42          0          100.0
Liabilities_Current           36          6           85.7
          NetIncome           42          0          100.0
        Receivables           41          1           97.6
          Inventory           33          9           78.6
               Cash           42          0          100.0
             Equity           42          0          100.0


In [87]:
# ============================================================
# IDENTIFICAR TODOS LOS DATOS QUE TODAVÍA FALTAN
# ============================================================

variables_pendientes = [
    "Liabilities_Current",
    "Receivables",
    "Inventory"
]

for variable in variables_pendientes:

    faltantes = dataset_financiero[
        dataset_financiero[variable].isna()
    ][
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Formulario",
            variable
        ]
    ]

    print("\n" + "=" * 90)
    print(
        f"{variable.upper()} - "
        f"{len(faltantes)} FALTANTES"
    )
    print("=" * 90)

    if faltantes.empty:
        print("Sin datos faltantes.")
    else:
        print(faltantes.to_string(index=False))


# Resumen por empresa
print("\n" + "=" * 90)
print("RESUMEN DE FALTANTES POR EMPRESA")
print("=" * 90)

resumen_pendientes = (
    dataset_financiero
    .groupby("Empresa")[variables_pendientes]
    .apply(lambda x: x.isna().sum())
)

resumen_pendientes["Total_Faltantes"] = (
    resumen_pendientes.sum(axis=1)
)

resumen_pendientes = resumen_pendientes[
    resumen_pendientes["Total_Faltantes"] > 0
].sort_values(
    "Total_Faltantes",
    ascending=False
)

print(resumen_pendientes.to_string())


LIABILITIES_CURRENT - 6 FALTANTES
         Empresa Periodo  Ejercicio Formulario  Liabilities_Current
General Electric     t-1       2016       10-K                  NaN
General Electric     t-2       2015       10-K                  NaN
General Electric     t-3       2014       10-K                  NaN
   VEREIT / ARCP     t-1       2013       10-K                  NaN
   VEREIT / ARCP     t-2       2012       10-K                  NaN
   VEREIT / ARCP     t-3       2011       10-K                  NaN

RECEIVABLES - 1 FALTANTES
      Empresa Periodo  Ejercicio Formulario  Receivables
VEREIT / ARCP     t-3       2011       10-K          NaN

INVENTORY - 9 FALTANTES
                  Empresa Periodo  Ejercicio Formulario  Inventory
Healthcare Services Group     t-1       2020       10-K        NaN
Healthcare Services Group     t-2       2019       10-K        NaN
Healthcare Services Group     t-3       2018       10-K        NaN
                 Pareteum     t-1       2018       10-K

In [88]:
# ============================================================
# BUSCAR RECEIVABLES - VEREIT / ARCP - 2011
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM investigacion academica contacto@example.com"
}

# CIK de VEREIT / American Realty Capital Properties
cik_vereit = "0001507385"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_vereit}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_vereit = r.json()

usgaap_vereit = facts_vereit["facts"].get("us-gaap", {})

palabras = [
    "receivable",
    "accountsreceivable",
    "rentreceivable",
    "leasereceivable",
    "tenantreceivable",
    "notesreceivable"
]

resultados = []

for concepto, info in usgaap_vereit.items():

    etiqueta = info.get("label") or ""

    texto_busqueda = (
        concepto.lower() + " " + etiqueta.lower()
    )

    if any(
        palabra.lower() in texto_busqueda
        for palabra in palabras
    ):

        unidades = info.get("units", {})

        for unidad, registros in unidades.items():

            # Nos interesan importes monetarios
            if unidad != "USD":
                continue

            df_temp = pd.DataFrame(registros)

            if df_temp.empty:
                continue

            # Convertimos fecha final
            if "end" in df_temp.columns:
                df_temp["end"] = pd.to_datetime(
                    df_temp["end"],
                    errors="coerce"
                )

                # Buscamos hechos correspondientes a 2011
                df_2011 = df_temp[
                    df_temp["end"].dt.year == 2011
                ].copy()

                if not df_2011.empty:

                    resultados.append({
                        "Concepto": concepto,
                        "Etiqueta": etiqueta,
                        "Unidad": unidad,
                        "Registros_2011": len(df_2011),
                        "Valor_max": df_2011["val"].max()
                    })


resultado_receivables_vereit = pd.DataFrame(resultados)

print("\nCANDIDATOS RECEIVABLES - VEREIT / ARCP 2011")
print("=" * 120)

if resultado_receivables_vereit.empty:
    print("No se encontraron candidatos.")
else:
    resultado_receivables_vereit = (
        resultado_receivables_vereit
        .sort_values(
            "Valor_max",
            ascending=False
        )
    )

    print(
        resultado_receivables_vereit.to_string(
            index=False
        )
    )

Status code: 200

CANDIDATOS RECEIVABLES - VEREIT / ARCP 2011
                                                       Concepto                                                                 Etiqueta Unidad  Registros_2011  Valor_max
   CommonStockShareSubscribedButUnissuedSubscriptionsReceivable    Common Stock, Share Subscribed but Unissued, Subscriptions Receivable    USD               1     969000
ReceivableFromShareholdersOrAffiliatesForIssuanceOfCapitalStock Receivable from Shareholders or Affiliates for Issuance of Capital Stock    USD               1     969000
                       ProceedsFromCollectionOfLeaseReceivables                            Proceeds from Collection of Lease Receivables    USD               3          0
                        ProceedsFromCollectionOfNotesReceivable                             Proceeds from Collection of Notes Receivable    USD               3          0


In [2]:
# ============================================================
# COMPROBAR INVENTORY EN EMPRESAS CON DATOS FALTANTES
# ============================================================

import requests
import pandas as pd

# Cabecera necesaria para consultar la API de la SEC
headers = {
    "User-Agent": "TFM investigacion academica contacto@example.com"
}

empresas_inventory = {
    "Healthcare Services Group": "0000731012",
    "Pareteum": "0001084384",
    "VEREIT / ARCP": "0001507385"
}

for empresa_nombre, cik_empresa in empresas_inventory.items():

    print("\n" + "=" * 100)
    print("EMPRESA:", empresa_nombre)
    print("=" * 100)

    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik_empresa}.json"
    )

    r = requests.get(url, headers=headers)

    print("Status:", r.status_code)

    if r.status_code != 200:
        print("No se pudo descargar Company Facts.")
        continue

    facts_emp = r.json()

    usgaap_emp = facts_emp.get(
        "facts", {}
    ).get(
        "us-gaap", {}
    )

    candidatos = []

    for concepto, info in usgaap_emp.items():

        etiqueta = info.get("label") or ""

        texto = f"{concepto} {etiqueta}".lower()

        # Buscamos cualquier concepto relacionado con inventario
        if "inventor" not in texto:
            continue

        unidades = info.get("units", {})

        for unidad, registros in unidades.items():

            if unidad != "USD":
                continue

            df_temp = pd.DataFrame(registros)

            if df_temp.empty:
                continue

            candidatos.append({
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Registros": len(df_temp),
                "Valor_max": (
                    df_temp["val"].max()
                    if "val" in df_temp.columns
                    else None
                )
            })

    resultado_inv = pd.DataFrame(candidatos)

    if resultado_inv.empty:

        print("No existen conceptos de Inventory.")

    else:

        resultado_inv = (
            resultado_inv
            .drop_duplicates(subset=["Concepto"])
            .sort_values(
                "Valor_max",
                ascending=False
            )
        )

        print(
            resultado_inv.to_string(
                index=False
            )
        )


EMPRESA: Healthcare Services Group
Status: 200
                     Concepto                           Etiqueta  Registros  Valor_max
               OtherInventory             Other Inventory, Gross        128   42393000
                 InventoryNet                     Inventory, Net          2   16797000
IncreaseDecreaseInInventories Increase (Decrease) in Inventories        146    4531000

EMPRESA: Pareteum
Status: 200
    Concepto       Etiqueta  Registros  Valor_max
InventoryNet Inventory, Net         12     189567

EMPRESA: VEREIT / ARCP
Status: 200
                             Concepto                                                                                     Etiqueta  Registros  Valor_max
InventoriesPropertyHeldForSaleCurrent Disposal Group, Including Discontinued Operation, Inventory, Current (Deprecated 2015-01-31)          1    1818000


In [3]:
# ============================================================
# INVENTORY - HEALTHCARE SERVICES GROUP
# 2018, 2019 Y 2020
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM investigacion academica contacto@example.com"
}

cik_hcsg = "0000731012"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_hcsg}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_hcsg = r.json()

usgaap_hcsg = facts_hcsg["facts"].get("us-gaap", {})

# Conceptos que hemos encontrado
conceptos = [
    "InventoryNet",
    "OtherInventory"
]

for concepto in conceptos:

    print("\n" + "=" * 110)
    print("CONCEPTO:", concepto)
    print("=" * 110)

    if concepto not in usgaap_hcsg:
        print("Concepto no disponible.")
        continue

    unidades = usgaap_hcsg[concepto].get("units", {})

    if "USD" not in unidades:
        print("No hay registros en USD.")
        continue

    df_inv = pd.DataFrame(
        unidades["USD"]
    )

    if df_inv.empty:
        print("Sin registros.")
        continue

    # Convertimos end a fecha
    df_inv["end"] = pd.to_datetime(
        df_inv["end"],
        errors="coerce"
    )

    # Solo cierres de 2018, 2019 y 2020
    df_obj = df_inv[
        df_inv["end"].dt.year.isin(
            [2018, 2019, 2020]
        )
    ].copy()

    # Ordenamos
    df_obj = df_obj.sort_values(
        ["end", "filed"]
    )

    columnas = [
        c for c in
        [
            "end",
            "val",
            "fy",
            "fp",
            "form",
            "filed",
            "accn",
            "frame"
        ]
        if c in df_obj.columns
    ]

    if df_obj.empty:
        print(
            "No hay registros para "
            "2018, 2019 o 2020."
        )
    else:
        print(
            df_obj[columnas]
            .to_string(index=False)
        )

Status code: 200

CONCEPTO: InventoryNet
No hay registros para 2018, 2019 o 2020.

CONCEPTO: OtherInventory
       end      val   fy fp form      filed                 accn     frame
2018-03-31 42274000 2018 Q1 10-Q 2018-04-27 0000731012-18-000051 CY2018Q1I
2018-06-30 41424000 2018 Q2 10-Q 2018-07-27 0000731012-18-000061 CY2018Q2I
2018-09-30 41595000 2018 Q3 10-Q 2018-10-19 0000731012-18-000073 CY2018Q3I
2018-12-31 41443000 2018 FY 10-K 2019-03-18 0000731012-19-000040       NaN
2018-12-31 41443000 2019 Q1 10-Q 2019-05-03 0000731012-19-000053       NaN
2018-12-31 41443000 2019 Q2 10-Q 2019-07-26 0000731012-19-000063       NaN
2018-12-31 41443000 2019 Q3 10-Q 2019-10-25 0000731012-19-000071       NaN
2018-12-31 41443000 2018 FY 10-K 2020-02-21 0000731012-20-000033 CY2018Q4I
2019-03-31 40982000 2019 Q1 10-Q 2019-05-03 0000731012-19-000053 CY2019Q1I
2019-06-30 39977000 2019 Q2 10-Q 2019-07-26 0000731012-19-000063 CY2019Q2I
2019-09-30 38888000 2019 Q3 10-Q 2019-10-25 0000731012-19-000071 CY

In [4]:
# ============================================================
# INVENTORY - PARETEUM
# 2016, 2017 Y 2018
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM investigacion academica contacto@example.com"
}

cik_pareteum = "0001084384"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_pareteum}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_pareteum = r.json()

usgaap_pareteum = facts_pareteum["facts"].get(
    "us-gaap", {}
)

concepto = "InventoryNet"

registros = (
    usgaap_pareteum
    .get(concepto, {})
    .get("units", {})
    .get("USD", [])
)

df_inv_pareteum = pd.DataFrame(registros)

if df_inv_pareteum.empty:

    print("No hay registros de InventoryNet.")

else:

    df_inv_pareteum["end"] = pd.to_datetime(
        df_inv_pareteum["end"],
        errors="coerce"
    )

    # Nos quedamos con cierres de 2016, 2017 y 2018
    df_obj = df_inv_pareteum[
        df_inv_pareteum["end"].dt.year.isin(
            [2016, 2017, 2018]
        )
    ].copy()

    df_obj = df_obj.sort_values(
        ["end", "filed"]
    )

    columnas = [
        c for c in [
            "end",
            "val",
            "fy",
            "fp",
            "form",
            "filed",
            "accn",
            "frame"
        ]
        if c in df_obj.columns
    ]

    print("\nINVENTORY NET - PARETEUM")
    print("=" * 100)

    if df_obj.empty:
        print(
            "No hay registros para "
            "2016, 2017 o 2018."
        )
    else:
        print(
            df_obj[columnas]
            .to_string(index=False)
        )

Status code: 200

INVENTORY NET - PARETEUM
       end  val   fy fp form      filed                 accn     frame
2016-03-31 7191 2016 Q1 10-Q 2016-05-16 0001144204-16-102613 CY2016Q1I
2016-06-30    0 2016 Q2 10-Q 2016-08-16 0001144204-16-119628 CY2016Q2I
2016-09-30    0 2016 Q3 10-Q 2016-11-14 0001144204-16-134360 CY2016Q3I


In [5]:
# ============================================================
# BUSCAR LIABILITIES CURRENT - GENERAL ELECTRIC
# 2014, 2015 Y 2016
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM investigacion academica contacto@example.com"
}

cik_ge = "0000040545"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_ge}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_ge = r.json()

usgaap_ge = facts_ge["facts"].get("us-gaap", {})

palabras = [
    "liabilitiescurrent",
    "currentliabilities",
    "current liabilities",
    "shorttermlabilities",
    "shorttermliabilities"
]

resultados = []

for concepto, info in usgaap_ge.items():

    etiqueta = info.get("label") or ""

    texto = (
        concepto.lower()
        + " "
        + etiqueta.lower()
    )

    if not any(
        palabra.lower() in texto
        for palabra in palabras
    ):
        continue

    for unidad, registros in info.get(
        "units", {}
    ).items():

        if unidad != "USD":
            continue

        df_temp = pd.DataFrame(registros)

        if df_temp.empty or "end" not in df_temp.columns:
            continue

        df_temp["end"] = pd.to_datetime(
            df_temp["end"],
            errors="coerce"
        )

        df_obj = df_temp[
            df_temp["end"].dt.year.isin(
                [2014, 2015, 2016]
            )
        ].copy()

        if df_obj.empty:
            continue

        resultados.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Registros": len(df_obj),
            "Años": sorted(
                df_obj["end"]
                .dt.year
                .dropna()
                .unique()
                .tolist()
            ),
            "Valor_max": df_obj["val"].max()
        })


resultado_ge_current = pd.DataFrame(resultados)

print(
    "\nCANDIDATOS LIABILITIES CURRENT "
    "- GENERAL ELECTRIC"
)
print("=" * 120)

if resultado_ge_current.empty:

    print("No se encontraron candidatos.")

else:

    resultado_ge_current = (
        resultado_ge_current
        .drop_duplicates(
            subset=["Concepto"]
        )
        .sort_values(
            "Valor_max",
            ascending=False
        )
    )

    print(
        resultado_ge_current.to_string(
            index=False
        )
    )

Status code: 200

CANDIDATOS LIABILITIES CURRENT - GENERAL ELECTRIC
                 Concepto                     Etiqueta  Registros               Años   Valor_max
AccruedLiabilitiesCurrent Accrued Liabilities, Current         32 [2014, 2015, 2016] 23597000000


In [6]:
# ============================================================
# BUSCAR LIABILITIES CURRENT - VEREIT / ARCP
# 2011, 2012 Y 2013
# ============================================================

import requests
import pandas as pd

headers = {
    "User-Agent": "TFM investigacion academica contacto@example.com"
}

cik_vereit = "0001507385"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_vereit}.json"
)

r = requests.get(url, headers=headers)

print("Status code:", r.status_code)

facts_vereit = r.json()

usgaap_vereit = facts_vereit["facts"].get("us-gaap", {})

resultados = []

for concepto, info in usgaap_vereit.items():

    etiqueta = info.get("label") or ""
    texto = f"{concepto} {etiqueta}".lower()

    # Buscamos cualquier concepto relacionado
    # con pasivos corrientes
    if not (
        "liabilit" in texto
        and (
            "current" in texto
            or "short term" in texto
            or "short-term" in texto
        )
    ):
        continue

    for unidad, registros in info.get("units", {}).items():

        if unidad != "USD":
            continue

        df_temp = pd.DataFrame(registros)

        if df_temp.empty or "end" not in df_temp.columns:
            continue

        df_temp["end"] = pd.to_datetime(
            df_temp["end"],
            errors="coerce"
        )

        df_obj = df_temp[
            df_temp["end"].dt.year.isin(
                [2011, 2012, 2013]
            )
        ].copy()

        if df_obj.empty:
            continue

        resultados.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Años": sorted(
                df_obj["end"]
                .dt.year
                .dropna()
                .unique()
                .tolist()
            ),
            "Registros": len(df_obj),
            "Valor_max": df_obj["val"].max()
        })


resultado_vereit_current = pd.DataFrame(resultados)

print(
    "\nCANDIDATOS LIABILITIES CURRENT "
    "- VEREIT / ARCP"
)
print("=" * 120)

if resultado_vereit_current.empty:

    print("No se encontraron candidatos.")

else:

    print(
        resultado_vereit_current
        .drop_duplicates(subset=["Concepto"])
        .sort_values(
            "Valor_max",
            ascending=False
        )
        .to_string(index=False)
    )

Status code: 200

CANDIDATOS LIABILITIES CURRENT - VEREIT / ARCP
                                                Concepto                                                Etiqueta               Años  Registros  Valor_max
AccountsPayableAndAccruedLiabilitiesCurrentAndNoncurrent                Accounts Payable and Accrued Liabilities [2011, 2012, 2013]         38  808900000
             OtherAccruedLiabilitiesCurrentAndNoncurrent                               Other Accrued Liabilities       [2012, 2013]         12  683197000
       DeferredCompensationLiabilityCurrentAndNoncurrent Deferred Compensation Liability, Current and Noncurrent             [2013]          1   59400000


In [8]:
for nombre, objeto in list(globals().items()):
    if isinstance(objeto, pd.DataFrame):
        print(
            nombre,
            "| filas:", len(objeto),
            "| columnas:", list(objeto.columns)
        )

df_temp | filas: 72 | columnas: ['end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']
resultado_inv | filas: 1 | columnas: ['Concepto', 'Etiqueta', 'Registros', 'Valor_max']
df_inv | filas: 128 | columnas: ['end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']
df_obj | filas: 12 | columnas: ['end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']
df_inv_pareteum | filas: 12 | columnas: ['end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']
resultado_ge_current | filas: 1 | columnas: ['Concepto', 'Etiqueta', 'Registros', 'Años', 'Valor_max']
resultado_vereit_current | filas: 3 | columnas: ['Concepto', 'Etiqueta', 'Años', 'Registros', 'Valor_max']


In [9]:
# ============================================================
# BUSCAR ARCHIVOS GUARDADOS EN EL ENTORNO
# ============================================================

import os

extensiones = (
    ".csv",
    ".xlsx",
    ".xls",
    ".pkl",
    ".pickle",
    ".parquet"
)

archivos_encontrados = []

for carpeta, subcarpetas, archivos in os.walk("."):

    # Evitamos carpetas ocultas/problemáticas
    subcarpetas[:] = [
        d for d in subcarpetas
        if not d.startswith(".")
    ]

    for archivo in archivos:

        if archivo.lower().endswith(extensiones):

            ruta = os.path.join(
                carpeta,
                archivo
            )

            archivos_encontrados.append(ruta)


print("ARCHIVOS DE DATOS ENCONTRADOS")
print("=" * 80)

if archivos_encontrados:

    for ruta in archivos_encontrados:
        print(ruta)

else:
    print("No se encontraron archivos de datos guardados.")

ARCHIVOS DE DATOS ENCONTRADOS
./Accrual.xlsx
./704.xlsx
./Leyenda.xlsx
./T160.xlsx
./Reversal.xlsx
./Comparativa_T160.xlsx


In [10]:
print("tabla_sec existe:", "tabla_sec" in globals())
print("tabla_periodos existe:", "tabla_periodos" in globals())

tabla_sec existe: False
tabla_periodos existe: False


In [12]:
print("tabla_sec existe:", "tabla_sec" in globals())
print("tabla_periodos existe:", "tabla_periodos" in globals())
print("dataset_positivo existe:", "dataset_positivo" in globals())

tabla_sec existe: False
tabla_periodos existe: False
dataset_positivo existe: False


In [13]:
# ============================================================
# RECONSTRUCCIÓN COMPLETA DEL DATASET POSITIVO
# ============================================================

import requests
import pandas as pd
import time

HEADERS = {
    "User-Agent": "TFM investigacion academica contacto@example.com",
    "Accept-Encoding": "gzip, deflate"
}

# ------------------------------------------------------------
# 1. TABLA MAESTRA DE EMPRESAS
# ------------------------------------------------------------

datos_empresas = [
    ["Belden Inc.", "BDC", "0000913142", 3357, "1231", "2018-02-01"],
    ["General Electric", "GE", "0000040545", 3600, "1231", "2017-07-01"],
    ["Manitex International", "MNTX", "0001302028", 3559, "1231", "2018-04-03"],
    ["Power Solutions International", "PSIX", "0001137091", 3510, "1231", "2016-08-01"],
    ["Revolution Lighting Technologies", "RVLT", "0000917523", 3640, "1231", "2018-10-19"],
    ["Super Micro Computer", "SMCI", "0001375365", 3571, "0630", "2017-08-29"],
    ["Bausch Health / Valeant", "BHC", "0000885590", 2834, "1231", "2015-10-26"],
    ["VEREIT / ARCP", "VER", "0001507385", 6798, "1231", "2014-10-29"],
    ["Healthcare Services Group", "HCSG", "0000731012", 8050, "1231", "2021-08-24"],
    ["Kraft Heinz", "KHC", "0001637459", 2030, "1226", "2019-02-21"],
    ["Pareteum", "TEUM", "0001084384", 7373, "1231", "2019-10-21"],
    ["Cronos Group", "CRON", "0001656472", 2833, "1231", "2020-03-17"],
    ["Koppers Holdings", "KOP", "0001315257", 2400, "1231", "2022-11-01"],
    ["Tupperware Brands", "TUP", "0001008654", 3089, "1230", "2021-08-01"],
    ["Compass Minerals", "CMP", "0001227654", 1400, "0930", "2018-10-23"]
]

tabla_sec = pd.DataFrame(
    datos_empresas,
    columns=[
        "Empresa",
        "Ticker",
        "CIK",
        "SIC",
        "Cierre_Fiscal",
        "t0"
    ]
)

# ------------------------------------------------------------
# 2. FUNCIONES DE FILINGS
# ------------------------------------------------------------

def obtener_todos_filings(cik_empresa):

    url = f"https://data.sec.gov/submissions/CIK{cik_empresa}.json"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return pd.DataFrame()

    data_empresa = response.json()

    frames = []

    recent = pd.DataFrame(data_empresa["filings"]["recent"])

    if not recent.empty:
        frames.append(recent)

    for archivo in data_empresa["filings"].get("files", []):

        url_historico = (
            f"https://data.sec.gov/submissions/{archivo['name']}"
        )

        r = requests.get(
            url_historico,
            headers=HEADERS
        )

        if r.status_code == 200:

            historicos = pd.DataFrame(r.json())

            if not historicos.empty:
                frames.append(historicos)

        time.sleep(0.10)

    if not frames:
        return pd.DataFrame()

    # Evitar warning de concat
    frames_limpios = [
        df.dropna(axis=1, how="all")
        for df in frames
        if not df.empty
    ]

    return pd.concat(
        frames_limpios,
        ignore_index=True
    )


def corregir_report_date(
    report_date,
    filing_date,
    cierre_fiscal
):

    report_date = pd.to_datetime(
        report_date,
        errors="coerce"
    )

    filing_date = pd.to_datetime(
        filing_date,
        errors="coerce"
    )

    if pd.isna(report_date) or pd.isna(filing_date):
        return report_date

    cierre = str(cierre_fiscal).zfill(4)

    mes = int(cierre[:2])
    dia = int(cierre[2:])

    # Si coincide con cierre fiscal, mantenemos
    if (
        report_date.month == mes
        and abs(report_date.day - dia) <= 7
    ):
        return report_date

    # Caso tipo GE 2014
    diferencia = abs(
        (filing_date - report_date).days
    )

    if diferencia < 90:

        anio = filing_date.year - 1

        try:
            return pd.Timestamp(
                year=anio,
                month=mes,
                day=dia
            )
        except:
            return report_date

    return report_date


# ------------------------------------------------------------
# 3. RECONSTRUIR LOS 45 PERIODOS
# ------------------------------------------------------------

resultados_periodos = []

FORMULARIOS_ANUALES = [
    "10-K",
    "20-F",
    "40-F"
]

for _, row in tabla_sec.iterrows():

    print(f"Periodos: {row['Empresa']}")

    filings = obtener_todos_filings(
        row["CIK"]
    )

    if filings.empty:
        continue

    filings["filingDate"] = pd.to_datetime(
        filings["filingDate"],
        errors="coerce"
    )

    filings["reportDate"] = pd.to_datetime(
        filings["reportDate"],
        errors="coerce"
    )

    t0_fecha = pd.to_datetime(
        row["t0"]
    )

    anuales = filings[
        (filings["form"].isin(FORMULARIOS_ANUALES))
        &
        (filings["filingDate"] < t0_fecha)
    ].copy()

    anuales["ReportDate_Corregido"] = (
        anuales.apply(
            lambda x: corregir_report_date(
                x["reportDate"],
                x["filingDate"],
                row["Cierre_Fiscal"]
            ),
            axis=1
        )
    )

    anuales["Ejercicio"] = (
        anuales["ReportDate_Corregido"]
        .dt.year
    )

    anuales = anuales.sort_values(
        "filingDate",
        ascending=False
    )

    anuales = anuales.drop_duplicates(
        subset=["Ejercicio"],
        keep="first"
    )

    anuales = anuales.head(3)

    for posicion, (_, filing) in enumerate(
        anuales.iterrows(),
        start=1
    ):

        resultados_periodos.append({
            "Empresa": row["Empresa"],
            "Ticker": row["Ticker"],
            "CIK": row["CIK"],
            "SIC": row["SIC"],
            "t0": row["t0"],
            "Periodo": f"t-{posicion}",
            "Ejercicio": int(filing["Ejercicio"]),
            "Formulario": filing["form"],
            "Fecha_10K": filing["filingDate"],
            "ReportDate": filing["ReportDate_Corregido"],
            "Accession": filing["accessionNumber"],
            "Documento": filing["primaryDocument"]
        })

    time.sleep(0.10)

tabla_periodos = pd.DataFrame(
    resultados_periodos
)

print(
    "\nObservaciones temporales:",
    len(tabla_periodos)
)

# ------------------------------------------------------------
# 4. FUNCIONES XBRL
# ------------------------------------------------------------

def descargar_companyfacts(cik_empresa):

    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik_empresa}.json"
    )

    r = requests.get(
        url,
        headers=HEADERS
    )

    if r.status_code != 200:
        return None

    return r.json()


def extraer(
    namespace,
    concepto,
    ejercicio,
    t0_fecha,
    formularios
):

    if concepto not in namespace:
        return None

    unidades = (
        namespace[concepto]
        .get("units", {})
    )

    if "USD" not in unidades:
        return None

    df = pd.DataFrame(
        unidades["USD"]
    )

    if df.empty:
        return None

    df["filed"] = pd.to_datetime(
        df["filed"],
        errors="coerce"
    )

    df["end"] = pd.to_datetime(
        df["end"],
        errors="coerce"
    )

    df = df[
        df["form"].isin(formularios)
    ].copy()

    df = df[
        df["filed"] < t0_fecha
    ]

    # Primero intentamos por FY
    if "fy" in df.columns:

        fy_match = df[
            pd.to_numeric(
                df["fy"],
                errors="coerce"
            ) == ejercicio
        ].copy()

        if not fy_match.empty:

            # Para conceptos de duración
            if "start" in fy_match.columns:

                fy_match["start"] = pd.to_datetime(
                    fy_match["start"],
                    errors="coerce"
                )

                fy_match["dias"] = (
                    fy_match["end"]
                    - fy_match["start"]
                ).dt.days

                anual = fy_match[
                    fy_match["dias"].between(
                        330,
                        380
                    )
                ]

                if not anual.empty:
                    fy_match = anual

            fy_match = fy_match.sort_values(
                "filed"
            )

            return fy_match.iloc[0]["val"]

    # Si no funciona FY, usamos año de end
    df = df[
        df["end"].dt.year == ejercicio
    ].copy()

    if df.empty:
        return None

    if "start" in df.columns:

        df["start"] = pd.to_datetime(
            df["start"],
            errors="coerce"
        )

        df["dias"] = (
            df["end"]
            - df["start"]
        ).dt.days

        anual = df[
            df["dias"].between(
                330,
                380
            )
        ]

        if not anual.empty:
            df = anual

    df = df.sort_values("filed")

    return df.iloc[0]["val"]


# ------------------------------------------------------------
# 5. CONCEPTOS POR EMPRESA PARA REVENUE
# ------------------------------------------------------------

MAPEO_REVENUE = {
    "Belden Inc.": "SalesRevenueGoodsNet",
    "General Electric": "Revenues",
    "Manitex International": "SalesRevenueNet",
    "Power Solutions International": "SalesRevenueGoodsNet",
    "Revolution Lighting Technologies": "SalesRevenueNet",
    "Super Micro Computer": "SalesRevenueNet",
    "Bausch Health / Valeant": "SalesRevenueGoodsNet",
    "VEREIT / ARCP": "Revenues",
    "Healthcare Services Group": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Kraft Heinz": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Pareteum": "Revenues",
    "Koppers Holdings": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Tupperware Brands": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Compass Minerals": "SalesRevenueGoodsNet"
}

# ------------------------------------------------------------
# 6. EXTRACCIÓN BASE
# ------------------------------------------------------------

filas = []

for empresa, grupo in tabla_periodos.groupby(
    "Empresa"
):

    print(f"XBRL: {empresa}")

    fila_empresa = tabla_sec[
        tabla_sec["Empresa"] == empresa
    ].iloc[0]

    companyfacts = descargar_companyfacts(
        fila_empresa["CIK"]
    )

    if companyfacts is None:
        continue

    usgaap = (
        companyfacts
        .get("facts", {})
        .get("us-gaap", {})
    )

    for _, obs in grupo.iterrows():

        ejercicio = int(
            obs["Ejercicio"]
        )

        t0_fecha = pd.to_datetime(
            obs["t0"]
        )

        formulario = obs["Formulario"]

        # Cronos queda sin extracción financiera
        if empresa == "Cronos Group":

            filas.append({
                "Empresa": empresa,
                "Ticker": obs["Ticker"],
                "CIK": obs["CIK"],
                "SIC": obs["SIC"],
                "Periodo": obs["Periodo"],
                "Ejercicio": ejercicio,
                "Formulario": formulario,
                "Fecha_10K": obs["Fecha_10K"],
                "t0": obs["t0"],
                "Namespace": (
                    "ifrs-full"
                    if formulario == "40-F"
                    else "us-gaap"
                ),
                "Revenue": None,
                "Assets": None,
                "Liabilities_Total": None,
                "Liabilities_Current": None,
                "NetIncome": None,
                "Receivables": None,
                "Inventory": None,
                "Cash": None,
                "Equity": None
            })

            continue

        revenue_concept = MAPEO_REVENUE.get(
            empresa
        )

        revenue = (
            extraer(
                usgaap,
                revenue_concept,
                ejercicio,
                t0_fecha,
                [formulario]
            )
            if revenue_concept
            else None
        )

        assets = extraer(
            usgaap,
            "Assets",
            ejercicio,
            t0_fecha,
            [formulario]
        )

        equity = extraer(
            usgaap,
            "StockholdersEquity",
            ejercicio,
            t0_fecha,
            [formulario]
        )

        if equity is None:
            equity = extraer(
                usgaap,
                "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
                ejercicio,
                t0_fecha,
                [formulario]
            )

        netincome = extraer(
            usgaap,
            "NetIncomeLoss",
            ejercicio,
            t0_fecha,
            [formulario]
        )

        receivables = None

        for concepto in [
            "AccountsReceivableNetCurrent",
            "ReceivablesNetCurrent",
            "AccountsAndNotesReceivableNet",
            "AccountsNotesAndLoansReceivableNetCurrent"
        ]:

            receivables = extraer(
                usgaap,
                concepto,
                ejercicio,
                t0_fecha,
                [formulario]
            )

            if receivables is not None:
                break

        inventory = extraer(
            usgaap,
            "InventoryNet",
            ejercicio,
            t0_fecha,
            [formulario]
        )

        cash = None

        for concepto in [
            "CashAndCashEquivalentsAtCarryingValue",
            "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
            "CashAndCashEquivalentsAtCarryingValueIncludingDiscontinuedOperations"
        ]:

            cash = extraer(
                usgaap,
                concepto,
                ejercicio,
                t0_fecha,
                [formulario]
            )

            if cash is not None:
                break

        liabilities_current = extraer(
            usgaap,
            "LiabilitiesCurrent",
            ejercicio,
            t0_fecha,
            [formulario]
        )

        liabilities_total = (
            assets - equity
            if assets is not None
            and equity is not None
            else None
        )

        filas.append({
            "Empresa": empresa,
            "Ticker": obs["Ticker"],
            "CIK": obs["CIK"],
            "SIC": obs["SIC"],
            "Periodo": obs["Periodo"],
            "Ejercicio": ejercicio,
            "Formulario": formulario,
            "Fecha_10K": obs["Fecha_10K"],
            "t0": obs["t0"],
            "Namespace": "us-gaap",
            "Revenue": revenue,
            "Assets": assets,
            "Liabilities_Total": liabilities_total,
            "Liabilities_Current": liabilities_current,
            "NetIncome": netincome,
            "Receivables": receivables,
            "Inventory": inventory,
            "Cash": cash,
            "Equity": equity
        })

    time.sleep(0.10)

dataset_positivo = pd.DataFrame(
    filas
)

# ------------------------------------------------------------
# 7. CORRECCIONES MANUALES YA VALIDADAS
# ------------------------------------------------------------

# ---- Bausch / Valeant NetIncome ----
corr_bausch = {
    2012: -116_025_000,
    2013: -866_142_000,
    2014: 913_500_000
}

for anio, valor in corr_bausch.items():

    mask = (
        (dataset_positivo["Empresa"] == "Bausch Health / Valeant")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "NetIncome"
    ] = valor


# ---- Kraft Heinz FY2015 ----
mask = (
    (dataset_positivo["Empresa"] == "Kraft Heinz")
    &
    (dataset_positivo["Ejercicio"] == 2015)
)

dataset_positivo.loc[mask, "Assets"] = 122_973_000_000
dataset_positivo.loc[mask, "Liabilities_Total"] = 56_737_000_000
dataset_positivo.loc[mask, "Liabilities_Current"] = 6_932_000_000
dataset_positivo.loc[mask, "Receivables"] = 871_000_000
dataset_positivo.loc[mask, "Inventory"] = 2_618_000_000
dataset_positivo.loc[mask, "Cash"] = 4_837_000_000
dataset_positivo.loc[mask, "Equity"] = 57_685_000_000


# ---- Healthcare Services Group Receivables ----
corr_hcsg_rec = {
    2018: 341_838_000,
    2019: 340_930_000,
    2020: 255_474_000
}

for anio, valor in corr_hcsg_rec.items():

    mask = (
        (dataset_positivo["Empresa"] == "Healthcare Services Group")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "Receivables"
    ] = valor


# ---- Healthcare Services Group Inventory ----
corr_hcsg_inv = {
    2018: 41_443_000,
    2019: 36_517_000,
    2020: 31_586_000
}

for anio, valor in corr_hcsg_inv.items():

    mask = (
        (dataset_positivo["Empresa"] == "Healthcare Services Group")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "Inventory"
    ] = valor


# ---- Healthcare Services Group Cash 2020 ----
mask = (
    (dataset_positivo["Empresa"] == "Healthcare Services Group")
    &
    (dataset_positivo["Ejercicio"] == 2020)
)

dataset_positivo.loc[
    mask,
    "Cash"
] = 139_330_000


# ---- General Electric Cash ----
corr_ge_cash = {
    2014: 91_017_000_000,
    2015: 90_879_000_000,
    2016: 49_558_000_000
}

for anio, valor in corr_ge_cash.items():

    mask = (
        (dataset_positivo["Empresa"] == "General Electric")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "Cash"
    ] = valor


# ---- Power Solutions International Cash ----
corr_psi_cash = {
    2013: 6_306_000,
    2014: 6_561_000,
    2015: 8_445_000
}

for anio, valor in corr_psi_cash.items():

    mask = (
        (dataset_positivo["Empresa"] == "Power Solutions International")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "Cash"
    ] = valor


# ------------------------------------------------------------
# 8. MARCAR CRONOS COMO EXCLUIDO DEL MODELO FINANCIERO
# ------------------------------------------------------------

dataset_positivo[
    "Usar_Modelo_Financiero"
] = True

dataset_positivo[
    "Motivo_Exclusion_Financiera"
] = None

mask_cronos = (
    dataset_positivo["Empresa"]
    == "Cronos Group"
)

dataset_positivo.loc[
    mask_cronos,
    "Usar_Modelo_Financiero"
] = False

dataset_positivo.loc[
    mask_cronos,
    "Motivo_Exclusion_Financiera"
] = (
    "Cambio IFRS/US-GAAP y CAD/USD; "
    "datos pre-t0 no comparables de forma homogénea"
)


# ------------------------------------------------------------
# 9. DATASET FINANCIERO
# ------------------------------------------------------------

dataset_financiero = (
    dataset_positivo[
        dataset_positivo[
            "Usar_Modelo_Financiero"
        ] == True
    ]
    .copy()
)


# ------------------------------------------------------------
# 10. GUARDAR BACKUPS
# ------------------------------------------------------------

dataset_positivo.to_csv(
    "dataset_positivo_backup.csv",
    index=False
)

dataset_financiero.to_csv(
    "dataset_financiero_backup.csv",
    index=False
)

tabla_periodos.to_csv(
    "tabla_periodos_backup.csv",
    index=False
)


# ------------------------------------------------------------
# 11. RESUMEN FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RECONSTRUCCIÓN TERMINADA")
print("=" * 70)

print(
    "Filas dataset positivo:",
    len(dataset_positivo)
)

print(
    "Filas dataset financiero:",
    len(dataset_financiero)
)

variables = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

resumen = []

for variable in variables:

    disponibles = (
        dataset_financiero[
            variable
        ]
        .notna()
        .sum()
    )

    faltantes = (
        dataset_financiero[
            variable
        ]
        .isna()
        .sum()
    )

    resumen.append({
        "Variable": variable,
        "Disponibles": disponibles,
        "Faltantes": faltantes,
        "Completitud_%": round(
            disponibles
            / len(dataset_financiero)
            * 100,
            1
        )
    })

resumen = pd.DataFrame(
    resumen
)

print("\nCOMPLETITUD:")
print(
    resumen.to_string(
        index=False
    )
)

print(
    "\nBackups guardados:"
)

print(
    "- dataset_positivo_backup.csv"
)

print(
    "- dataset_financiero_backup.csv"
)

print(
    "- tabla_periodos_backup.csv"
)

Periodos: Belden Inc.
Periodos: General Electric
Periodos: Manitex International
Periodos: Power Solutions International
Periodos: Revolution Lighting Technologies
Periodos: Super Micro Computer
Periodos: Bausch Health / Valeant
Periodos: VEREIT / ARCP
Periodos: Healthcare Services Group
Periodos: Kraft Heinz
Periodos: Pareteum
Periodos: Cronos Group
Periodos: Koppers Holdings
Periodos: Tupperware Brands
Periodos: Compass Minerals

Observaciones temporales: 45
XBRL: Bausch Health / Valeant
XBRL: Belden Inc.
XBRL: Compass Minerals
XBRL: Cronos Group
XBRL: General Electric
XBRL: Healthcare Services Group
XBRL: Koppers Holdings
XBRL: Kraft Heinz
XBRL: Manitex International
XBRL: Pareteum
XBRL: Power Solutions International
XBRL: Revolution Lighting Technologies
XBRL: Super Micro Computer
XBRL: Tupperware Brands
XBRL: VEREIT / ARCP

RECONSTRUCCIÓN TERMINADA
Filas dataset positivo: 45
Filas dataset financiero: 42

COMPLETITUD:
           Variable  Disponibles  Faltantes  Completitud_%
     

In [14]:
# ============================================================
# DIAGNÓSTICO DE LOS FALTANTES TRAS LA RECONSTRUCCIÓN
# ============================================================

for variable in [
    "Revenue",
    "Receivables",
    "Liabilities_Current",
    "Inventory"
]:

    faltantes = dataset_financiero[
        dataset_financiero[variable].isna()
    ][
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Formulario",
            variable
        ]
    ]

    print("\n" + "=" * 90)
    print(
        f"{variable.upper()} - "
        f"{len(faltantes)} FALTANTES"
    )
    print("=" * 90)

    if faltantes.empty:
        print("Sin faltantes.")
    else:
        print(
            faltantes.to_string(index=False)
        )


REVENUE - 3 FALTANTES
    Empresa Periodo  Ejercicio Formulario  Revenue
Kraft Heinz     t-1       2017       10-K      NaN
Kraft Heinz     t-2       2016       10-K      NaN
Kraft Heinz     t-3       2015       10-K      NaN

RECEIVABLES - 3 FALTANTES
      Empresa Periodo  Ejercicio Formulario  Receivables
VEREIT / ARCP     t-1       2013       10-K          NaN
VEREIT / ARCP     t-2       2012       10-K          NaN
VEREIT / ARCP     t-3       2011       10-K          NaN

LIABILITIES_CURRENT - 6 FALTANTES
         Empresa Periodo  Ejercicio Formulario  Liabilities_Current
General Electric     t-1       2016       10-K                  NaN
General Electric     t-2       2015       10-K                  NaN
General Electric     t-3       2014       10-K                  NaN
   VEREIT / ARCP     t-1       2013       10-K                  NaN
   VEREIT / ARCP     t-2       2012       10-K                  NaN
   VEREIT / ARCP     t-3       2011       10-K                  NaN

INVENT

In [15]:
# ============================================================
# CORREGIR RECEIVABLES DE VEREIT YA VALIDADOS
# ============================================================

receivables_vereit = {
    2012: 2_422_000,
    2013: 11_628_000
}

for anio, valor in receivables_vereit.items():

    mask = (
        (dataset_positivo["Empresa"] == "VEREIT / ARCP")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "Receivables"
    ] = valor

print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "VEREIT / ARCP",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Receivables"
        ]
    ].to_string(index=False)
)

      Empresa Periodo  Ejercicio  Receivables
VEREIT / ARCP     t-1       2013   11628000.0
VEREIT / ARCP     t-2       2012    2422000.0
VEREIT / ARCP     t-3       2011          NaN


In [16]:
# ============================================================
# KRAFT HEINZ - BUSCAR REVENUE EN LOS FILINGS EXACTOS
# ============================================================

import requests
import pandas as pd

HEADERS = {
    "User-Agent": "TFM investigacion academica contacto@example.com",
    "Accept-Encoding": "gzip, deflate"
}

cik_kraft = "0001637459"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik_kraft}.json"
)

r = requests.get(url, headers=HEADERS)
r.raise_for_status()

facts_kraft = r.json()

usgaap_kraft = (
    facts_kraft
    .get("facts", {})
    .get("us-gaap", {})
)

# Filings exactos que seleccionamos para t-1, t-2 y t-3
filings_kraft = {
    2017: "0001637459-18-000013",
    2016: "0001637459-17-000007",
    2015: "0001637459-16-000100"
}

palabras_revenue = [
    "revenue",
    "salesrevenuenet",
    "salesrevenuegoodsnet"
]

resultados = []

for concepto, info in usgaap_kraft.items():

    etiqueta = info.get("label") or ""
    texto = f"{concepto} {etiqueta}".lower()

    if not any(
        palabra in texto
        for palabra in palabras_revenue
    ):
        continue

    unidades = info.get("units", {})

    if "USD" not in unidades:
        continue

    df_temp = pd.DataFrame(
        unidades["USD"]
    )

    if df_temp.empty or "accn" not in df_temp.columns:
        continue

    for ejercicio, accession in filings_kraft.items():

        df_filing = df_temp[
            df_temp["accn"] == accession
        ].copy()

        if df_filing.empty:
            continue

        if "start" in df_filing.columns:

            df_filing["start"] = pd.to_datetime(
                df_filing["start"],
                errors="coerce"
            )

            df_filing["end"] = pd.to_datetime(
                df_filing["end"],
                errors="coerce"
            )

            df_filing["dias"] = (
                df_filing["end"]
                - df_filing["start"]
            ).dt.days

            # Solo periodos aproximadamente anuales
            df_filing = df_filing[
                df_filing["dias"].between(330, 380)
            ]

        for _, reg in df_filing.iterrows():

            resultados.append({
                "Ejercicio_objetivo": ejercicio,
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Start": reg.get("start"),
                "End": reg.get("end"),
                "Valor": reg.get("val"),
                "fy": reg.get("fy"),
                "fp": reg.get("fp"),
                "form": reg.get("form"),
                "filed": reg.get("filed"),
                "accn": reg.get("accn")
            })


kraft_revenue_filings = pd.DataFrame(resultados)

print("REVENUE CANDIDATOS - KRAFT HEINZ")
print("=" * 120)

if kraft_revenue_filings.empty:

    print("No se encontraron candidatos.")

else:

    print(
        kraft_revenue_filings
        .sort_values(
            [
                "Ejercicio_objetivo",
                "Concepto",
                "End"
            ]
        )
        .to_string(index=False)
    )

REVENUE CANDIDATOS - KRAFT HEINZ
 Ejercicio_objetivo             Concepto                                          Etiqueta      Start        End       Valor   fy fp form      filed                 accn
               2016 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2013-12-30 2014-12-28 10922000000 2016 FY 10-K 2017-02-23 0001637459-17-000007
               2016 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2014-12-29 2016-01-03 18338000000 2016 FY 10-K 2017-02-23 0001637459-17-000007
               2016 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2016-01-04 2016-12-31 26487000000 2016 FY 10-K 2017-02-23 0001637459-17-000007


In [17]:
# ============================================================
# VER LOS FILINGS EXACTOS DE KRAFT HEINZ
# ============================================================

kraft_periodos = tabla_periodos[
    tabla_periodos["Empresa"] == "Kraft Heinz"
][
    [
        "Empresa",
        "Periodo",
        "Ejercicio",
        "Formulario",
        "Fecha_10K",
        "ReportDate",
        "Accession",
        "Documento"
    ]
].copy()

print("FILINGS SELECCIONADOS - KRAFT HEINZ")
print("=" * 110)

print(
    kraft_periodos.to_string(index=False)
)

FILINGS SELECCIONADOS - KRAFT HEINZ
    Empresa Periodo  Ejercicio Formulario  Fecha_10K ReportDate            Accession        Documento
Kraft Heinz     t-1       2017       10-K 2018-02-16 2017-12-30 0001637459-18-000015 form10-k2017.htm
Kraft Heinz     t-2       2016       10-K 2017-02-23 2016-12-31 0001637459-17-000007   khc201610k.htm
Kraft Heinz     t-3       2015       10-K 2016-03-03 2015-12-26 0001637459-16-000100   khc10k1316.htm


In [18]:
# ============================================================
# KRAFT HEINZ - REVENUE 2015, 2016 Y 2017
# CON LOS ACCESSION CORRECTOS
# ============================================================

filings_kraft_correctos = {
    2017: "0001637459-18-000015",
    2016: "0001637459-17-000007",
    2015: "0001637459-16-000100"
}

resultados = []

for concepto, info in usgaap_kraft.items():

    etiqueta = info.get("label") or ""

    texto = (
        f"{concepto} {etiqueta}"
        .lower()
    )

    if not any(
        palabra in texto
        for palabra in [
            "revenue",
            "salesrevenuenet",
            "salesrevenuegoodsnet"
        ]
    ):
        continue

    unidades = info.get("units", {})

    if "USD" not in unidades:
        continue

    df_temp = pd.DataFrame(
        unidades["USD"]
    )

    if (
        df_temp.empty
        or "accn" not in df_temp.columns
    ):
        continue

    for ejercicio, accession in filings_kraft_correctos.items():

        df_filing = df_temp[
            df_temp["accn"] == accession
        ].copy()

        if df_filing.empty:
            continue

        # Convertimos fechas
        if "start" in df_filing.columns:

            df_filing["start"] = pd.to_datetime(
                df_filing["start"],
                errors="coerce"
            )

        if "end" in df_filing.columns:

            df_filing["end"] = pd.to_datetime(
                df_filing["end"],
                errors="coerce"
            )

        # Duración
        if (
            "start" in df_filing.columns
            and "end" in df_filing.columns
        ):

            df_filing["dias"] = (
                df_filing["end"]
                - df_filing["start"]
            ).dt.days

            # Solo datos anuales
            df_filing = df_filing[
                df_filing["dias"].between(
                    330,
                    380
                )
            ]

        for _, reg in df_filing.iterrows():

            resultados.append({
                "Ejercicio_objetivo": ejercicio,
                "Concepto": concepto,
                "Etiqueta": etiqueta,
                "Start": reg.get("start"),
                "End": reg.get("end"),
                "Dias": reg.get("dias"),
                "Valor": reg.get("val"),
                "fy": reg.get("fy"),
                "fp": reg.get("fp"),
                "form": reg.get("form"),
                "filed": reg.get("filed"),
                "accn": reg.get("accn")
            })


kraft_revenue_final = pd.DataFrame(
    resultados
)

print(
    "REVENUE ANUAL - KRAFT HEINZ"
)

print("=" * 130)

if kraft_revenue_final.empty:

    print(
        "No se encontraron candidatos."
    )

else:

    print(
        kraft_revenue_final
        .sort_values(
            [
                "Ejercicio_objetivo",
                "Concepto",
                "End"
            ]
        )
        .to_string(index=False)
    )

REVENUE ANUAL - KRAFT HEINZ
 Ejercicio_objetivo             Concepto                                          Etiqueta      Start        End  Dias       Valor   fy fp form      filed                 accn
               2016 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2013-12-30 2014-12-28   363 10922000000 2016 FY 10-K 2017-02-23 0001637459-17-000007
               2016 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2014-12-29 2016-01-03   370 18338000000 2016 FY 10-K 2017-02-23 0001637459-17-000007
               2016 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2016-01-04 2016-12-31   362 26487000000 2016 FY 10-K 2017-02-23 0001637459-17-000007
               2017 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2014-12-29 2016-01-03   370 18338000000 2017 FY 10-K 2018-02-16 0001637459-18-000015
               2017 SalesRevenueGoodsNet Sales Revenue, Goods, Net (Deprecated 2018-01-31) 2

In [19]:
# ============================================================
# KRAFT HEINZ 2015
# BUSCAR TODOS LOS CONCEPTOS DEL 10-K CON CIFRAS ANUALES
# ============================================================

accession_2015 = "0001637459-16-000100"

resultados_2015 = []

for concepto, info in usgaap_kraft.items():

    etiqueta = info.get("label") or ""
    unidades = info.get("units", {})

    if "USD" not in unidades:
        continue

    df_temp = pd.DataFrame(
        unidades["USD"]
    )

    if (
        df_temp.empty
        or "accn" not in df_temp.columns
    ):
        continue

    # Solo hechos pertenecientes al 10-K 2015
    df_filing = df_temp[
        df_temp["accn"] == accession_2015
    ].copy()

    if df_filing.empty:
        continue

    # Necesitamos conceptos de duración
    if (
        "start" not in df_filing.columns
        or "end" not in df_filing.columns
    ):
        continue

    df_filing["start"] = pd.to_datetime(
        df_filing["start"],
        errors="coerce"
    )

    df_filing["end"] = pd.to_datetime(
        df_filing["end"],
        errors="coerce"
    )

    df_filing["dias"] = (
        df_filing["end"]
        - df_filing["start"]
    ).dt.days

    # Periodos aproximadamente anuales
    df_filing = df_filing[
        df_filing["dias"].between(
            330,
            380
        )
    ]

    if df_filing.empty:
        continue

    # Nos interesan especialmente hechos que
    # terminan alrededor del cierre de 2015
    for _, reg in df_filing.iterrows():

        resultados_2015.append({
            "Concepto": concepto,
            "Etiqueta": etiqueta,
            "Start": reg.get("start"),
            "End": reg.get("end"),
            "Dias": reg.get("dias"),
            "Valor": reg.get("val"),
            "fy": reg.get("fy"),
            "fp": reg.get("fp"),
            "form": reg.get("form")
        })


kraft_2015_todos = pd.DataFrame(
    resultados_2015
)

# ------------------------------------------------------------
# Filtrar conceptos que puedan estar relacionados con
# ventas, ingresos, productos, operaciones, etc.
# ------------------------------------------------------------

patron = (
    "revenue|sales|net sales|"
    "product|goods|customer|"
    "operating revenue|income"
)

candidatos_2015 = kraft_2015_todos[
    kraft_2015_todos["Concepto"].str.contains(
        patron,
        case=False,
        na=False
    )
    |
    kraft_2015_todos["Etiqueta"].str.contains(
        patron,
        case=False,
        na=False
    )
].copy()


# Nos quedamos preferentemente con periodos
# que terminan en 2015 o comienzos de 2016
candidatos_2015 = candidatos_2015[
    candidatos_2015["End"].dt.year.isin(
        [2015, 2016]
    )
]


print("CANDIDATOS REVENUE - KRAFT HEINZ 2015")
print("=" * 130)

if candidatos_2015.empty:

    print(
        "No se encontraron candidatos."
    )

else:

    print(
        candidatos_2015
        .sort_values(
            ["Valor"],
            ascending=False
        )
        .to_string(index=False)
    )

CANDIDATOS REVENUE - KRAFT HEINZ 2015
No se encontraron candidatos.


In [20]:
# ============================================================
# CORREGIR REVENUE - KRAFT HEINZ
# ============================================================

revenue_kraft = {
    2017: 26_232_000_000,
    2016: 26_487_000_000
}

for anio, valor in revenue_kraft.items():

    mask = (
        (dataset_positivo["Empresa"] == "Kraft Heinz")
        &
        (dataset_positivo["Ejercicio"] == anio)
    )

    dataset_positivo.loc[
        mask,
        "Revenue"
    ] = valor


# 2015 se mantiene como NaN
mask_2015 = (
    (dataset_positivo["Empresa"] == "Kraft Heinz")
    &
    (dataset_positivo["Ejercicio"] == 2015)
)

dataset_positivo.loc[
    mask_2015,
    "Revenue"
] = pd.NA


print(
    dataset_positivo.loc[
        dataset_positivo["Empresa"] == "Kraft Heinz",
        [
            "Empresa",
            "Periodo",
            "Ejercicio",
            "Revenue"
        ]
    ].to_string(index=False)
)

    Empresa Periodo  Ejercicio      Revenue
Kraft Heinz     t-1       2017 2.623200e+10
Kraft Heinz     t-2       2016 2.648700e+10
Kraft Heinz     t-3       2015          NaN


In [21]:
# ============================================================
# ACTUALIZAR DATASETS Y BACKUPS
# ============================================================

dataset_financiero = dataset_positivo[
    dataset_positivo["Empresa"] != "Cronos Group"
].copy()

dataset_positivo.to_csv(
    "dataset_positivo_backup.csv",
    index=False
)

dataset_financiero.to_csv(
    "dataset_financiero_backup.csv",
    index=False
)

tabla_periodos.to_csv(
    "tabla_periodos_backup.csv",
    index=False
)

print("Backups actualizados correctamente.")

Backups actualizados correctamente.


In [22]:
# ============================================================
# COMPROBACIÓN FINAL DEL DATASET FINANCIERO
# ============================================================

variables = [
    "Revenue",
    "Assets",
    "Liabilities_Total",
    "Liabilities_Current",
    "NetIncome",
    "Receivables",
    "Inventory",
    "Cash",
    "Equity"
]

resumen_final = []

for variable in variables:

    disponibles = dataset_financiero[variable].notna().sum()
    faltantes = dataset_financiero[variable].isna().sum()

    resumen_final.append({
        "Variable": variable,
        "Disponibles": disponibles,
        "Faltantes": faltantes,
        "Completitud_%": round(
            disponibles / len(dataset_financiero) * 100,
            1
        )
    })

resumen_final = pd.DataFrame(resumen_final)

print("OBSERVACIONES FINANCIERAS:", len(dataset_financiero))

print("\nCOMPLETITUD FINAL")
print("=" * 65)

print(
    resumen_final.to_string(index=False)
)

print("\n" + "=" * 65)
print("TOTAL DE VALORES FALTANTES")
print("=" * 65)

print(
    dataset_financiero[variables]
    .isna()
    .sum()
    .sum()
)

OBSERVACIONES FINANCIERAS: 42

COMPLETITUD FINAL
           Variable  Disponibles  Faltantes  Completitud_%
            Revenue           41          1           97.6
             Assets           42          0          100.0
  Liabilities_Total           42          0          100.0
Liabilities_Current           36          6           85.7
          NetIncome           42          0          100.0
        Receivables           41          1           97.6
          Inventory           36          6           85.7
               Cash           42          0          100.0
             Equity           42          0          100.0

TOTAL DE VALORES FALTANTES
14


In [23]:
# ============================================================
# GUARDAR DATASET FINANCIERO EN EXCEL
# ============================================================

dataset_financiero.to_excel(
    "dataset_financiero_final.xlsx",
    index=False
)

print("Archivo guardado: dataset_financiero_final.xlsx")

Archivo guardado: dataset_financiero_final.xlsx
